In [1]:
from __future__ import annotations

import os

os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "4")
os.environ.setdefault("GDAL_NUM_THREADS", "1")
os.environ.setdefault("GDAL_CACHEMAX", "512")
# Evita la ruta NHWC de BatchNorm en MIOpen. El codificador se convierte
# además a GroupNorm para trabajar de forma estable con lotes pequeños.
os.environ.setdefault("PYTORCH_MIOPEN_SUGGEST_NHWC_BATCHNORM", "0")
os.environ.setdefault("MIOPEN_FIND_MODE", "FAST")
os.environ.setdefault(
    "PYTORCH_ALLOC_CONF",
    "expandable_segments:True,max_split_size_mb:128",
)

import gc
import json
import os
import random
import re
import shutil
import time
import traceback
import warnings
from contextlib import nullcontext
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import models
from tqdm.auto import tqdm

# =============================================================================
# 1. RUTAS — MODIFICAR ÚNICAMENTE ESTA SECCIÓN
# =============================================================================
DATASET_ROOT = Path(r"C:\Users\Ivan sua\anaconda3\Dataset")
METADATA_DIR = DATASET_ROOT / "metadata"
RUNS_DIR = Path(r"C:\Users\Ivan sua\anaconda3\Modelos_DenseNet121_Segmentacion")

LEVELS_TO_TRAIN = ["nivel_1", "nivel_2"]
SKIP_COMPLETED_LEVELS = True

CREATE_NEW_RUNS = False  # Reanuda automáticamente la última ejecución
RESUME_BY_LEVEL = {"nivel_1": None, "nivel_2": None}

REQUIRE_CUDA = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Uso controlado del Ryzen 5 5600X sin saturar la RAM.
torch.set_num_threads(4)
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass
torch.set_float32_matmul_precision("high")
SEED = 42
DETERMINISTIC = False

# Misma unidad de salida y mismas métricas que U-Net.
EPOCHS = 120
# Configuración estable para 16 GB de VRAM.
BATCH_SIZE_BY_LEVEL = {"nivel_1": 1, "nivel_2": 1}
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
OPTIMIZER_TYPE = "adamw"
# Ejecutar preferiblemente como archivo .py desde Anaconda Prompt.
NUM_WORKERS = 0
PREFETCH_FACTOR = None
PIN_MEMORY = False
PERSISTENT_WORKERS = False
EARLY_STOPPING_PATIENCE = 20
USE_AMP = False
GRADIENT_ACCUMULATION_BY_LEVEL = {"nivel_1": 4, "nivel_2": 4}

# Pausas y limpieza para evitar carga sostenida del controlador AMD.
EPOCH_COOLDOWN_SECONDS = 3
LEVEL_COOLDOWN_SECONDS = 20
PROGRESS_UPDATE_EVERY = 20
GPU_CLEANUP_EVERY_STEPS = 0  # 0 = solo al terminar cada época; evita vaciar caché en cada lote.
USE_DENSENET_MEMORY_EFFICIENT = True

ENCODER_PRETRAINED = True
FREEZE_ENCODER = False
DECODER_DROPOUT = 0.10

BAND_INDEXES = (1, 2, 3, 4)
EXPECTED_CHIP_BANDS = 4
REFLECTANCE_SCALE = 10000.0
REFLECTANCE_ADD_OFFSET = 0.0
USE_RASTER_SCALE_OFFSET = True
AUTO_SCALE_REFLECTANCE = True
STATS_SAMPLE_SIZE = 300

MASK_NODATA_CODE = 0
IGNORE_INDEX = -100
LOSS_TYPE = "ce_dice"
DICE_WEIGHT = 0.7
USE_CLASS_WEIGHTS = True
USE_WEIGHTED_SAMPLER = False
USE_AUGMENTATION = True

SAVE_EVERY_EPOCH = False
SAVE_TEST_PREDICTIONS = False
MAX_TEST_PREDICTIONS_TO_SAVE = 20
SAVE_REPORT_PLOTS = False
PLOT_AFTER_TRAINING = False


CLASES_NIVEL_1 = {
    0: "NoData",
    1: "2. Territorios agrícolas",
    2: "3. Bosques y áreas seminaturales",
    3: "5. Superficies de agua",
    4: "1. Territorios artificializados",
    5: "4. Áreas húmedas",
}

CLASES_NIVEL_2 = {
    0: "NoData",
    1: "2.4 Áreas agrícolas heterogéneas",
    2: "3.2 Áreas con vegetación herbácea o arbustiva",
    3: "3.1 Bosques",
    4: "2.3 Pastos",
    5: "5.1 Aguas continentales",
    6: "1.1 Zonas urbanizadas",
    7: "1.3 Zonas de extracción minera y escombreras",
    8: "4.1 Áreas húmedas continentales",
    9: "1.2 Zonas industriales o comerciales y redes de comunicación",
    10: "1.4 Zonas verdes artificializadas no agrícolas",
    11: "2.1 Cultivos transitorios",
    12: "2.2 Cultivos permanentes",
    13: "3.3 Áreas abiertas sin o con poca vegetación",
}

LEVEL_FOLDER_BY_NAME = {
    "nivel_1": "Nivel_1",
    "nivel_2": "Nivel_2",
    "metadata": "metadata",
}

def get_class_dict(level_name: str) -> dict[int, str]:
    if level_name == "nivel_1":
        return CLASES_NIVEL_1
    if level_name == "nivel_2":
        return CLASES_NIVEL_2
    raise ValueError(f"Nivel no reconocido: {level_name}")

def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

        # En ROCm la interfaz sigue siendo torch.cuda, pero cuDNN no es el backend.
        # Evitamos tocar opciones específicas de cuDNN cuando la GPU es AMD/ROCm.
        if torch.version.hip is None:
            torch.backends.cudnn.deterministic = DETERMINISTIC
            torch.backends.cudnn.benchmark = not DETERMINISTIC


def gpu_cleanup(synchronize: bool = False) -> None:
    """Libera referencias/caché de GPU sin convertir una limpieza en otro error."""
    gc.collect()

    if DEVICE.type != "cuda":
        return

    try:
        if synchronize:
            torch.cuda.synchronize()
    except Exception:
        pass

    try:
        torch.cuda.empty_cache()
    except Exception:
        pass


def is_gpu_oom(error: BaseException) -> bool:
    text = str(error).lower()
    return (
        "out of memory" in text
        or "hip out of memory" in text
        or "cuda out of memory" in text
        or "memory allocation" in text
    )


def ensure_dir(path: str | Path) -> Path:
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def json_default(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    raise TypeError(f"Objeto no serializable: {type(value).__name__}")


def save_json(data, path: str | Path) -> None:
    path = Path(path)
    ensure_dir(path.parent)
    with path.open("w", encoding="utf-8") as file:
        json.dump(data, file, indent=2, ensure_ascii=False, default=json_default)


def safe_name(text: str) -> str:
    normalized = (
        str(text)
        .translate(str.maketrans("áéíóúÁÉÍÓÚñÑ", "aeiouAEIOUnN"))
    )
    normalized = re.sub(r"[^A-Za-z0-9]+", "_", normalized)
    return re.sub(r"_+", "_", normalized).strip("_").lower()


def resolve_case_insensitive(path: Path) -> Path:
    if path.exists():
        return path

    parts = path.parts
    if not parts:
        return path

    current = Path(parts[0])
    for part in parts[1:]:
        if current.exists() and current.is_dir():
            try:
                match = next(
                    (item.name for item in current.iterdir() if item.name.lower() == part.lower()),
                    part,
                )
            except OSError:
                match = part
            current = current / match
        else:
            current = current / part
    return current


def resolve_dataset_path(value) -> str:
    if pd.isna(value):
        return ""

    raw = os.path.expandvars(os.path.expanduser(str(value).strip().strip('"').strip("'")))
    if not raw:
        return ""

    direct = resolve_case_insensitive(Path(raw))
    if direct.exists():
        return str(direct)

    parts = [part for part in raw.replace("\\", "/").split("/") if part not in {"", "."}]
    lower_parts = [part.lower() for part in parts]

    if "dataset" in lower_parts:
        index = len(lower_parts) - 1 - lower_parts[::-1].index("dataset")
        candidate = DATASET_ROOT.joinpath(*parts[index + 1 :])
    else:
        candidate = DATASET_ROOT.joinpath(*parts)

    relative_parts = list(candidate.relative_to(DATASET_ROOT).parts)
    if relative_parts:
        relative_parts[0] = LEVEL_FOLDER_BY_NAME.get(relative_parts[0].lower(), relative_parts[0])
    return str(resolve_case_insensitive(DATASET_ROOT.joinpath(*relative_parts)))


def get_splits_dir(level_name: str) -> Path:
    candidates = [
        METADATA_DIR / "splits" / level_name,
        METADATA_DIR / "splits" / LEVEL_FOLDER_BY_NAME.get(level_name, level_name),
    ]
    return next((path for path in candidates if path.exists()), candidates[0])


def find_split_file(splits_dir: Path, *names: str) -> Path:
    if not splits_dir.exists():
        raise FileNotFoundError(f"No existe la carpeta de splits: {splits_dir}")

    available = {item.name.lower(): item for item in splits_dir.iterdir() if item.is_file()}
    for name in names:
        if name.lower() in available:
            return available[name.lower()]

    raise FileNotFoundError(f"No se encontró ninguno de {names} en {splits_dir}")


def read_split_csv(csv_path: Path, split_name: str) -> pd.DataFrame:
    if not csv_path.exists() or csv_path.stat().st_size == 0:
        raise FileNotFoundError(f"CSV inexistente o vacío: {csv_path}")

    df = pd.read_csv(csv_path)
    columns = {str(column).strip().lower(): column for column in df.columns}

    image_column = next(
        (columns[name] for name in ("image_path", "image", "img_path", "img") if name in columns),
        None,
    )
    mask_column = next(
        (columns[name] for name in ("mask_path", "label_path", "mask", "label") if name in columns),
        None,
    )

    if image_column is None or mask_column is None:
        raise ValueError(
            f"El split {csv_path} debe contener image_path y mask_path; "
            f"columnas disponibles: {list(df.columns)}"
        )

    df = df.rename(columns={image_column: "image_path", mask_column: "mask_path"})
    df = df.dropna(subset=["image_path", "mask_path"]).copy()
    df["image_path"] = df["image_path"].apply(resolve_dataset_path)
    df["mask_path"] = df["mask_path"].apply(resolve_dataset_path)

    duplicate_count = int(df.duplicated(subset=["image_path", "mask_path"]).sum())
    if duplicate_count:
        warnings.warn(
            f"{split_name}: se eliminaron {duplicate_count} pares duplicados.",
            RuntimeWarning,
        )
        df = df.drop_duplicates(subset=["image_path", "mask_path"], keep="first")

    missing_images = [path for path in df["image_path"] if not Path(path).exists()]
    missing_masks = [path for path in df["mask_path"] if not Path(path).exists()]
    if missing_images or missing_masks:
        raise FileNotFoundError(
            f"{split_name}: faltan {len(missing_images)} imágenes y {len(missing_masks)} máscaras."
        )

    if df.empty:
        raise ValueError(f"El split {split_name} quedó vacío después de la validación.")

    return df.reset_index(drop=True)


def normalized_pair_set(df: pd.DataFrame) -> set[tuple[str, str]]:
    return {
        (
            str(Path(image)).replace("\\", "/").lower(),
            str(Path(mask)).replace("\\", "/").lower(),
        )
        for image, mask in zip(df["image_path"], df["mask_path"])
    }


def lineage_set(df: pd.DataFrame) -> set[str]:
    columns = {str(column).strip().lower(): column for column in df.columns}
    for candidate in ("source_chip_id", "original_chip_id", "chip_id"):
        if candidate in columns:
            values = df[columns[candidate]].dropna().astype(str).str.strip()
            return {value.lower() for value in values if value}
    return set()


def validate_split_disjointness(train: pd.DataFrame, val: pd.DataFrame, test: pd.DataFrame) -> None:
    pair_sets = {
        "train": normalized_pair_set(train),
        "val": normalized_pair_set(val),
        "test": normalized_pair_set(test),
    }
    pair_overlaps = {
        "train-val": pair_sets["train"] & pair_sets["val"],
        "train-test": pair_sets["train"] & pair_sets["test"],
        "val-test": pair_sets["val"] & pair_sets["test"],
    }

    lineage_sets = {
        "train": lineage_set(train),
        "val": lineage_set(val),
        "test": lineage_set(test),
    }
    lineage_overlaps = {
        "train-val": lineage_sets["train"] & lineage_sets["val"],
        "train-test": lineage_sets["train"] & lineage_sets["test"],
        "val-test": lineage_sets["val"] & lineage_sets["test"],
    }

    conflicts = {name: len(values) for name, values in pair_overlaps.items() if values}
    lineage_conflicts = {
        name: len(values)
        for name, values in lineage_overlaps.items()
        if values
    }

    if conflicts or lineage_conflicts:
        details = []
        if conflicts:
            details.append(
                "pares repetidos: "
                + ", ".join(f"{name}={count}" for name, count in conflicts.items())
            )
        if lineage_conflicts:
            details.append(
                "chips originales compartidos entre splits: "
                + ", ".join(f"{name}={count}" for name, count in lineage_conflicts.items())
            )
        raise ValueError(
            "Se detectó fuga de información entre entrenamiento, validación y prueba; "
            + "; ".join(details)
            + ". Los aumentos de un mismo chip deben permanecer en un único split."
        )


def validate_image_mask_pairs(
    dataframes: dict[str, pd.DataFrame],
    class_dict: dict[int, str],
) -> tuple[int, tuple[int, int]]:
    expected_shape = None
    observed_classes = set()

    for split_name, dataframe in dataframes.items():
        for image_path, mask_path in tqdm(
            zip(dataframe["image_path"], dataframe["mask_path"]),
            total=len(dataframe),
            desc=f"Validando {split_name}",
        ):
            with rasterio.open(image_path) as image_src, rasterio.open(mask_path) as mask_src:
                if image_src.count != EXPECTED_CHIP_BANDS:
                    raise ValueError(
                        f"{image_path} tiene {image_src.count} bandas; se esperaban {EXPECTED_CHIP_BANDS}."
                    )
                if max(BAND_INDEXES) > image_src.count:
                    raise ValueError(f"Índices de banda inválidos para {image_path}.")
                if (image_src.height, image_src.width) != (mask_src.height, mask_src.width):
                    raise ValueError(f"Dimensiones incompatibles: {image_path} y {mask_path}.")
                if image_src.crs != mask_src.crs:
                    raise ValueError(f"CRS incompatible: {image_path} y {mask_path}.")
                if not image_src.transform.almost_equals(mask_src.transform):
                    raise ValueError(f"Transformación espacial incompatible: {image_path} y {mask_path}.")

                shape = (image_src.height, image_src.width)
                if expected_shape is None:
                    expected_shape = shape
                elif shape != expected_shape:
                    raise ValueError(
                        f"Todos los chips de un nivel deben tener el mismo tamaño; "
                        f"se detectaron {expected_shape} y {shape}."
                    )

                mask = mask_src.read(1)
                observed_classes.update(int(value) for value in np.unique(mask))

    allowed = set(class_dict)
    unexpected = observed_classes - allowed
    if unexpected:
        raise ValueError(f"Las máscaras contienen códigos no definidos: {sorted(unexpected)}.")

    train_classes = set()
    for mask_path in dataframes["train"]["mask_path"]:
        with rasterio.open(mask_path) as src:
            train_classes.update(int(value) for value in np.unique(src.read(1)))
    missing_train = (allowed - {MASK_NODATA_CODE}) - train_classes
    if missing_train:
        raise ValueError(
            f"Clases sin píxeles en entrenamiento: {sorted(missing_train)}."
        )

    return EXPECTED_CHIP_BANDS, expected_shape


def read_splits(level_name: str, class_dict: dict[int, str]):
    splits_dir = get_splits_dir(level_name)
    train = read_split_csv(
        find_split_file(splits_dir, "train_samples.csv", "train.csv"),
        "train",
    )
    val = read_split_csv(
        find_split_file(splits_dir, "val_samples.csv", "val.csv"),
        "val",
    )
    test = read_split_csv(
        find_split_file(splits_dir, "test_samples.csv", "test.csv"),
        "test",
    )

    validate_split_disjointness(train, val, test)
    band_count, chip_shape = validate_image_mask_pairs(
        {"train": train, "val": val, "test": test},
        class_dict,
    )
    return train, val, test, splits_dir, band_count, chip_shape

def read_reflectance(path: str | Path, band_indexes=None):
    band_indexes = BAND_INDEXES if band_indexes is None else band_indexes

    with rasterio.open(path) as src:
        array = src.read(indexes=band_indexes).astype(np.float32)
        nodata = src.nodata
        scales = np.asarray(
            [src.scales[index - 1] for index in band_indexes],
            dtype=np.float32,
        )[:, None, None]
        offsets = np.asarray(
            [src.offsets[index - 1] for index in band_indexes],
            dtype=np.float32,
        )[:, None, None]

    finite = np.isfinite(array)
    if nodata is not None and np.isfinite(nodata):
        finite &= array != nodata

    valid_pixel = finite.all(axis=0)
    array = np.where(finite, array, np.nan)

    finite_values = array[np.isfinite(array)]
    if finite_values.size == 0:
        raise ValueError(f"La imagen no contiene valores válidos: {path}")

    has_raster_transform = (
        USE_RASTER_SCALE_OFFSET
        and (
            not np.allclose(scales, 1.0)
            or not np.allclose(offsets, 0.0)
        )
    )
    if has_raster_transform:
        array = array * scales + offsets
    elif AUTO_SCALE_REFLECTANCE and float(np.nanpercentile(finite_values, 99.5)) > 2.0:
        array = (
            array + float(REFLECTANCE_ADD_OFFSET)
        ) / float(REFLECTANCE_SCALE)

    return np.clip(array, 0.0, 1.0).astype(np.float32), valid_pixel


class LULCSegmentationDataset(Dataset):
    def __init__(
        self,
        dataframe: pd.DataFrame,
        code_to_idx: dict[int, int],
        means: list[float],
        stds: list[float],
        augment: bool = False,
    ):
        self.df = dataframe.reset_index(drop=True)
        self.code_to_idx = code_to_idx
        self.means = np.asarray(means, dtype=np.float32)[:, None, None]
        self.stds = np.asarray(stds, dtype=np.float32)[:, None, None]
        self.augment = augment

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, index: int):
        row = self.df.iloc[index]
        image, valid_pixel = read_reflectance(row["image_path"])

        with rasterio.open(row["mask_path"]) as src:
            original_mask = src.read(1).astype(np.int64)

        if image.shape[1:] != original_mask.shape:
            raise ValueError(
                f"Dimensiones incompatibles: {row['image_path']} y {row['mask_path']}."
            )

        mask = np.full(original_mask.shape, IGNORE_INDEX, dtype=np.int64)
        for original_code, internal_index in self.code_to_idx.items():
            mask[(original_mask == original_code) & valid_pixel] = internal_index

        if self.augment:
            if random.random() < 0.5:
                image = np.flip(image, axis=2)
                mask = np.flip(mask, axis=1)
            if random.random() < 0.5:
                image = np.flip(image, axis=1)
                mask = np.flip(mask, axis=0)
            rotation = random.randint(0, 3)
            if rotation:
                image = np.rot90(image, rotation, axes=(1, 2))
                mask = np.rot90(mask, rotation, axes=(0, 1))

        valid = np.isfinite(image).all(axis=0) & (mask != IGNORE_INDEX)
        image = np.nan_to_num(image, nan=0.0)
        image = (image - self.means) / np.maximum(self.stds, 1e-6)
        image[:, ~valid] = 0.0

        return {
            "image": torch.from_numpy(np.ascontiguousarray(image)).float(),
            "mask": torch.from_numpy(np.ascontiguousarray(mask)).long(),
            "image_path": row["image_path"],
            "mask_path": row["mask_path"],
        }


def compute_image_stats(
    dataframe: pd.DataFrame,
    sample_size: int | None = None,
) -> tuple[list[float], list[float]]:
    sample_size = STATS_SAMPLE_SIZE if sample_size is None else sample_size
    paths = dataframe["image_path"].tolist()

    if sample_size and len(paths) > sample_size:
        rng = np.random.default_rng(SEED)
        paths = rng.choice(paths, size=sample_size, replace=False).tolist()

    channel_sum = None
    channel_squared_sum = None
    channel_count = None

    for path in tqdm(paths, desc="Estadísticas radiométricas"):
        array, _ = read_reflectance(path)
        flat = array.reshape(array.shape[0], -1)
        valid = np.isfinite(flat)

        sums = np.nansum(flat, axis=1)
        squared_sums = np.nansum(flat * flat, axis=1)
        counts = valid.sum(axis=1)

        if channel_sum is None:
            channel_sum = sums
            channel_squared_sum = squared_sums
            channel_count = counts
        else:
            channel_sum += sums
            channel_squared_sum += squared_sums
            channel_count += counts

    if channel_sum is None or np.any(channel_count == 0):
        raise RuntimeError("No fue posible calcular las estadísticas radiométricas.")

    means = channel_sum / channel_count
    variances = channel_squared_sum / channel_count - means**2
    stds = np.sqrt(np.maximum(variances, 1e-12))
    return means.astype(float).tolist(), stds.astype(float).tolist()


def compute_class_pixel_counts(
    dataframe: pd.DataFrame,
    code_to_idx: dict[int, int],
) -> np.ndarray:
    counts = np.zeros(len(code_to_idx), dtype=np.int64)

    iterator = dataframe[["image_path", "mask_path"]].itertuples(index=False)
    for image_path, mask_path in tqdm(
        iterator,
        total=len(dataframe),
        desc="Píxeles por clase",
    ):
        _, valid_pixel = read_reflectance(image_path)
        with rasterio.open(mask_path) as src:
            mask = src.read(1)

        if mask.shape != valid_pixel.shape:
            raise ValueError(
                f"Dimensiones incompatibles al contar clases: {image_path} y {mask_path}."
            )

        for original_code, internal_index in code_to_idx.items():
            counts[internal_index] += int(
                ((mask == original_code) & valid_pixel).sum()
            )

    return counts


def make_class_weights(counts: np.ndarray) -> torch.Tensor:
    if np.any(counts == 0):
        missing = np.where(counts == 0)[0].tolist()
        raise ValueError(f"Clases internas sin píxeles de entrenamiento: {missing}")

    frequencies = counts / counts.sum()
    weights = 1.0 / np.sqrt(frequencies)
    weights /= weights.mean()
    return torch.tensor(weights, dtype=torch.float32)


def build_weighted_sampler(dataframe: pd.DataFrame):
    if not USE_WEIGHTED_SAMPLER:
        return None
    if "class_code" not in dataframe.columns:
        warnings.warn(
            "No existe class_code; se omite el muestreo balanceado por chip.",
            RuntimeWarning,
        )
        return None

    codes = pd.to_numeric(dataframe["class_code"], errors="coerce").fillna(-1).astype(int)
    counts = codes.value_counts().to_dict()
    sample_weights = codes.map(lambda code: 1.0 / counts[int(code)]).to_numpy(dtype=np.float64)

    generator = torch.Generator()
    generator.manual_seed(SEED)
    return WeightedRandomSampler(
        torch.as_tensor(sample_weights, dtype=torch.double),
        num_samples=len(sample_weights),
        replacement=True,
        generator=generator,
    )


def save_dataset_summary(
    train: pd.DataFrame,
    val: pd.DataFrame,
    test: pd.DataFrame,
    pixel_counts: np.ndarray,
    idx_to_code: dict[int, int],
    class_dict: dict[int, str],
    output_dir: Path,
) -> None:
    output_dir = ensure_dir(output_dir)

    split_summary = pd.DataFrame(
        {
            "split": ["train", "val", "test"],
            "chips": [len(train), len(val), len(test)],
        }
    )
    split_summary["percentage"] = (
        split_summary["chips"] / split_summary["chips"].sum() * 100.0
    )
    split_summary.to_csv(
        output_dir / "dataset_splits.csv",
        index=False,
        encoding="utf-8-sig",
    )

    pixel_summary = pd.DataFrame(
        {
            "class_index": np.arange(len(pixel_counts)),
            "class_code": [idx_to_code[index] for index in range(len(pixel_counts))],
            "class_name": [
                class_dict[idx_to_code[index]]
                for index in range(len(pixel_counts))
            ],
            "pixels_train": pixel_counts,
        }
    )
    pixel_summary["percentage_train"] = (
        pixel_summary["pixels_train"] / max(pixel_summary["pixels_train"].sum(), 1) * 100.0
    )
    pixel_summary.to_csv(
        output_dir / "dataset_pixels_por_clase.csv",
        index=False,
        encoding="utf-8-sig",
    )

class ConvNormAct(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, dropout: float = 0.0):
        super().__init__()
        groups = min(32, out_channels)
        while out_channels % groups != 0:
            groups -= 1
        layers = [
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(groups, out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(groups, out_channels),
            nn.ReLU(inplace=True),
        ]
        if dropout > 0:
            layers.insert(3, nn.Dropout2d(dropout))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class DecoderBlock(nn.Module):
    def __init__(self, in_channels: int, skip_channels: int, out_channels: int, dropout: float = 0.0):
        super().__init__()
        self.conv = ConvNormAct(in_channels + skip_channels, out_channels, dropout)

    def forward(self, decoder, skip):
        decoder = F.interpolate(
            decoder,
            size=skip.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )
        return self.conv(torch.cat([decoder, skip], dim=1))



def _valid_group_count(channels: int, maximum_groups: int = 32) -> int:
    """Selecciona el mayor número de grupos que divide exactamente los canales."""
    for groups in (maximum_groups, 16, 8, 4, 2, 1):
        if groups <= channels and channels % groups == 0:
            return groups
    return 1


def replace_batchnorm_with_groupnorm(module: nn.Module) -> nn.Module:
    """
    Sustituye recursivamente BatchNorm2d por GroupNorm.

    Esta conversión evita la operación BatchNorm de MIOpen que produjo
    miopenStatusUnknownError y es más estable con batch size 1–4.
    Los parámetros afines de los pesos preentrenados se conservan.
    """
    for name, child in list(module.named_children()):
        if isinstance(child, nn.BatchNorm2d):
            replacement = nn.GroupNorm(
                num_groups=_valid_group_count(child.num_features),
                num_channels=child.num_features,
                eps=child.eps,
                affine=child.affine,
            )
            if child.affine:
                with torch.no_grad():
                    replacement.weight.copy_(child.weight)
                    replacement.bias.copy_(child.bias)
            setattr(module, name, replacement)
        else:
            replace_batchnorm_with_groupnorm(child)
    return module


def adapt_first_convolution(old_conv: nn.Conv2d, in_channels: int, pretrained_loaded: bool) -> nn.Conv2d:
    new_conv = nn.Conv2d(
        in_channels,
        old_conv.out_channels,
        kernel_size=old_conv.kernel_size,
        stride=old_conv.stride,
        padding=old_conv.padding,
        bias=False,
    )
    with torch.no_grad():
        if pretrained_loaded:
            # Entrada Sentinel-2: B2, B3, B4, B8. Los pesos ImageNet vienen en RGB.
            if in_channels >= 3:
                new_conv.weight[:, 0] = old_conv.weight[:, 2]  # B2 <- azul
                new_conv.weight[:, 1] = old_conv.weight[:, 1]  # B3 <- verde
                new_conv.weight[:, 2] = old_conv.weight[:, 0]  # B4 <- rojo
                if in_channels > 3:
                    mean_weight = old_conv.weight.mean(dim=1, keepdim=True)
                    new_conv.weight[:, 3:] = mean_weight.repeat(1, in_channels - 3, 1, 1)
            else:
                new_conv.weight[:, :in_channels] = old_conv.weight[:, :in_channels]
        else:
            nn.init.kaiming_normal_(new_conv.weight, mode="fan_out", nonlinearity="relu")
    return new_conv


class DenseNet121UNet(nn.Module):
    """Segmentación semántica con DenseNet-121 como codificador y decodificador tipo U-Net."""

    def __init__(
        self,
        in_channels: int,
        num_classes: int,
        pretrained: bool = True,
        freeze_encoder: bool = False,
        decoder_dropout: float = 0.10,
    ):
        super().__init__()
        pretrained_loaded = False
        if pretrained:
            try:
                weights = models.DenseNet121_Weights.DEFAULT
                encoder = models.densenet121(
                    weights=weights,
                    memory_efficient=USE_DENSENET_MEMORY_EFFICIENT,
                )
                pretrained_loaded = True
            except Exception as exc:
                warnings.warn(
                    f"No fue posible cargar pesos ImageNet de DenseNet-121 ({exc}); se inicializa sin pesos.",
                    RuntimeWarning,
                )
                encoder = models.densenet121(
                    weights=None,
                    memory_efficient=USE_DENSENET_MEMORY_EFFICIENT,
                )
        else:
            encoder = models.densenet121(
                weights=None,
                memory_efficient=USE_DENSENET_MEMORY_EFFICIENT,
            )

        encoder = replace_batchnorm_with_groupnorm(encoder)
        features = encoder.features
        features.conv0 = adapt_first_convolution(features.conv0, in_channels, pretrained_loaded)

        self.conv0 = features.conv0
        self.norm0 = features.norm0
        self.relu0 = features.relu0
        self.pool0 = features.pool0
        self.denseblock1 = features.denseblock1     # 256 canales, 1/4
        self.transition1 = features.transition1     # 128 canales, 1/8
        self.denseblock2 = features.denseblock2     # 512 canales, 1/8
        self.transition2 = features.transition2     # 256 canales, 1/16
        self.denseblock3 = features.denseblock3     # 1024 canales, 1/16
        self.transition3 = features.transition3     # 512 canales, 1/32
        self.denseblock4 = features.denseblock4     # 1024 canales, 1/32
        self.norm5 = features.norm5

        self.decoder4 = DecoderBlock(1024, 1024, 512, decoder_dropout)
        self.decoder3 = DecoderBlock(512, 512, 256, decoder_dropout)
        self.decoder2 = DecoderBlock(256, 256, 128, decoder_dropout)
        self.decoder1 = DecoderBlock(128, 64, 64, decoder_dropout)
        self.final_refine = ConvNormAct(64, 32, decoder_dropout)
        self.segmentation_head = nn.Conv2d(32, num_classes, kernel_size=1)
        self.pretrained_loaded = pretrained_loaded

        if freeze_encoder:
            for module in (
                self.conv0, self.norm0, self.denseblock1, self.transition1,
                self.denseblock2, self.transition2, self.denseblock3,
                self.transition3, self.denseblock4, self.norm5,
            ):
                for parameter in module.parameters():
                    parameter.requires_grad = False

    def forward(self, x):
        input_size = x.shape[-2:]
        skip0 = self.relu0(self.norm0(self.conv0(x)))
        x = self.pool0(skip0)
        skip1 = self.denseblock1(x)
        x = self.transition1(skip1)
        skip2 = self.denseblock2(x)
        x = self.transition2(skip2)
        skip3 = self.denseblock3(x)
        x = self.transition3(skip3)
        x = self.norm5(self.denseblock4(x))
        x = F.relu(x, inplace=True)

        x = self.decoder4(x, skip3)
        x = self.decoder3(x, skip2)
        x = self.decoder2(x, skip1)
        x = self.decoder1(x, skip0)
        x = F.interpolate(x, size=input_size, mode="bilinear", align_corners=False)
        x = self.final_refine(x)
        return self.segmentation_head(x)


def build_segmentation_model(
    in_channels: int,
    num_classes: int,
    pretrained: bool = ENCODER_PRETRAINED,
    freeze_encoder: bool = FREEZE_ENCODER,
):
    model = DenseNet121UNet(
        in_channels=in_channels,
        num_classes=num_classes,
        pretrained=pretrained,
        freeze_encoder=freeze_encoder,
        decoder_dropout=DECODER_DROPOUT,
    )
    return model, bool(model.pretrained_loaded)

class DiceLoss(nn.Module):
    def __init__(self, num_classes: int, ignore_index: int = IGNORE_INDEX, smooth: float = 1.0):
        super().__init__()
        self.num_classes = num_classes
        self.ignore_index = ignore_index
        self.smooth = smooth

    def forward(self, logits, target):
        probabilities = torch.softmax(logits, dim=1)
        valid = target != self.ignore_index
        losses = []

        for class_index in range(self.num_classes):
            if class_index == self.ignore_index:
                continue

            truth = ((target == class_index) & valid).float()
            if truth.sum() == 0:
                continue

            prediction = probabilities[:, class_index] * valid
            intersection = (prediction * truth).sum()
            denominator = prediction.sum() + truth.sum()
            dice = (2.0 * intersection + self.smooth) / (denominator + self.smooth)
            losses.append(1.0 - dice)

        return torch.stack(losses).mean() if losses else logits.sum() * 0.0


class CombinedSegmentationLoss(nn.Module):
    def __init__(
        self,
        num_classes: int,
        class_weights: torch.Tensor | None,
    ):
        super().__init__()
        self.cross_entropy = nn.CrossEntropyLoss(
            weight=class_weights,
            ignore_index=IGNORE_INDEX,
        )
        self.dice = DiceLoss(num_classes, IGNORE_INDEX)

    def forward(self, logits, target):
        ce_loss = self.cross_entropy(logits, target)
        if LOSS_TYPE == "weighted_ce":
            return ce_loss
        if LOSS_TYPE == "ce_dice":
            return ce_loss + DICE_WEIGHT * self.dice(logits, target)
        raise ValueError(f"LOSS_TYPE no reconocido: {LOSS_TYPE}")


class ConfusionAccumulator:
    def __init__(self, num_classes: int, ignore_index: int = IGNORE_INDEX):
        self.num_classes = num_classes
        self.ignore_index = ignore_index
        self.matrix = torch.zeros((num_classes, num_classes), dtype=torch.int64)

    @torch.no_grad()
    def update(self, logits, target) -> None:
        prediction = logits.argmax(dim=1)
        valid = (
            (target != self.ignore_index)
            & (target >= 0)
            & (target < self.num_classes)
            & (prediction >= 0)
            & (prediction < self.num_classes)
        )
        encoded = self.num_classes * target[valid] + prediction[valid]
        counts = torch.bincount(
            encoded.detach().cpu(),
            minlength=self.num_classes**2,
        )
        self.matrix += counts.reshape(self.num_classes, self.num_classes)


def metrics_from_confusion(matrix: np.ndarray, ignore_index: int = IGNORE_INDEX):
    matrix = matrix.astype(np.float64)
    true_positive = np.diag(matrix)
    support = matrix.sum(axis=1)
    predicted = matrix.sum(axis=0)
    total = matrix.sum()

    precision = np.divide(
        true_positive,
        predicted,
        out=np.zeros_like(true_positive),
        where=predicted > 0,
    )
    recall = np.divide(
        true_positive,
        support,
        out=np.zeros_like(true_positive),
        where=support > 0,
    )
    f1 = np.divide(
        2 * precision * recall,
        precision + recall,
        out=np.zeros_like(true_positive),
        where=(precision + recall) > 0,
    )
    iou = np.divide(
        true_positive,
        support + predicted - true_positive,
        out=np.zeros_like(true_positive),
        where=(support + predicted - true_positive) > 0,
    )

    active = (support > 0) & (np.arange(len(support)) != ignore_index)
    pixel_accuracy = float(true_positive.sum() / total) if total else 0.0
    expected = float((support * predicted).sum() / (total**2)) if total else 0.0
    kappa = (
        (pixel_accuracy - expected) / (1.0 - expected)
        if total and expected < 1.0
        else 0.0
    )

    summary = {
        "pixel_accuracy": pixel_accuracy,
        "macro_precision": float(precision[active].mean()) if active.any() else 0.0,
        "macro_recall": float(recall[active].mean()) if active.any() else 0.0,
        "macro_f1": float(f1[active].mean()) if active.any() else 0.0,
        "mean_iou": float(iou[active].mean()) if active.any() else 0.0,
        "kappa": float(kappa),
    }
    per_class = pd.DataFrame(
        {
            "class_code": np.arange(len(support)),
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "iou": iou,
            "support": support.astype(np.int64),
        }
    )
    return summary, per_class


def amp_enabled() -> bool:
    return bool(USE_AMP and DEVICE.type == "cuda")


def autocast_context():
    if amp_enabled():
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    return nullcontext()


def make_grad_scaler():
    try:
        return torch.amp.GradScaler("cuda", enabled=amp_enabled())
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=amp_enabled())


def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    scaler,
    epoch,
    num_classes,
    gradient_accumulation_steps,
):
    model.train()
    accumulator = ConfusionAccumulator(num_classes)
    total_loss = 0.0
    total_valid_pixels = 0
    optimizer.zero_grad(set_to_none=True)

    progress = tqdm(
        loader,
        desc=f"Entrenamiento {epoch}",
        leave=False,
        mininterval=5.0,
        dynamic_ncols=False,
    )

    try:
        for step, batch in enumerate(progress, start=1):
            images = None
            masks = None
            logits = None
            loss = None
            scaled_loss = None

            try:
                images = batch["image"].to(
                    DEVICE,
                    non_blocking=PIN_MEMORY,
                )
                masks = batch["mask"].to(
                    DEVICE,
                    non_blocking=PIN_MEMORY,
                )

                group_start = (
                    (step - 1) // gradient_accumulation_steps
                ) * gradient_accumulation_steps
                group_size = min(
                    gradient_accumulation_steps,
                    len(loader) - group_start,
                )

                with autocast_context():
                    logits = model(images)
                    loss = criterion(logits, masks)
                    scaled_loss = loss / group_size

                if amp_enabled():
                    scaler.scale(scaled_loss).backward()
                else:
                    scaled_loss.backward()

                should_step = (
                    step % gradient_accumulation_steps == 0
                    or step == len(loader)
                )

                if should_step:
                    if amp_enabled():
                        scaler.step(optimizer)
                        scaler.update()
                    else:
                        optimizer.step()

                    optimizer.zero_grad(set_to_none=True)

                valid_pixels = int(
                    (masks != IGNORE_INDEX).sum().item()
                )
                total_loss += (
                    float(loss.item()) * max(valid_pixels, 1)
                )
                total_valid_pixels += valid_pixels
                accumulator.update(
                    logits.detach(),
                    masks,
                )

                if (
                    step % PROGRESS_UPDATE_EVERY == 0
                    or step == len(loader)
                ):
                    metrics, _ = metrics_from_confusion(
                        accumulator.matrix.numpy()
                    )
                    progress.set_postfix(
                        loss=(
                            f"{total_loss / max(total_valid_pixels, 1):.4f}"
                        ),
                        mIoU=f"{metrics['mean_iou']:.4f}",
                        refresh=False,
                    )

            except RuntimeError as error:
                optimizer.zero_grad(set_to_none=True)

                if is_gpu_oom(error):
                    gpu_cleanup(synchronize=False)
                    raise RuntimeError(
                        "Se detectó falta de memoria de GPU durante el "
                        f"entrenamiento (época {epoch}, lote {step}). "
                        "El batch ya está en 1. Mantén "
                        "USE_DENSENET_MEMORY_EFFICIENT=True y USE_AMP=False; "
                        "si persiste, reduce el tamaño de los chips."
                    ) from error

                raise

            finally:
                del images, masks, logits, loss, scaled_loss

            if (
                GPU_CLEANUP_EVERY_STEPS > 0
                and step % GPU_CLEANUP_EVERY_STEPS == 0
            ):
                gpu_cleanup(synchronize=False)

    finally:
        progress.close()

    metrics, _ = metrics_from_confusion(
        accumulator.matrix.numpy()
    )
    metrics["loss"] = (
        total_loss / max(total_valid_pixels, 1)
    )
    return metrics


@torch.no_grad()
def evaluate(
    model,
    loader,
    criterion,
    split_name,
    num_classes,
):
    model.eval()
    accumulator = ConfusionAccumulator(num_classes)
    total_loss = 0.0
    total_valid_pixels = 0

    progress = tqdm(
        loader,
        desc=split_name,
        leave=False,
        mininterval=5.0,
        dynamic_ncols=False,
    )

    try:
        for step, batch in enumerate(progress, start=1):
            images = None
            masks = None
            logits = None
            loss = None

            try:
                images = batch["image"].to(
                    DEVICE,
                    non_blocking=PIN_MEMORY,
                )
                masks = batch["mask"].to(
                    DEVICE,
                    non_blocking=PIN_MEMORY,
                )

                with autocast_context():
                    logits = model(images)
                    loss = criterion(logits, masks)

                valid_pixels = int(
                    (masks != IGNORE_INDEX).sum().item()
                )
                total_loss += (
                    float(loss.item()) * max(valid_pixels, 1)
                )
                total_valid_pixels += valid_pixels
                accumulator.update(
                    logits,
                    masks,
                )

                if (
                    step % PROGRESS_UPDATE_EVERY == 0
                    or step == len(loader)
                ):
                    metrics, _ = metrics_from_confusion(
                        accumulator.matrix.numpy()
                    )
                    progress.set_postfix(
                        loss=(
                            f"{total_loss / max(total_valid_pixels, 1):.4f}"
                        ),
                        mIoU=f"{metrics['mean_iou']:.4f}",
                        refresh=False,
                    )

            except RuntimeError as error:
                if is_gpu_oom(error):
                    gpu_cleanup(synchronize=False)
                    raise RuntimeError(
                        "Se agotó la memoria de GPU durante "
                        f"{split_name}. El batch está configurado en 1."
                    ) from error
                raise

            finally:
                del images, masks, logits, loss

    finally:
        progress.close()

    metrics, per_class = metrics_from_confusion(
        accumulator.matrix.numpy()
    )
    metrics["loss"] = (
        total_loss / max(total_valid_pixels, 1)
    )
    return (
        metrics,
        per_class,
        accumulator.matrix.numpy(),
    )

def atomic_torch_save(payload, path: str | Path) -> None:
    path = Path(path)
    ensure_dir(path.parent)
    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, temporary)
    os.replace(temporary, path)


def save_checkpoint(
    path,
    epoch,
    model,
    optimizer,
    scheduler,
    scaler,
    config,
    validation_metrics,
    best_value,
    best_epoch,
    patience,
):
    """
    Guarda únicamente los pesos y metadatos esenciales.

    Se omiten los estados de AdamW, scheduler y GradScaler para evitar un
    incremento brusco de RAM al finalizar cada época en Windows/ROCm.
    """
    cpu_state = {
        key: value.detach().cpu()
        for key, value in model.state_dict().items()
    }
    atomic_torch_save(
        {
            "epoch": int(epoch),
            "model_state_dict": cpu_state,
            "config": config,
            "validation_metrics": validation_metrics,
            "best_val_iou": float(best_value),
            "best_epoch": int(best_epoch),
            "patience": int(patience),
            "saved_at": datetime.now().isoformat(timespec="seconds"),
            "checkpoint_type": "model_only",
        },
        path,
    )
    del cpu_state
    gc.collect()

def load_checkpoint_if_needed(
    last_checkpoint,
    model,
    optimizer,
    scheduler,
    scaler,
    auto_resume,
):
    if not auto_resume or not Path(last_checkpoint).exists():
        return 1, float("-inf"), 0, 0

    try:
        checkpoint = torch.load(last_checkpoint, map_location="cpu", weights_only=False)
    except TypeError:
        checkpoint = torch.load(last_checkpoint, map_location="cpu")

    model.load_state_dict(checkpoint["model_state_dict"], strict=True)

    if checkpoint.get("checkpoint_type") == "model_only":
        warnings.warn(
            "Se reanudaron los pesos de DenseNet-121, pero AdamW, scheduler "
            "y GradScaler comienzan con estado nuevo para reducir la RAM.",
            RuntimeWarning,
        )

    for key, target in (
        ("optimizer_state_dict", optimizer),
        ("scheduler_state_dict", scheduler),
    ):
        state = checkpoint.get(key)
        if state:
            try:
                target.load_state_dict(state)
            except Exception as exc:
                warnings.warn(f"No se restauró {key}: {exc}", RuntimeWarning)

    scaler_state = checkpoint.get("scaler_state_dict")
    if scaler is not None and isinstance(scaler_state, dict) and scaler_state:
        try:
            scaler.load_state_dict(scaler_state)
        except Exception as exc:
            warnings.warn(f"No se restauró GradScaler: {exc}", RuntimeWarning)

    return (
        int(checkpoint.get("epoch", 0)) + 1,
        float(checkpoint.get("best_val_iou", float("-inf"))),
        int(checkpoint.get("best_epoch", 0)),
        int(checkpoint.get("patience", 0)),
    )


def plot_history(history: pd.DataFrame, output_dir: Path) -> None:
    if history.empty:
        return

    output_dir = ensure_dir(output_dir)
    pairs = [
        ("train_loss", "val_loss", "Pérdida"),
        ("train_pixel_accuracy", "val_pixel_accuracy", "Exactitud por píxel"),
        ("train_mean_iou", "val_mean_iou", "IoU media"),
        ("train_macro_f1", "val_macro_f1", "F1 macro"),
        ("train_kappa", "val_kappa", "Kappa"),
    ]

    for train_column, val_column, title in pairs:
        plt.figure(figsize=(9, 5))
        if train_column in history:
            plt.plot(history["epoch"], history[train_column], label="Entrenamiento")
        if val_column in history:
            plt.plot(history["epoch"], history[val_column], label="Validación")
        plt.xlabel("Época")
        plt.ylabel(title)
        plt.title(title)
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        plt.savefig(output_dir / f"{safe_name(title)}.png", dpi=180)
        plt.close()


def plot_confusion_matrix(matrix, idx_to_name, path):
    labels = list(range(matrix.shape[0]))
    names = [idx_to_name[index] for index in labels]

    plt.figure(figsize=(10, 8))
    plt.imshow(matrix, interpolation="nearest")
    plt.colorbar()
    ticks = np.arange(len(labels))
    plt.xticks(ticks, names, rotation=90)
    plt.yticks(ticks, names)
    plt.xlabel("Predicción")
    plt.ylabel("Referencia")
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()


def save_test_predictions(
    model,
    loader,
    output_dir,
    max_items,
    idx_to_code: dict[int, int],
):
    output_dir = ensure_dir(output_dir)
    model.eval()
    saved = 0
    lookup = np.asarray(
        [idx_to_code[index] for index in range(len(idx_to_code))],
        dtype=np.uint8,
    )

    with torch.no_grad():
        for batch in tqdm(loader, desc="Predicciones de prueba"):
            images = batch["image"].to(DEVICE)
            with autocast_context():
                internal_predictions = model(images).argmax(dim=1).cpu().numpy()

            for index, internal_prediction in enumerate(internal_predictions):
                if saved >= max_items:
                    return

                image_path = batch["image_path"][index]
                prediction = lookup[internal_prediction]
                _, valid_pixel = read_reflectance(image_path)
                prediction = prediction.copy()
                prediction[~valid_pixel] = MASK_NODATA_CODE

                with rasterio.open(image_path) as src:
                    profile = src.profile.copy()
                profile.update(
                    count=1,
                    dtype="uint8",
                    nodata=MASK_NODATA_CODE,
                    compress="lzw",
                )

                output = output_dir / f"prediccion_test_{saved:04d}.tif"
                with rasterio.open(output, "w", **profile) as dst:
                    dst.write(prediction, 1)
                saved += 1


def write_markdown_report(summary, config, path):
    lines = ["# Informe de entrenamiento DenseNet121-UNet\n", "## Configuración\n"]
    for key, value in config.items():
        lines.append(f"- **{key}**: {value}")

    lines.append("\n## Resultados de prueba\n")
    for key, value in summary.items():
        formatted = f"{float(value):.6f}" if isinstance(value, (int, float, np.integer, np.floating)) else str(value)
        lines.append(f"- **{key}**: {formatted}")

    path = Path(path)
    ensure_dir(path.parent)
    path.write_text("\n".join(lines), encoding="utf-8")

def find_completed_run(level_name: str):
    """
    Devuelve la ejecución final más reciente de un nivel, si existe.
    Permite reiniciar Jupyter y continuar sin volver a entrenar un nivel terminado.
    """
    if not RUNS_DIR.exists():
        return None

    candidates = []

    for run_dir in RUNS_DIR.glob(
        f"DenseNet121Seg_{level_name}_*"
    ):
        marker = (
            run_dir / "LEVEL_COMPLETED.json"
        )
        report = (
            run_dir
            / "reports"
            / "resumen_metricas_test.json"
        )
        checkpoint = (
            run_dir
            / "checkpoints"
            / "last_checkpoint.pt"
        )

        completed = (
            marker.exists()
            or report.exists()
        )

        if (
            not completed
            and checkpoint.exists()
        ):
            try:
                try:
                    state = torch.load(
                        checkpoint,
                        map_location="cpu",
                        weights_only=False,
                    )
                except TypeError:
                    state = torch.load(
                        checkpoint,
                        map_location="cpu",
                    )

                checkpoint_epoch = int(
                    state.get("epoch", 0)
                )
                checkpoint_patience = int(
                    state.get("patience", 0)
                )

                completed = (
                    checkpoint_epoch >= EPOCHS
                    or checkpoint_patience
                    >= EARLY_STOPPING_PATIENCE
                )

                del state
                gc.collect()

            except Exception as error:
                print(
                    "No fue posible revisar el checkpoint de "
                    f"{run_dir.name}: {error}"
                )

        if completed:
            evidence_times = [
                path.stat().st_mtime
                for path in (
                    marker,
                    report,
                    checkpoint,
                )
                if path.exists()
            ]
            modified = (
                max(evidence_times)
                if evidence_times
                else 0
            )
            candidates.append(
                (modified, run_dir)
            )

    if not candidates:
        return None

    return max(
        candidates,
        key=lambda item: item[0],
    )[1]


def mark_level_completed(
    level_name: str,
    run_dir: Path,
    best_epoch: int,
    best_val_iou: float,
):
    save_json(
        {
            "completed": True,
            "level": level_name,
            "run_dir": str(run_dir),
            "best_epoch": int(best_epoch),
            "best_val_iou": float(best_val_iou),
            "completed_at": datetime.now().isoformat(
                timespec="seconds"
            ),
        },
        Path(run_dir) / "LEVEL_COMPLETED.json",
    )


def build_run_dir(level_name: str):
    forced = RESUME_BY_LEVEL.get(level_name)

    if forced:
        experiment_name = forced
        auto_resume = True
    elif CREATE_NEW_RUNS:
        experiment_name = f"DenseNet121Seg_{level_name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        auto_resume = False
    else:
        candidates = []
        if RUNS_DIR.exists():
            for run_dir in RUNS_DIR.glob(f"DenseNet121Seg_{level_name}_*"):
                checkpoint = run_dir / "checkpoints" / "last_checkpoint.pt"
                if checkpoint.exists():
                    candidates.append((checkpoint.stat().st_mtime, run_dir.name))

        if candidates:
            experiment_name = max(candidates)[1]
            auto_resume = True
        else:
            experiment_name = f"DenseNet121Seg_{level_name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
            auto_resume = False

    run_dir = ensure_dir(RUNS_DIR / experiment_name)
    checkpoint_dir = ensure_dir(run_dir / "checkpoints")
    report_dir = ensure_dir(run_dir / "reports")
    prediction_dir = ensure_dir(run_dir / "test_predictions")
    return run_dir, checkpoint_dir, report_dir, prediction_dir, experiment_name, auto_resume


def train_level(level_name: str):
    batch_size = int(BATCH_SIZE_BY_LEVEL[level_name])
    gradient_accumulation_steps = int(
        GRADIENT_ACCUMULATION_BY_LEVEL[level_name]
    )
    print(
        f"\n{level_name}: batch={batch_size}, "
        f"acumulación={gradient_accumulation_steps}, "
        f"batch efectivo={batch_size * gradient_accumulation_steps}"
    )
    class_dict = get_class_dict(level_name)
    thematic_codes = sorted(code for code in class_dict if code != MASK_NODATA_CODE)
    code_to_idx = {code: index for index, code in enumerate(thematic_codes)}
    idx_to_code = {index: code for code, index in code_to_idx.items()}
    idx_to_name = {
        index: class_dict[code]
        for index, code in idx_to_code.items()
    }
    num_classes = len(code_to_idx)

    train_df, val_df, test_df, splits_dir, band_count, chip_shape = read_splits(
        level_name,
        class_dict,
    )

    means, stds = compute_image_stats(train_df)
    pixel_counts = compute_class_pixel_counts(train_df, code_to_idx)
    class_weights = (
        make_class_weights(pixel_counts).to(DEVICE)
        if USE_CLASS_WEIGHTS
        else None
    )
    sampler = build_weighted_sampler(train_df)

    if USE_CLASS_WEIGHTS and USE_WEIGHTED_SAMPLER:
        warnings.warn(
            "Se están aplicando simultáneamente pesos de clase y muestreo balanceado.",
            RuntimeWarning,
        )

    train_loader = DataLoader(
        LULCSegmentationDataset(
            train_df,
            code_to_idx,
            means,
            stds,
            augment=USE_AUGMENTATION,
        ),
        batch_size=batch_size,
        shuffle=sampler is None,
        sampler=sampler,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=PERSISTENT_WORKERS and NUM_WORKERS > 0,
        prefetch_factor=PREFETCH_FACTOR if NUM_WORKERS > 0 else None,
    )
    val_loader = DataLoader(
        LULCSegmentationDataset(val_df, code_to_idx, means, stds, augment=False),
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=PERSISTENT_WORKERS and NUM_WORKERS > 0,
        prefetch_factor=PREFETCH_FACTOR if NUM_WORKERS > 0 else None,
    )
    test_loader = DataLoader(
        LULCSegmentationDataset(test_df, code_to_idx, means, stds, augment=False),
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=PERSISTENT_WORKERS and NUM_WORKERS > 0,
        prefetch_factor=PREFETCH_FACTOR if NUM_WORKERS > 0 else None,
    )

    model, pretrained_loaded = build_segmentation_model(
        in_channels=len(BAND_INDEXES),
        num_classes=num_classes,
        pretrained=ENCODER_PRETRAINED,
        freeze_encoder=FREEZE_ENCODER,
    )
    model = model.to(DEVICE)

    print("Validando la arquitectura en la GPU...")
    first_batch = next(iter(train_loader))["image"].to(
        DEVICE,
        non_blocking=PIN_MEMORY,
    )
    model.eval()
    with torch.inference_mode():
        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=USE_AMP and DEVICE.type == "cuda",
        ):
            output = model(first_batch[:1])
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    model.train()
    print("Arquitectura validada. Iniciando entrenamiento.")

    expected_output = (1, num_classes, first_batch.shape[-2], first_batch.shape[-1])
    if tuple(output.shape) != expected_output:
        raise RuntimeError(
            f"Salida inesperada del modelo: {tuple(output.shape)}; "
            f"se esperaba {expected_output}."
        )

    del first_batch, output
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    if OPTIMIZER_TYPE == "adamw":
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=LEARNING_RATE,
            weight_decay=WEIGHT_DECAY,
        )
    elif OPTIMIZER_TYPE == "sgd":
        optimizer = torch.optim.SGD(
            model.parameters(),
            lr=LEARNING_RATE,
            momentum=0.99,
            weight_decay=WEIGHT_DECAY,
        )
    else:
        raise ValueError(f"OPTIMIZER_TYPE no reconocido: {OPTIMIZER_TYPE}")

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=8,
        min_lr=1e-6,
    )
    criterion = CombinedSegmentationLoss(num_classes, class_weights)
    scaler = make_grad_scaler()

    (
        run_dir,
        checkpoint_dir,
        report_dir,
        prediction_dir,
        experiment_name,
        auto_resume,
    ) = build_run_dir(level_name)

    config = {
        "model": "DenseNet121-UNet",
        "encoder": "densenet121",
        "task": "segmentacion_semantica",
        "level": level_name,
        "dataset_root": str(DATASET_ROOT),
        "splits_dir": str(splits_dir),
        "run_dir": str(run_dir),
        "experiment_name": experiment_name,
        "band_indexes": list(BAND_INDEXES),
        "band_order": ["B2", "B3", "B4", "B8"],
        "reflectance_scale": REFLECTANCE_SCALE,
        "reflectance_add_offset": REFLECTANCE_ADD_OFFSET,
        "use_raster_scale_offset": USE_RASTER_SCALE_OFFSET,
        "auto_scale_reflectance": AUTO_SCALE_REFLECTANCE,
        "band_count": band_count,
        "chip_shape": chip_shape,
        "num_classes": num_classes,
        "mask_nodata_code": MASK_NODATA_CODE,
        "ignore_index": IGNORE_INDEX,
        "code_to_idx": code_to_idx,
        "idx_to_code": idx_to_code,
        "idx_to_name": idx_to_name,
        "means": means,
        "stds": stds,
        "encoder_pretrained_requested": ENCODER_PRETRAINED,
        "encoder_pretrained_loaded": pretrained_loaded,
        "freeze_encoder": FREEZE_ENCODER,
        "decoder_dropout": DECODER_DROPOUT,
        "densenet_memory_efficient": USE_DENSENET_MEMORY_EFFICIENT,
        "use_amp": USE_AMP,
        "cpu_threads": torch.get_num_threads(),
        "dataloader_workers": NUM_WORKERS,
        "optimizer": OPTIMIZER_TYPE,
        "loss": LOSS_TYPE,
        "dice_weight": DICE_WEIGHT,
        "epochs": EPOCHS,
        "batch_size": batch_size,
        "gradient_accumulation_steps": gradient_accumulation_steps,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "use_class_weights": USE_CLASS_WEIGHTS,
        "use_weighted_sampler": USE_WEIGHTED_SAMPLER,
        "use_augmentation": USE_AUGMENTATION,
        "seed": SEED,
    }
    save_json(config, run_dir / "config.json")
    save_dataset_summary(
        train_df,
        val_df,
        test_df,
        pixel_counts,
        idx_to_code,
        class_dict,
        report_dir / "dataset",
    )

    history_path = run_dir / "history.csv"
    history = (
        pd.read_csv(history_path).to_dict("records")
        if history_path.exists() and history_path.stat().st_size > 0
        else []
    )

    last_checkpoint = checkpoint_dir / "last_checkpoint.pt"
    start_epoch, best_value, best_epoch, patience = load_checkpoint_if_needed(
        last_checkpoint,
        model,
        optimizer,
        scheduler,
        scaler,
        auto_resume,
    )

    for epoch in range(start_epoch, EPOCHS + 1):
        train_metrics = train_one_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
            scaler,
            epoch,
            num_classes,
            gradient_accumulation_steps,
        )
        val_metrics, _, _ = evaluate(
            model,
            val_loader,
            criterion,
            f"Validación {level_name} época {epoch}",
            num_classes,
        )

        scheduler.step(val_metrics["mean_iou"])
        improved = val_metrics["mean_iou"] > best_value

        if improved:
            best_value = val_metrics["mean_iou"]
            best_epoch = epoch
            patience = 0
        else:
            patience += 1

        row = {"epoch": epoch, "learning_rate": optimizer.param_groups[0]["lr"]}
        row.update({f"train_{key}": value for key, value in train_metrics.items()})
        row.update({f"val_{key}": value for key, value in val_metrics.items()})
        history.append(row)

        history_df = (
            pd.DataFrame(history)
            .drop_duplicates(subset=["epoch"], keep="last")
            .sort_values("epoch")
        )
        history_df.to_csv(history_path, index=False, encoding="utf-8-sig")

        print("Guardando checkpoint liviano...")
        save_checkpoint(
            last_checkpoint,
            epoch,
            model,
            optimizer,
            scheduler,
            scaler,
            config,
            val_metrics,
            best_value,
            best_epoch,
            patience,
        )

        if improved:
            shutil.copy2(last_checkpoint, run_dir / "best_model.pt")

        if SAVE_EVERY_EPOCH:
            shutil.copy2(
                last_checkpoint,
                checkpoint_dir / f"checkpoint_epoch_{epoch:03d}.pt",
            )

        print(
            f"[{level_name} | época {epoch}] "
            f"train_loss={train_metrics['loss']:.4f} "
            f"val_loss={val_metrics['loss']:.4f} "
            f"val_mIoU={val_metrics['mean_iou']:.4f} "
            f"mejor_mIoU={best_value:.4f} "
            f"paciencia={patience}/{EARLY_STOPPING_PATIENCE}"
        )

        if DEVICE.type == "cuda":
            allocated = torch.cuda.memory_allocated() / 1024**3
            reserved = torch.cuda.memory_reserved() / 1024**3
            peak = torch.cuda.max_memory_allocated() / 1024**3
            print(
                f"VRAM: asignada={allocated:.2f} GB, "
                f"reservada={reserved:.2f} GB, pico={peak:.2f} GB"
            )

        gpu_cleanup(synchronize=True)

        if DEVICE.type == "cuda":
            try:
                torch.cuda.reset_peak_memory_stats()
            except Exception:
                pass

        if EPOCH_COOLDOWN_SECONDS > 0:
            print(
                f"Pausa de estabilización: "
                f"{EPOCH_COOLDOWN_SECONDS} segundos."
            )
            time.sleep(EPOCH_COOLDOWN_SECONDS)

        if patience >= EARLY_STOPPING_PATIENCE:
            break

    if PLOT_AFTER_TRAINING:
        final_history_df = (
            pd.DataFrame(history)
            .drop_duplicates(subset=["epoch"], keep="last")
            .sort_values("epoch")
        )
        plot_history(final_history_df, report_dir)

    best_path = run_dir / "best_model.pt"
    checkpoint_path = best_path if best_path.exists() else last_checkpoint
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"No se generó un checkpoint en {run_dir}")

    try:
        checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    except TypeError:
        checkpoint = torch.load(checkpoint_path, map_location="cpu")
    model.load_state_dict(checkpoint["model_state_dict"], strict=True)

    test_metrics, per_class, confusion = evaluate(
        model,
        test_loader,
        criterion,
        f"Prueba {level_name}",
        num_classes,
    )

    per_class = per_class.rename(columns={"class_code": "class_index"})
    per_class["class_code"] = per_class["class_index"].map(idx_to_code)
    per_class["class_name"] = per_class["class_index"].map(idx_to_name)
    per_class.to_csv(
        report_dir / "metricas_por_clase_test.csv",
        index=False,
        encoding="utf-8-sig",
    )

    labels = list(range(num_classes))
    names = [idx_to_name[index] for index in labels]
    pd.DataFrame(
        confusion,
        index=names,
        columns=names,
    ).to_csv(
        report_dir / "matriz_confusion_test.csv",
        encoding="utf-8-sig",
    )

    if SAVE_REPORT_PLOTS:
        plot_confusion_matrix(
            confusion,
            idx_to_name,
            report_dir / "matriz_confusion_test.png",
        )

    if SAVE_TEST_PREDICTIONS:
        save_test_predictions(
            model,
            test_loader,
            prediction_dir,
            MAX_TEST_PREDICTIONS_TO_SAVE,
            idx_to_code,
        )

    summary = {
        **test_metrics,
        "train_level": level_name,
        "run_dir": str(run_dir),
        "best_epoch": int(best_epoch),
        "best_val_iou": float(best_value),
        "completed": True,
    }
    save_json(
        summary,
        report_dir / "resumen_metricas_test.json",
    )
    mark_level_completed(
        level_name=level_name,
        run_dir=run_dir,
        best_epoch=best_epoch,
        best_val_iou=best_value,
    )
    write_markdown_report(
        summary,
        config,
        report_dir / "informe_entrenamiento_densenet121_segmentacion.md",
    )

    del model
    gpu_cleanup(synchronize=False)

    return summary


def run_all_levels():
    """
    Entrena Nivel 1 y Nivel 2 secuencialmente.

    Si Jupyter se reinicia, al volver a ejecutar la celda:
    - un nivel terminado se omite;
    - un nivel incompleto se reanuda desde last_checkpoint.pt.
    """
    set_seed(SEED)
    ensure_dir(RUNS_DIR)

    summaries = []

    for position, level_name in enumerate(
        LEVELS_TO_TRAIN,
        start=1,
    ):
        print("\n" + "=" * 88)
        print(
            f"REVISANDO {level_name} "
            f"({position}/{len(LEVELS_TO_TRAIN)})"
        )
        print("=" * 88)

        completed_run = (
            find_completed_run(level_name)
            if SKIP_COMPLETED_LEVELS
            else None
        )

        if completed_run is not None:
            print(
                f"{level_name} ya está terminado en:\n"
                f"{completed_run}\n"
                "Se omitirá y se continuará con el "
                "siguiente nivel."
            )
            summaries.append(
                {
                    "train_level": level_name,
                    "status": "already_completed",
                    "run_dir": str(completed_run),
                }
            )
            continue

        print(
            f"{level_name} no tiene una ejecución final "
            "completa. El entrenamiento se iniciará o "
            "se reanudarán los pesos desde el último "
            "checkpoint disponible."
        )

        try:
            summary = train_level(level_name)
            summary["status"] = "completed"
            summaries.append(summary)

        except KeyboardInterrupt:
            print(
                f"\nEntrenamiento interrumpido durante "
                f"{level_name}. El último checkpoint "
                "guardado se conserva."
            )
            summaries.append(
                {
                    "train_level": level_name,
                    "status": "interrupted",
                }
            )
            break

        except Exception as error:
            print(
                f"\nSe produjo un error durante "
                f"{level_name}: {error}\n"
                "El último checkpoint terminado se "
                "conserva."
            )
            traceback.print_exc()

            summaries.append(
                {
                    "train_level": level_name,
                    "status": "error",
                    "error": str(error),
                }
            )

            # Ante un error de GPU no se inicia inmediatamente el siguiente
            # nivel; primero se libera memoria y se detiene esta ejecución.
            gpu_cleanup(synchronize=False)
            break

        gpu_cleanup(synchronize=True)

        if LEVEL_COOLDOWN_SECONDS > 0:
            print(
                f"\n{level_name} terminó. Pausa de "
                f"{LEVEL_COOLDOWN_SECONDS} segundos "
                "antes de continuar."
            )
            time.sleep(LEVEL_COOLDOWN_SECONDS)

    summary_df = pd.DataFrame(summaries)

    output = RUNS_DIR / (
        "resumen_densenet121_segmentacion_"
        f"{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    )
    summary_df.to_csv(
        output,
        index=False,
        encoding="utf-8-sig",
    )

    print("\n" + "=" * 88)
    print("RESUMEN GENERAL")
    print("=" * 88)
    print(summary_df)
    print(f"\nArchivo de resumen: {output}")

    return summary_df


def main():
    print("=" * 88)
    print(
        "DENSENET-121 + DECODIFICADOR U-NET "
        "PARA SEGMENTACIÓN SEMÁNTICA"
    )
    print(
        "ENTRENAMIENTO PÍXEL A PÍXEL — "
        "NIVEL 1 Y NIVEL 2"
    )
    print("=" * 88)

    if REQUIRE_CUDA and DEVICE.type != "cuda":
        raise RuntimeError(
            "La GPU no está disponible para PyTorch. "
            "En ROCm la GPU AMD también debe aparecer "
            "como torch.cuda. Cambia REQUIRE_CUDA a "
            "False únicamente si deseas entrenamiento "
            "por CPU."
        )

    print(f"Dispositivo: {DEVICE}")
    print(f"PyTorch: {torch.__version__}")
    print(f"ROCm/HIP: {torch.version.hip}")
    print(f"Hilos PyTorch CPU: {torch.get_num_threads()}")
    print(f"DataLoader workers: {NUM_WORKERS}")
    print(
        "CPU: lectura Rasterio, normalización, "
        "aumentos, métricas y checkpoints."
    )

    if DEVICE.type == "cuda":
        print(
            f"GPU: {torch.cuda.get_device_name(0)}"
        )
        total_vram = (
            torch.cuda.get_device_properties(0)
            .total_memory / 1024**3
        )
        print(
            f"VRAM total reportada: "
            f"{total_vram:.2f} GB"
        )
        print(
            "GPU: forward, backward y actualización "
            "de parámetros."
        )

    print(
        f"AMP: {USE_AMP} | "
        "DenseNet memory_efficient: "
        f"{USE_DENSENET_MEMORY_EFFICIENT}"
    )
    print(
        f"Dataset: {DATASET_ROOT}\n"
        f"Modelos: {RUNS_DIR}"
    )

    return run_all_levels()


# Ejecución directa en una sola celda de Jupyter.
summary_df = main()


DENSENET-121 + DECODIFICADOR U-NET PARA SEGMENTACIÓN SEMÁNTICA
ENTRENAMIENTO PÍXEL A PÍXEL — NIVEL 1 Y NIVEL 2
Dispositivo: cuda
PyTorch: 2.9.1+rocm7.2.1
ROCm/HIP: 7.2.53211-158bd99533
Hilos PyTorch CPU: 4
DataLoader workers: 0
CPU: lectura Rasterio, normalización, aumentos, métricas y checkpoints.
GPU: AMD Radeon RX 9060 XT
VRAM total reportada: 15.92 GB
GPU: forward, backward y actualización de parámetros.
AMP: False | DenseNet memory_efficient: True
Dataset: C:\Users\Ivan sua\anaconda3\Dataset
Modelos: C:\Users\Ivan sua\anaconda3\Modelos_DenseNet121_Segmentacion

REVISANDO nivel_1 (1/2)
nivel_1 no tiene una ejecución final completa. El entrenamiento se iniciará o se reanudarán los pesos desde el último checkpoint disponible.

nivel_1: batch=1, acumulación=4, batch efectivo=4


Validando train:   0%|          | 0/1017 [00:00<?, ?it/s]

Validando val:   0%|          | 0/218 [00:00<?, ?it/s]

Validando test:   0%|          | 0/219 [00:00<?, ?it/s]

Estadísticas radiométricas:   0%|          | 0/300 [00:00<?, ?it/s]

...

Entrenamiento 120:   0%|          | 0/3727 [00:00<?, ?it/s]

Validación nivel_2 época 120:   0%|          | 0/799 [00:00<?, ?it/s]

Guardando checkpoint liviano...
[nivel_2 | época 120] train_loss=0.5646 val_loss=1.2614 val_mIoU=0.5230 mejor_mIoU=0.5259 paciencia=3/20
VRAM: asignada=0.25 GB, reservada=0.45 GB, pico=0.42 GB
Pausa de estabilización: 3 segundos.


Prueba nivel_2:   0%|          | 0/799 [00:00<?, ?it/s]


nivel_2 terminó. Pausa de 20 segundos antes de continuar.

RESUMEN GENERAL
   pixel_accuracy  macro_precision  macro_recall  macro_f1  mean_iou  \
0        0.906824         0.882418      0.889432  0.885500  0.796565   
1        0.737256         0.643669      0.678158  0.654354  0.504580   

      kappa      loss train_level  \
0  0.824900  0.491612     nivel_1   
1  0.655708  1.290779     nivel_2   

                                             run_dir  best_epoch  \
0  C:\Users\Ivan sua\anaconda3\Modelos_DenseNet12...          75   
1  C:\Users\Ivan sua\anaconda3\Modelos_DenseNet12...         117   

   best_val_iou  completed     status  
0      0.742628       True  completed  
1      0.525949       True  completed  

Archivo de resumen: C:\Users\Ivan sua\anaconda3\Modelos_DenseNet121_Segmentacion\resumen_densenet121_segmentacion_20260816_112448.csv


In [2]:
from __future__ import annotations

# =============================================================================
# DENSENET121-UNET — PREDICCIÓN SEMÁNTICA PÍXEL A PÍXEL
# NIVEL 1 + NIVEL 2 CLC
#
# Diseñado para el mismo entrenamiento DenseNet121-UNet utilizado en este
# proyecto. La predicción:
#   - usa DenseNet121 como encoder + decoder tipo U-Net;
#   - conserva exactamente la grilla del ráster de entrada;
#   - usa CPU para Rasterio, preparación, métricas y escritura;
#   - por defecto usa CPU también para inferencia para proteger el kernel;
#   - permite activar GPU AMD/ROCm mediante PREDICTION_BACKEND="gpu_safe";
#   - procesa una sola tesela a la vez;
#   - NO mantiene un cubo global de probabilidades ni usa np.memmap gigante;
#   - escribe directamente el núcleo de cada tesela en los GeoTIFF;
#   - cierra y reabre los GeoTIFF por sesiones cortas para tolerar interrupciones;
#   - SIEMPRE genera una predicción nueva; no reutiliza ni reanuda;
#   - valida idx_to_code y armoniza su semántica numérica con CLC;
#   - estima la correspondencia con 20 % y evalúa en 80 % HOLDOUT;
#   - calcula métricas globales, por clase, matrices de confusión,
#     áreas, confianza, ECE/MCE y comparación entrenamiento-predicción.
#
# IMPORTANTE:
# Ejecutar TODO este archivo en una celda nueva de Jupyter, preferiblemente
# después de reiniciar el kernel, para que las variables de entorno de ROCm/
# MIOpen se apliquen antes de importar PyTorch.
# =============================================================================

# =============================================================================
# 0. RECURSOS — RYZEN 5 5600X, 16 GB RAM, GPU AMD 16 GB
# =============================================================================
import os

# -----------------------------------------------------------------------------
# MODO DE EJECUCIÓN
# -----------------------------------------------------------------------------
# "cpu_ultra_stable" = RECOMENDADO cuando el kernel se está cerrando.
#                       No inicializa la GPU durante la inferencia.
# "gpu_safe"          = usa la GPU con teselas pequeñas y FP32.
#                       Si el controlador AMD se reinicia a nivel nativo,
#                       Python/Jupyter no puede capturar ese fallo.
#
# IMPORTANTE: si cambias este valor, reinicia el kernel ANTES de ejecutar todo
# el archivo para que las variables de entorno se apliquen antes de PyTorch.
PREDICTION_BACKEND = "cpu_ultra_stable"

os.environ.setdefault("OMP_NUM_THREADS", "3")
os.environ.setdefault("MKL_NUM_THREADS", "3")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "3")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "3")
os.environ.setdefault("GDAL_NUM_THREADS", "1")
os.environ.setdefault("GDAL_CACHEMAX", "128")

if PREDICTION_BACKEND == "cpu_ultra_stable":
    # Oculta los aceleradores antes de importar PyTorch. Esta es la única forma
    # fiable de evitar que un reinicio del controlador AMD mate el kernel.
    os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
    os.environ["HIP_VISIBLE_DEVICES"] = "-1"
    os.environ["ROCR_VISIBLE_DEVICES"] = "-1"
else:
    # Configuración conservadora para ROCm/MIOpen.
    os.environ.setdefault("PYTORCH_MIOPEN_SUGGEST_NHWC_BATCHNORM", "0")
    os.environ.setdefault("MIOPEN_FIND_MODE", "FAST")
    os.environ.setdefault(
        "PYTORCH_ALLOC_CONF",
        "expandable_segments:True,max_split_size_mb:128",
    )

import gc
import json
import math
import time
import traceback
import warnings
from contextlib import nullcontext
from datetime import datetime
from pathlib import Path
from typing import Any, Mapping, Sequence

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import torch
import torch.nn as nn
import torch.nn.functional as F
from rasterio.enums import Resampling
from rasterio.vrt import WarpedVRT
from rasterio.windows import Window
from torchvision import models
from tqdm.auto import tqdm


# =============================================================================
# 1. RUTAS — YA AJUSTADAS AL PROYECTO
# =============================================================================

MODEL_NAME = "DenseNet121-UNet"

# Modelos generados por el entrenamiento DenseNet121.
MODEL_ROOT = Path(
    r"C:\Users\Ivan sua\anaconda3\Modelos_DenseNet121_Segmentacion"
)

# Imagen Sentinel-2 sobre la cual se realizará la predicción.
INPUT_RASTER = Path(
    r"C:\Users\Ivan sua\anaconda3\Rasters_CLC\zona_estudio.tif"
)

# Rásteres CLC suministrados para evaluar la predicción.
REFERENCE_BY_LEVEL = {
    "nivel_1": Path(
        r"C:\Users\Ivan sua\anaconda3\Rasters_CLC\Raster_Almeidas_Nivel1.tif"
    ),
    "nivel_2": Path(
        r"C:\Users\Ivan sua\anaconda3\Rasters_CLC\Raster_Almeidas_Nivel2.tif"
    ),
}

# Carpeta independiente para todos los productos de DenseNet121.
OUTPUT_DIR = Path(
    r"C:\Users\Ivan sua\anaconda3\Predicciones_DenseNet121_Segmentacion_VALIDADA_CPU_GPU"
)

LEVELS_TO_PREDICT = [
    "nivel_1",
    "nivel_2",
]

# None = seleccionar automáticamente el run final más reciente.
# También puede escribirse el nombre exacto de una carpeta, por ejemplo:
# "DenseNet121Seg_nivel_1_20260815_123456"
RUN_BY_LEVEL = {
    "nivel_1": None,
    "nivel_2": None,
}

# Predicción final: usar best_model.pt.
# Solo activar si se necesita recuperar un entrenamiento sin best_model.pt.
ALLOW_LAST_CHECKPOINT = False

# Esta versión SIEMPRE genera una predicción nueva.
# No reutiliza GeoTIFF finales ni estados parciales de ejecuciones anteriores.
FORCE_NEW_PREDICTION = True
REUSE_COMPLETED_PREDICTIONS = False
RESUME_PREDICTION = False

# =============================================================================
# 2. CLASES CLC Y VALIDACIÓN DE CÓDIGOS
# =============================================================================

REFERENCE_BACKGROUND_CODE = 0
OUTPUT_NODATA = 0

CLASES_NIVEL_1 = {
    0: "NoData",
    1: "2. Territorios agrícolas",
    2: "3. Bosques y áreas seminaturales",
    3: "5. Superficies de agua",
    4: "1. Territorios artificializados",
    5: "4. Áreas húmedas",
}

CLASES_NIVEL_2 = {
    0: "NoData",
    1: "2.4 Áreas agrícolas heterogéneas",
    2: "3.2 Áreas con vegetación herbácea o arbustiva",
    3: "3.1 Bosques",
    4: "2.3 Pastos",
    5: "5.1 Aguas continentales",
    6: "1.1 Zonas urbanizadas",
    7: "1.3 Zonas de extracción minera y escombreras",
    8: "4.1 Áreas húmedas continentales",
    9: "1.2 Zonas industriales o comerciales y redes de comunicación",
    10: "1.4 Zonas verdes artificializadas no agrícolas",
    11: "2.1 Cultivos transitorios",
    12: "2.2 Cultivos permanentes",
    13: "3.3 Áreas abiertas sin o con poca vegetación",
}

EXPECTED_MODEL_CODES_BY_LEVEL = {
    "nivel_1": [1, 2, 3, 4, 5],
    "nivel_2": list(range(1, 14)),
}

# Los rásteres suministrados fueron inspeccionados previamente:
# Nivel 1: 0..5
# Nivel 2: 0..12
# La clase 13 de Nivel 2 puede ser válida para el modelo aunque no exista
# en la referencia; por eso NO se elimina y NO se convierte en otra clase.
EXPECTED_REFERENCE_CODES_BY_LEVEL = {
    "nivel_1": [1, 2, 3, 4, 5],
    "nivel_2": list(range(1, 13)),
}


def get_class_dict(level_name: str) -> dict[int, str]:
    if level_name == "nivel_1":
        return CLASES_NIVEL_1
    if level_name == "nivel_2":
        return CLASES_NIVEL_2
    raise ValueError(
        f"Nivel no reconocido: {level_name}"
    )


# =============================================================================
# 3. CONFIGURACIÓN DE INFERENCIA Y ESTABILIDAD
# =============================================================================

CPU_THREADS = 3

torch.set_num_threads(CPU_THREADS)
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass

try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

if PREDICTION_BACKEND not in {
    "cpu_ultra_stable",
    "gpu_safe",
}:
    raise ValueError(
        "PREDICTION_BACKEND debe ser 'cpu_ultra_stable' o 'gpu_safe'."
    )

GPU_RUNTIME_ENABLED = (
    PREDICTION_BACKEND == "gpu_safe"
    and torch.cuda.is_available()
)

DEVICE = torch.device(
    "cuda" if GPU_RUNTIME_ENABLED else "cpu"
)

# FP32 es deliberado. Evita introducir mixed precision/MIOpen como otra fuente
# de inestabilidad durante la predicción.
USE_AMP = False

# Solo puede actuar frente a un OOM que PyTorch alcance a reportar. Un reinicio
# del driver AMD puede cerrar el proceso completo y no admite try/except.
FALLBACK_TO_CPU_ON_GPU_OOM = True

# Sin acumulación global de probabilidades. Cada tesela se predice con contexto
# solapado, pero solo se escribe su núcleo central; así cada píxel se escribe una
# sola vez y desaparecen los archivos probability_sum.dat / weight_sum.dat.
#
# En CPU se usan 96 px. En GPU segura se usan 64 px para reducir al mínimo
# el pico de VRAM.
if DEVICE.type == "cuda":
    TILE_SIZE_BY_LEVEL = {
        "nivel_1": 64,
        "nivel_2": 64,
    }
    OVERLAP_BY_LEVEL = {
        "nivel_1": 16,
        "nivel_2": 16,
    }
else:
    TILE_SIZE_BY_LEVEL = {
        "nivel_1": 96,
        "nivel_2": 96,
    }
    OVERLAP_BY_LEVEL = {
        "nivel_1": 32,
        "nivel_2": 32,
    }

# memory_efficient de torchvision solo aporta ahorro durante backward/checkpoint.
# En inferencia no es necesario y se deja False para recorrer la ruta más simple.
USE_DENSENET_MEMORY_EFFICIENT = False

# Cada sesión escribe un número pequeño de teselas y CIERRA los GeoTIFF antes
# de guardar el estado. Si Jupyter se interrumpe, al reanudar se repite como
# máximo una sesión y no se depende de un memmap de varios GB.
WRITE_SESSION_TILES = 12
CPU_GC_EVERY_TILES = 6

# Pausa entre niveles para liberar objetos y cachés del sistema.
LEVEL_COOLDOWN_SECONDS = 15

# Máscara externa opcional: 1 = válido, 0 = excluir.
VALIDITY_MASK_RASTER = None

# Salidas adicionales. Se desactivan imágenes por defecto para reducir RAM.
SAVE_QUICKLOOK = False
SAVE_CONFUSION_PLOT = False
SAVE_FULL_METRICS = True
CONFIDENCE_BINS = 10
GTIFF_BLOCK_SIZE = 128
QUICKLOOK_MAX_SIZE = 1200

TEMP_DIR = OUTPUT_DIR / "_temporales"

EVALUATION_NOTE = (
    "Las métricas comparan la predicción DenseNet121 con el ráster CLC "
    "de referencia en todos los píxeles válidos coincidentes. Cuando las "
    "fechas de la referencia y de la imagen difieren, los valores expresan "
    "concordancia espacial con esa referencia y no necesariamente exactitud "
    "temática contemporánea."
)


# =============================================================================
# 4. UTILIDADES GENERALES
# =============================================================================

def ensure_dir(path: str | Path) -> Path:
    path = Path(path)
    path.mkdir(
        parents=True,
        exist_ok=True,
    )
    return path


def json_ready(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, Mapping):
        return {
            str(key): json_ready(item)
            for key, item in value.items()
        }
    if isinstance(value, (list, tuple)):
        return [
            json_ready(item)
            for item in value
        ]
    if isinstance(value, float) and not np.isfinite(value):
        return None
    return value


def load_json(path: str | Path) -> dict:
    path = Path(path)
    with path.open(
        "r",
        encoding="utf-8",
    ) as file:
        return json.load(file)


def save_json(
    data: Mapping[str, Any],
    path: str | Path,
) -> Path:
    path = Path(path)
    ensure_dir(path.parent)

    temporary = path.with_suffix(
        path.suffix + ".tmp"
    )

    with temporary.open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            json_ready(dict(data)),
            file,
            indent=2,
            ensure_ascii=False,
        )

    temporary.replace(path)
    return path


def close_memmap(array) -> None:
    if array is None:
        return

    try:
        array.flush()
    except Exception:
        pass

    mmap_object = getattr(
        array,
        "_mmap",
        None,
    )

    if mmap_object is not None:
        try:
            mmap_object.close()
        except Exception:
            pass


def safe_unlink(
    path: str | Path,
    retries: int = 5,
    wait_seconds: float = 1.0,
) -> bool:
    path = Path(path)

    if not path.exists():
        return True

    last_error = None

    for _ in range(retries):
        try:
            path.unlink()
            return True
        except FileNotFoundError:
            return True
        except (PermissionError, OSError) as error:
            last_error = error
            gc.collect()
            time.sleep(wait_seconds)

    warnings.warn(
        f"No fue posible eliminar el temporal {path}. "
        f"Se conservará. Error: {last_error}",
        RuntimeWarning,
    )
    return False


def safe_cleanup(
    synchronize: bool = False,
) -> None:
    gc.collect()

    # En cpu_ultra_stable no se llama a ninguna API CUDA/HIP.
    if not GPU_RUNTIME_ENABLED:
        return

    try:
        if synchronize:
            torch.cuda.synchronize()
    except Exception:
        pass

    try:
        torch.cuda.empty_cache()
    except Exception:
        pass


def accelerator_name() -> str:
    if DEVICE.type != "cuda":
        return "CPU"

    try:
        return str(
            torch.cuda.get_device_name(0)
        )
    except Exception:
        return "GPU AMD/ROCm"


def print_resource_status() -> None:
    print(
        f"Dispositivo inicial: {DEVICE}"
    )
    print(
        f"PyTorch: {torch.__version__}"
    )
    print(
        f"Backend solicitado: {PREDICTION_BACKEND}"
    )
    print(
        f"Hilos CPU: {torch.get_num_threads()}"
    )

    if DEVICE.type == "cuda":
        print(
            f"ROCm/HIP: {torch.version.hip}"
        )
        try:
            properties = (
                torch.cuda.get_device_properties(0)
            )
            total_vram = (
                properties.total_memory / 1024**3
            )

            print(
                f"Acelerador: {accelerator_name()}"
            )
            print(
                f"VRAM total: {total_vram:.2f} GB"
            )
        except Exception as error:
            print(
                "No fue posible consultar toda la "
                f"información GPU: {error}"
            )


def autocast_context(
    device: torch.device,
):
    if USE_AMP and device.type == "cuda":
        return torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        )
    return nullcontext()


def is_gpu_oom(error: BaseException) -> bool:
    message = str(error).lower()

    tokens = (
        "out of memory",
        "hip out of memory",
        "cuda out of memory",
        "memory allocation",
    )

    return any(
        token in message
        for token in tokens
    )


# =============================================================================
# 5. ARQUITECTURA DENSENET121-UNET
#    DEBE COINCIDIR CON EL ENTRENAMIENTO
# =============================================================================

class ConvNormAct(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        dropout: float = 0.0,
    ):
        super().__init__()

        groups = min(
            32,
            out_channels,
        )

        while out_channels % groups != 0:
            groups -= 1

        layers = [
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            nn.GroupNorm(
                groups,
                out_channels,
            ),
            nn.ReLU(inplace=True),
            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            nn.GroupNorm(
                groups,
                out_channels,
            ),
            nn.ReLU(inplace=True),
        ]

        if dropout > 0:
            layers.insert(
                3,
                nn.Dropout2d(dropout),
            )

        self.block = nn.Sequential(
            *layers
        )

    def forward(self, x):
        return self.block(x)


class DecoderBlock(nn.Module):
    def __init__(
        self,
        in_channels: int,
        skip_channels: int,
        out_channels: int,
        dropout: float = 0.0,
    ):
        super().__init__()

        self.conv = ConvNormAct(
            in_channels + skip_channels,
            out_channels,
            dropout,
        )

    def forward(
        self,
        decoder,
        skip,
    ):
        decoder = F.interpolate(
            decoder,
            size=skip.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )

        return self.conv(
            torch.cat(
                [decoder, skip],
                dim=1,
            )
        )


def _valid_group_count(
    channels: int,
    maximum_groups: int = 32,
) -> int:
    for groups in (
        maximum_groups,
        16,
        8,
        4,
        2,
        1,
    ):
        if (
            groups <= channels
            and channels % groups == 0
        ):
            return groups

    return 1


def replace_batchnorm_with_groupnorm(
    module: nn.Module,
) -> nn.Module:
    """
    Replica exactamente la modificación aplicada en el entrenamiento.
    """
    for name, child in list(
        module.named_children()
    ):
        if isinstance(
            child,
            nn.BatchNorm2d,
        ):
            replacement = nn.GroupNorm(
                num_groups=_valid_group_count(
                    child.num_features
                ),
                num_channels=child.num_features,
                eps=child.eps,
                affine=child.affine,
            )

            if child.affine:
                with torch.no_grad():
                    replacement.weight.copy_(
                        child.weight
                    )
                    replacement.bias.copy_(
                        child.bias
                    )

            setattr(
                module,
                name,
                replacement,
            )
        else:
            replace_batchnorm_with_groupnorm(
                child
            )

    return module


def adapt_first_convolution(
    old_conv: nn.Conv2d,
    in_channels: int,
    pretrained_loaded: bool,
) -> nn.Conv2d:
    new_conv = nn.Conv2d(
        in_channels,
        old_conv.out_channels,
        kernel_size=old_conv.kernel_size,
        stride=old_conv.stride,
        padding=old_conv.padding,
        bias=False,
    )

    with torch.no_grad():
        if pretrained_loaded:
            if in_channels >= 3:
                new_conv.weight[:, 0] = (
                    old_conv.weight[:, 2]
                )
                new_conv.weight[:, 1] = (
                    old_conv.weight[:, 1]
                )
                new_conv.weight[:, 2] = (
                    old_conv.weight[:, 0]
                )

                if in_channels > 3:
                    mean_weight = (
                        old_conv.weight.mean(
                            dim=1,
                            keepdim=True,
                        )
                    )

                    new_conv.weight[:, 3:] = (
                        mean_weight.repeat(
                            1,
                            in_channels - 3,
                            1,
                            1,
                        )
                    )
            else:
                new_conv.weight[
                    :,
                    :in_channels,
                ] = old_conv.weight[
                    :,
                    :in_channels,
                ]
        else:
            nn.init.kaiming_normal_(
                new_conv.weight,
                mode="fan_out",
                nonlinearity="relu",
            )

    return new_conv


class DenseNet121UNet(nn.Module):
    """
    Segmentación semántica con DenseNet-121 como encoder
    y decoder tipo U-Net.
    """

    def __init__(
        self,
        in_channels: int,
        num_classes: int,
        pretrained: bool = False,
        freeze_encoder: bool = False,
        decoder_dropout: float = 0.10,
        memory_efficient: bool = (
            USE_DENSENET_MEMORY_EFFICIENT
        ),
    ):
        super().__init__()

        pretrained_loaded = False

        if pretrained:
            try:
                weights = (
                    models.DenseNet121_Weights.DEFAULT
                )

                encoder = models.densenet121(
                    weights=weights,
                    memory_efficient=memory_efficient,
                )

                pretrained_loaded = True

            except Exception as error:
                warnings.warn(
                    "No fue posible cargar pesos "
                    "ImageNet de DenseNet121 "
                    f"({error}).",
                    RuntimeWarning,
                )

                encoder = models.densenet121(
                    weights=None,
                    memory_efficient=memory_efficient,
                )
        else:
            encoder = models.densenet121(
                weights=None,
                memory_efficient=memory_efficient,
            )

        encoder = (
            replace_batchnorm_with_groupnorm(
                encoder
            )
        )

        features = encoder.features

        features.conv0 = (
            adapt_first_convolution(
                features.conv0,
                in_channels,
                pretrained_loaded,
            )
        )

        self.conv0 = features.conv0
        self.norm0 = features.norm0
        self.relu0 = features.relu0
        self.pool0 = features.pool0

        self.denseblock1 = (
            features.denseblock1
        )
        self.transition1 = (
            features.transition1
        )

        self.denseblock2 = (
            features.denseblock2
        )
        self.transition2 = (
            features.transition2
        )

        self.denseblock3 = (
            features.denseblock3
        )
        self.transition3 = (
            features.transition3
        )

        self.denseblock4 = (
            features.denseblock4
        )
        self.norm5 = features.norm5

        self.decoder4 = DecoderBlock(
            1024,
            1024,
            512,
            decoder_dropout,
        )
        self.decoder3 = DecoderBlock(
            512,
            512,
            256,
            decoder_dropout,
        )
        self.decoder2 = DecoderBlock(
            256,
            256,
            128,
            decoder_dropout,
        )
        self.decoder1 = DecoderBlock(
            128,
            64,
            64,
            decoder_dropout,
        )

        self.final_refine = ConvNormAct(
            64,
            32,
            decoder_dropout,
        )

        self.segmentation_head = nn.Conv2d(
            32,
            num_classes,
            kernel_size=1,
        )

        self.pretrained_loaded = (
            pretrained_loaded
        )

        if freeze_encoder:
            for module in (
                self.conv0,
                self.norm0,
                self.denseblock1,
                self.transition1,
                self.denseblock2,
                self.transition2,
                self.denseblock3,
                self.transition3,
                self.denseblock4,
                self.norm5,
            ):
                for parameter in (
                    module.parameters()
                ):
                    parameter.requires_grad = False

    def forward(self, x):
        input_size = x.shape[-2:]

        skip0 = self.relu0(
            self.norm0(
                self.conv0(x)
            )
        )

        x = self.pool0(skip0)

        skip1 = self.denseblock1(x)
        x = self.transition1(skip1)

        skip2 = self.denseblock2(x)
        x = self.transition2(skip2)

        skip3 = self.denseblock3(x)
        x = self.transition3(skip3)

        x = self.norm5(
            self.denseblock4(x)
        )
        x = F.relu(
            x,
            inplace=True,
        )

        x = self.decoder4(
            x,
            skip3,
        )
        x = self.decoder3(
            x,
            skip2,
        )
        x = self.decoder2(
            x,
            skip1,
        )
        x = self.decoder1(
            x,
            skip0,
        )

        x = F.interpolate(
            x,
            size=input_size,
            mode="bilinear",
            align_corners=False,
        )

        x = self.final_refine(x)

        return self.segmentation_head(x)


def build_segmentation_model(
    in_channels: int,
    num_classes: int,
    decoder_dropout: float,
    memory_efficient: bool,
):
    model = DenseNet121UNet(
        in_channels=in_channels,
        num_classes=num_classes,
        pretrained=False,
        freeze_encoder=False,
        decoder_dropout=decoder_dropout,
        memory_efficient=memory_efficient,
    )

    return model


# =============================================================================
# 6. DETECCIÓN Y CARGA DEL MODELO ENTRENADO
# =============================================================================

def parse_int_mapping(
    mapping: Mapping,
) -> dict[int, int]:
    return {
        int(key): int(value)
        for key, value in mapping.items()
    }


def parse_str_mapping(
    mapping: Mapping,
) -> dict[int, str]:
    return {
        int(key): str(value)
        for key, value in mapping.items()
    }


def validate_run_config(
    run_dir: Path,
    level_name: str,
) -> dict:
    config_path = (
        run_dir / "config.json"
    )

    if not config_path.exists():
        raise FileNotFoundError(
            "La ejecución no contiene "
            f"config.json: {run_dir}"
        )

    config = load_json(
        config_path
    )

    if (
        str(
            config.get(
                "task",
                "",
            )
        ).lower()
        != "segmentacion_semantica"
    ):
        raise ValueError(
            "La ejecución no corresponde "
            f"a segmentación semántica: {run_dir}"
        )

    if (
        str(
            config.get(
                "level",
                "",
            )
        ).lower()
        != level_name
    ):
        raise ValueError(
            "El nivel del config.json "
            f"no coincide con {level_name}: "
            f"{run_dir}"
        )

    model_name = str(
        config.get(
            "model",
            "",
        )
    ).lower()

    encoder_name = str(
        config.get(
            "encoder",
            "",
        )
    ).lower()

    if (
        "densenet121" not in model_name
        and encoder_name != "densenet121"
    ):
        raise ValueError(
            "La ejecución no corresponde "
            f"a DenseNet121: {run_dir}"
        )

    band_indexes = tuple(
        int(value)
        for value in config.get(
            "band_indexes",
            [],
        )
    )

    if len(band_indexes) != 4:
        raise ValueError(
            "El entrenamiento debe usar "
            "cuatro bandas B2, B3, B4 y B8. "
            f"Encontrado: {band_indexes}"
        )

    if "idx_to_code" not in config:
        raise KeyError(
            "config.json no contiene "
            "idx_to_code. No se inferirán "
            "códigos de clase automáticamente."
        )

    if "idx_to_name" not in config:
        raise KeyError(
            "config.json no contiene "
            "idx_to_name."
        )

    idx_to_code = parse_int_mapping(
        config["idx_to_code"]
    )

    expected_codes = (
        EXPECTED_MODEL_CODES_BY_LEVEL[
            level_name
        ]
    )

    observed_codes = [
        idx_to_code[index]
        for index in sorted(idx_to_code)
    ]

    if observed_codes != expected_codes:
        raise ValueError(
            f"{level_name}: idx_to_code no "
            "coincide con los códigos CLC "
            "esperados.\n"
            f"Guardado en entrenamiento: "
            f"{observed_codes}\n"
            f"Esperado: {expected_codes}\n"
            "Se detiene la ejecución para "
            "evitar una predicción con clases "
            "desplazadas."
        )

    means = config.get(
        "means"
    )
    stds = config.get(
        "stds"
    )

    if (
        not isinstance(means, Sequence)
        or not isinstance(stds, Sequence)
        or len(means) != 4
        or len(stds) != 4
    ):
        raise ValueError(
            "config.json debe contener "
            "means y stds para las cuatro bandas."
        )

    return config


def checkpoint_path(
    run_dir: Path,
) -> Path:
    best_model = (
        run_dir / "best_model.pt"
    )

    if best_model.exists():
        return best_model

    last_checkpoint = (
        run_dir
        / "checkpoints"
        / "last_checkpoint.pt"
    )

    if (
        ALLOW_LAST_CHECKPOINT
        and last_checkpoint.exists()
    ):
        warnings.warn(
            "No existe best_model.pt; se "
            "utilizará last_checkpoint.pt.",
            RuntimeWarning,
        )

        return last_checkpoint

    raise FileNotFoundError(
        f"No existe best_model.pt en {run_dir}."
    )


def find_run_dir(
    level_name: str,
) -> Path:
    forced = RUN_BY_LEVEL.get(
        level_name
    )

    if forced:
        candidate = (
            MODEL_ROOT / forced
        )

        if not candidate.exists():
            raise FileNotFoundError(
                candidate
            )

        validate_run_config(
            candidate,
            level_name,
        )

        checkpoint_path(
            candidate
        )

        return candidate

    candidates = []

    if not MODEL_ROOT.exists():
        raise FileNotFoundError(
            f"No existe MODEL_ROOT: {MODEL_ROOT}"
        )

    for run_dir in MODEL_ROOT.glob(
        f"DenseNet121Seg_{level_name}_*"
    ):
        if not run_dir.is_dir():
            continue

        try:
            validate_run_config(
                run_dir,
                level_name,
            )
            selected_checkpoint = (
                checkpoint_path(
                    run_dir
                )
            )
        except Exception:
            continue

        marker = (
            run_dir
            / "LEVEL_COMPLETED.json"
        )
        report = (
            run_dir
            / "reports"
            / "resumen_metricas_test.json"
        )

        completed = (
            marker.exists()
            or report.exists()
        )

        evidence = [
            item.stat().st_mtime
            for item in (
                marker,
                report,
                selected_checkpoint,
            )
            if item.exists()
        ]

        modified = (
            max(evidence)
            if evidence
            else run_dir.stat().st_mtime
        )

        candidates.append(
            (
                1 if completed else 0,
                modified,
                run_dir,
            )
        )

    if not candidates:
        raise FileNotFoundError(
            "No se encontró un modelo "
            f"DenseNet121 válido para "
            f"{level_name} dentro de "
            f"{MODEL_ROOT}.\n"
            "Debe existir config.json y "
            "best_model.pt."
        )

    return max(
        candidates,
        key=lambda item: (
            item[0],
            item[1],
        ),
    )[2]


def load_model_for_level(
    level_name: str,
):
    run_dir = find_run_dir(
        level_name
    )

    config = validate_run_config(
        run_dir,
        level_name,
    )

    idx_to_code = parse_int_mapping(
        config["idx_to_code"]
    )

    idx_to_name = parse_str_mapping(
        config["idx_to_name"]
    )

    band_indexes = tuple(
        int(value)
        for value in config.get(
            "band_indexes",
            [1, 2, 3, 4],
        )
    )

    num_classes = int(
        config["num_classes"]
    )

    if num_classes != len(
        idx_to_code
    ):
        raise ValueError(
            "num_classes no coincide con "
            "idx_to_code."
        )

    decoder_dropout = float(
        config.get(
            "decoder_dropout",
            0.10,
        )
    )

    memory_efficient = bool(
        config.get(
            "densenet_memory_efficient",
            USE_DENSENET_MEMORY_EFFICIENT,
        )
    )

    model = build_segmentation_model(
        in_channels=len(
            band_indexes
        ),
        num_classes=num_classes,
        decoder_dropout=decoder_dropout,
        memory_efficient=memory_efficient,
    )

    selected_checkpoint = (
        checkpoint_path(
            run_dir
        )
    )

    try:
        checkpoint = torch.load(
            selected_checkpoint,
            map_location="cpu",
            weights_only=False,
        )
    except TypeError:
        checkpoint = torch.load(
            selected_checkpoint,
            map_location="cpu",
        )

    state_dict = checkpoint.get(
        "model_state_dict",
        checkpoint,
    )

    model.load_state_dict(
        state_dict,
        strict=True,
    )

    del state_dict
    del checkpoint
    gc.collect()

    inference_device = DEVICE

    try:
        model = (
            model.to(
                inference_device
            )
            .eval()
        )
    except RuntimeError as error:
        if (
            inference_device.type == "cuda"
            and FALLBACK_TO_CPU_ON_GPU_OOM
            and is_gpu_oom(error)
        ):
            print(
                "[ADVERTENCIA] No fue posible "
                "cargar el modelo completo en GPU. "
                "La inferencia continuará en CPU."
            )

            safe_cleanup(
                synchronize=False
            )

            inference_device = (
                torch.device("cpu")
            )

            model = (
                model.cpu()
                .eval()
            )
        else:
            raise

    return {
        "model": model,
        "config": config,
        "run_dir": run_dir,
        "checkpoint": selected_checkpoint,
        "idx_to_code": idx_to_code,
        "idx_to_name": idx_to_name,
        "band_indexes": band_indexes,
        "num_classes": num_classes,
        "device": inference_device,
    }


# =============================================================================
# 7. PREPROCESAMIENTO — MISMA NORMALIZACIÓN DEL ENTRENAMIENTO
# =============================================================================

def reflectance_window(
    source,
    window: Window,
    band_indexes: Sequence[int],
    config: Mapping[str, Any],
):
    array = source.read(
        indexes=band_indexes,
        window=window,
    ).astype(np.float32)

    finite = np.isfinite(
        array
    )

    if (
        source.nodata is not None
        and np.isfinite(
            source.nodata
        )
    ):
        finite &= (
            array != source.nodata
        )

    valid = finite.all(
        axis=0
    )

    array = np.where(
        finite,
        array,
        np.nan,
    )

    scales = np.asarray(
        [
            source.scales[index - 1]
            for index in band_indexes
        ],
        dtype=np.float32,
    )[:, None, None]

    offsets = np.asarray(
        [
            source.offsets[index - 1]
            for index in band_indexes
        ],
        dtype=np.float32,
    )[:, None, None]

    use_transform = bool(
        config.get(
            "use_raster_scale_offset",
            True,
        )
    ) and (
        not np.allclose(
            scales,
            1.0,
        )
        or not np.allclose(
            offsets,
            0.0,
        )
    )

    finite_values = array[
        np.isfinite(array)
    ]

    if use_transform:
        array = (
            array * scales
            + offsets
        )

    elif (
        bool(
            config.get(
                "auto_scale_reflectance",
                True,
            )
        )
        and finite_values.size
        and float(
            np.nanpercentile(
                finite_values,
                99.5,
            )
        )
        > 2.0
    ):
        array = (
            array
            + float(
                config.get(
                    "reflectance_add_offset",
                    0.0,
                )
            )
        ) / float(
            config.get(
                "reflectance_scale",
                10000.0,
            )
        )

    array = np.clip(
        array,
        0.0,
        1.0,
    )

    means = np.asarray(
        config["means"],
        dtype=np.float32,
    )[:, None, None]

    stds = np.asarray(
        config["stds"],
        dtype=np.float32,
    )[:, None, None]

    array = (
        array - means
    ) / np.maximum(
        stds,
        1e-6,
    )

    array = np.nan_to_num(
        array,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )

    array[
        :,
        ~valid,
    ] = 0.0

    return (
        np.ascontiguousarray(
            array,
            dtype=np.float32,
        ),
        valid,
    )


def pad_to_multiple(
    tensor: torch.Tensor,
    multiple: int = 32,
):
    height = tensor.shape[-2]
    width = tensor.shape[-1]

    pad_h = (
        multiple
        - height % multiple
    ) % multiple

    pad_w = (
        multiple
        - width % multiple
    ) % multiple

    padded = F.pad(
        tensor,
        (
            0,
            pad_w,
            0,
            pad_h,
        ),
        mode="replicate",
    )

    return (
        padded,
        height,
        width,
    )


def starts(
    length: int,
    tile_size: int,
    overlap: int,
) -> list[int]:
    if tile_size <= overlap:
        raise ValueError(
            "TILE_SIZE debe ser mayor "
            "que OVERLAP."
        )

    if length <= tile_size:
        return [0]

    stride = (
        tile_size - overlap
    )

    values = list(
        range(
            0,
            length - tile_size + 1,
            stride,
        )
    )

    final = (
        length - tile_size
    )

    if values[-1] != final:
        values.append(
            final
        )

    return values


def blend_weight(
    height: int,
    width: int,
) -> np.ndarray:
    wy = (
        np.hanning(height)
        if height > 2
        else np.ones(height)
    )

    wx = (
        np.hanning(width)
        if width > 2
        else np.ones(width)
    )

    weight = np.outer(
        wy,
        wx,
    ).astype(np.float32)

    return np.maximum(
        weight,
        0.05,
    )


# =============================================================================
# 8. PLAN DE TESELAS Y ESTADO DE REANUDACIÓN
# =============================================================================

def state_path_for_level(
    temp_dir: Path,
) -> Path:
    return (
        temp_dir
        / "estado_prediccion_escritura_directa.json"
    )


def state_compatible(
    state: Mapping[str, Any],
    expected: Mapping[str, Any],
) -> bool:
    return all(
        state.get(key) == value
        for key, value in expected.items()
    )


def build_axis_plan(
    length: int,
    tile_size: int,
    overlap: int,
):
    """
    Genera ventanas de lectura solapadas y divide la zona de solape por la
    mitad. Cada píxel pertenece al núcleo de UNA sola tesela.

    Ventaja:
      - la red recibe contexto alrededor del núcleo;
      - no se necesita sumar probabilidades de todas las clases a escala global;
      - se puede escribir directamente en el GeoTIFF final.
    """
    axis_starts = starts(
        length,
        tile_size,
        overlap,
    )

    read_ends = [
        min(
            start + tile_size,
            length,
        )
        for start in axis_starts
    ]

    boundaries = [0]

    for index in range(
        len(axis_starts) - 1
    ):
        # Punto medio de la intersección entre la tesela actual y la siguiente.
        boundary = (
            read_ends[index]
            + axis_starts[index + 1]
        ) // 2

        boundary = max(
            boundaries[-1],
            min(
                int(boundary),
                length,
            ),
        )

        boundaries.append(
            boundary
        )

    boundaries.append(
        length
    )

    plan = []

    for index, read_start in enumerate(
        axis_starts
    ):
        read_end = read_ends[
            index
        ]

        write_start = boundaries[
            index
        ]

        write_end = boundaries[
            index + 1
        ]

        crop_start = (
            write_start
            - read_start
        )

        crop_end = (
            write_end
            - read_start
        )

        if not (
            0 <= crop_start
            <= crop_end
            <= read_end - read_start
        ):
            raise RuntimeError(
                "Plan de teselas inválido: "
                f"read=({read_start},{read_end}), "
                f"write=({write_start},{write_end}), "
                f"crop=({crop_start},{crop_end})."
            )

        plan.append(
            {
                "read_start": int(
                    read_start
                ),
                "read_end": int(
                    read_end
                ),
                "write_start": int(
                    write_start
                ),
                "write_end": int(
                    write_end
                ),
                "crop_start": int(
                    crop_start
                ),
                "crop_end": int(
                    crop_end
                ),
            }
        )

    return plan


def build_tile_plan(
    height: int,
    width: int,
    tile_size: int,
    overlap: int,
):
    row_plan = build_axis_plan(
        height,
        tile_size,
        overlap,
    )

    col_plan = build_axis_plan(
        width,
        tile_size,
        overlap,
    )

    tiles = []

    for row_info in row_plan:
        for col_info in col_plan:
            tiles.append(
                {
                    "read_row": row_info[
                        "read_start"
                    ],
                    "read_col": col_info[
                        "read_start"
                    ],
                    "read_height": (
                        row_info[
                            "read_end"
                        ]
                        - row_info[
                            "read_start"
                        ]
                    ),
                    "read_width": (
                        col_info[
                            "read_end"
                        ]
                        - col_info[
                            "read_start"
                        ]
                    ),
                    "write_row": row_info[
                        "write_start"
                    ],
                    "write_col": col_info[
                        "write_start"
                    ],
                    "write_height": (
                        row_info[
                            "write_end"
                        ]
                        - row_info[
                            "write_start"
                        ]
                    ),
                    "write_width": (
                        col_info[
                            "write_end"
                        ]
                        - col_info[
                            "write_start"
                        ]
                    ),
                    "crop_row_start": row_info[
                        "crop_start"
                    ],
                    "crop_row_end": row_info[
                        "crop_end"
                    ],
                    "crop_col_start": col_info[
                        "crop_start"
                    ],
                    "crop_col_end": col_info[
                        "crop_end"
                    ],
                }
            )

    return tiles


def create_empty_prediction_files(
    source,
    prediction_path: Path,
    confidence_path: Path,
) -> None:
    """
    Crea y CIERRA inmediatamente los GeoTIFF. Las teselas se escribirán después
    usando modo r+. Esto hace que el archivo tenga un encabezado válido antes de
    comenzar la inferencia.
    """
    class_profile = (
        source.profile.copy()
    )
    class_profile.update(
        driver="GTiff",
        count=1,
        dtype="uint16",
        nodata=OUTPUT_NODATA,
        compress="deflate",
        predictor=2,
        tiled=True,
        blockxsize=GTIFF_BLOCK_SIZE,
        blockysize=GTIFF_BLOCK_SIZE,
        BIGTIFF="IF_SAFER",
    )

    confidence_profile = (
        source.profile.copy()
    )
    confidence_profile.update(
        driver="GTiff",
        count=1,
        dtype="float32",
        nodata=0.0,
        compress="deflate",
        predictor=3,
        tiled=True,
        blockxsize=GTIFF_BLOCK_SIZE,
        blockysize=GTIFF_BLOCK_SIZE,
        BIGTIFF="IF_SAFER",
    )

    safe_unlink(
        prediction_path
    )
    safe_unlink(
        confidence_path
    )

    with rasterio.open(
        prediction_path,
        "w",
        **class_profile,
    ):
        pass

    with rasterio.open(
        confidence_path,
        "w",
        **confidence_profile,
    ):
        pass


# =============================================================================
# 9. INFERENCIA DE UNA TESELA
# =============================================================================

@torch.inference_mode()
def infer_tile(
    model: nn.Module,
    image: np.ndarray,
    inference_device: torch.device,
):
    """
    Devuelve únicamente probabilidades de UNA tesela.

    En modo cpu_ultra_stable no ejecuta ninguna operación CUDA/HIP.
    """
    tensor = (
        torch.from_numpy(
            image
        )
        .unsqueeze(0)
        .to(
            inference_device,
            dtype=torch.float32,
            non_blocking=False,
        )
    )

    (
        tensor,
        original_h,
        original_w,
    ) = pad_to_multiple(
        tensor,
        32,
    )

    try:
        with autocast_context(
            inference_device
        ):
            logits = model(
                tensor
            )

            probabilities = torch.softmax(
                logits,
                dim=1,
            )

        result = (
            probabilities[
                0,
                :,
                :original_h,
                :original_w,
            ]
            .to(
                device="cpu",
                dtype=torch.float32,
            )
            .numpy()
            .copy()
        )

        del tensor
        del logits
        del probabilities

        return (
            result,
            inference_device,
            model,
        )

    except RuntimeError as error:
        try:
            del tensor
        except Exception:
            pass

        if (
            inference_device.type == "cuda"
            and FALLBACK_TO_CPU_ON_GPU_OOM
            and is_gpu_oom(error)
        ):
            print(
                "\n[ADVERTENCIA] PyTorch reportó falta de memoria GPU. "
                "Se moverá DenseNet121 a CPU y se continuará desde esta tesela."
            )

            try:
                model = model.cpu().eval()
            except Exception:
                raise

            safe_cleanup(
                synchronize=False
            )

            inference_device = (
                torch.device("cpu")
            )

            cpu_tensor = (
                torch.from_numpy(
                    image
                )
                .unsqueeze(0)
                .to(
                    inference_device,
                    dtype=torch.float32,
                )
            )

            (
                cpu_tensor,
                original_h,
                original_w,
            ) = pad_to_multiple(
                cpu_tensor,
                32,
            )

            logits = model(
                cpu_tensor
            )

            probabilities = torch.softmax(
                logits,
                dim=1,
            )

            result = (
                probabilities[
                    0,
                    :,
                    :original_h,
                    :original_w,
                ]
                .float()
                .cpu()
                .numpy()
                .copy()
            )

            del cpu_tensor
            del logits
            del probabilities
            gc.collect()

            return (
                result,
                inference_device,
                model,
            )

        raise


# =============================================================================
# 10. PREDICCIÓN COMPLETA DE UN NIVEL — ESCRITURA DIRECTA POR NÚCLEOS
# =============================================================================

@torch.inference_mode()
def predict_level(
    level_name: str,
):
    model_info = load_model_for_level(
        level_name
    )

    model = model_info[
        "model"
    ]
    # Eliminar la segunda referencia al modelo inmediatamente. Si se conserva
    # dentro de model_info, `del model` al terminar el nivel NO libera realmente
    # la RAM/VRAM y el siguiente nivel puede comenzar con memoria retenida.
    model_info[
        "model"
    ] = None

    config = model_info[
        "config"
    ]
    run_dir = model_info[
        "run_dir"
    ]
    checkpoint = model_info[
        "checkpoint"
    ]
    idx_to_code = model_info[
        "idx_to_code"
    ]
    idx_to_name = model_info[
        "idx_to_name"
    ]
    band_indexes = model_info[
        "band_indexes"
    ]
    num_classes = model_info[
        "num_classes"
    ]
    inference_device = model_info[
        "device"
    ]

    if not INPUT_RASTER.exists():
        raise FileNotFoundError(
            f"No existe INPUT_RASTER: {INPUT_RASTER}"
        )

    level_dir = ensure_dir(
        OUTPUT_DIR / level_name
    )

    temp_dir = ensure_dir(
        TEMP_DIR / level_name
    )

    prediction_path = (
        level_dir
        / f"DenseNet121_{level_name}_prediccion.tif"
    )

    confidence_path = (
        level_dir
        / f"DenseNet121_{level_name}_confianza.tif"
    )

    completed_path = (
        level_dir
        / "PREDICTION_COMPLETED.json"
    )

    state_file = state_path_for_level(
        temp_dir
    )

    # -----------------------------------------------------------------
    # EJECUCIÓN NUEVA OBLIGATORIA
    # -----------------------------------------------------------------
    # La predicción anterior, su confianza y cualquier estado parcial se
    # eliminan antes de comenzar. Esto garantiza que cada ejecución sea una
    # inferencia nueva del checkpoint seleccionado sobre INPUT_RASTER.
    if FORCE_NEW_PREDICTION:
        safe_unlink(completed_path)
        safe_unlink(state_file)
        safe_unlink(prediction_path)
        safe_unlink(confidence_path)

    if (
        REUSE_COMPLETED_PREDICTIONS
        and prediction_path.exists()
        and confidence_path.exists()
        and completed_path.exists()
    ):
        try:
            completed = load_json(
                completed_path
            )

            same_input = (
                str(
                    Path(
                        completed.get(
                            "input_raster",
                            "",
                        )
                    )
                )
                == str(
                    INPUT_RASTER
                )
            )

            same_run = (
                str(
                    Path(
                        completed.get(
                            "run_dir",
                            "",
                        )
                    )
                )
                == str(
                    run_dir
                )
            )

            if same_input and same_run:
                print(
                    f"[REUTILIZACIÓN] {level_name} ya tiene una predicción "
                    "final para el mismo modelo/input."
                )

                del model
                safe_cleanup(
                    synchronize=False
                )

                return {
                    **model_info,
                    "model": None,
                    "prediction": prediction_path,
                    "confidence": confidence_path,
                    "device_used": completed.get(
                        "device_used",
                        str(
                            inference_device
                        ),
                    ),
                    "elapsed_seconds": completed.get(
                        "elapsed_seconds",
                        None,
                    ),
                    "reused": True,
                }

        except Exception:
            pass

    tile_size = int(
        TILE_SIZE_BY_LEVEL[
            level_name
        ]
    )

    overlap = int(
        OVERLAP_BY_LEVEL[
            level_name
        ]
    )

    validity_source = None
    validity_vrt = None

    started_at = time.time()

    try:
        with rasterio.open(
            INPUT_RASTER
        ) as source:
            height = int(
                source.height
            )
            width = int(
                source.width
            )

            if max(
                band_indexes
            ) > source.count:
                raise ValueError(
                    f"La imagen tiene {source.count} bandas y "
                    f"el modelo solicita {band_indexes}."
                )

            tile_plan = build_tile_plan(
                height=height,
                width=width,
                tile_size=tile_size,
                overlap=overlap,
            )

            total_tiles = len(
                tile_plan
            )

            expected_state = {
                "strategy": (
                    "direct_core_write_v1"
                ),
                "height": height,
                "width": width,
                "num_classes": num_classes,
                "tile_size": tile_size,
                "overlap": overlap,
                "level": level_name,
                "input_raster": str(
                    INPUT_RASTER.resolve()
                ),
                "run_dir": str(
                    run_dir.resolve()
                ),
                "checkpoint": str(
                    checkpoint.resolve()
                ),
                "band_indexes": list(
                    band_indexes
                ),
                "prediction_path": str(
                    prediction_path.resolve()
                ),
                "confidence_path": str(
                    confidence_path.resolve()
                ),
            }

            next_tile_index = 0
            can_resume = False

            if (
                RESUME_PREDICTION
                and state_file.exists()
                and prediction_path.exists()
                and confidence_path.exists()
            ):
                try:
                    state = load_json(
                        state_file
                    )

                    can_resume = state_compatible(
                        state,
                        expected_state,
                    )

                    if can_resume:
                        next_tile_index = int(
                            state.get(
                                "next_tile_index",
                                0,
                            )
                        )

                        if not (
                            0
                            <= next_tile_index
                            <= total_tiles
                        ):
                            can_resume = False

                except Exception as error:
                    warnings.warn(
                        f"No fue posible leer el estado de {level_name}: "
                        f"{error}. Se comenzará de nuevo.",
                        RuntimeWarning,
                    )
                    can_resume = False

            if can_resume:
                print(
                    f"[REANUDACIÓN] {level_name}: tesela "
                    f"{next_tile_index:,}/{total_tiles:,}."
                )
            else:
                create_empty_prediction_files(
                    source,
                    prediction_path,
                    confidence_path,
                )

                safe_unlink(
                    state_file
                )

                next_tile_index = 0

                save_json(
                    {
                        **expected_state,
                        "next_tile_index": 0,
                        "total_tiles": total_tiles,
                        "device_used": str(
                            inference_device
                        ),
                    },
                    state_file,
                )

            if VALIDITY_MASK_RASTER is not None:
                validity_source = rasterio.open(
                    VALIDITY_MASK_RASTER
                )

                validity_vrt = WarpedVRT(
                    validity_source,
                    crs=source.crs,
                    transform=source.transform,
                    width=source.width,
                    height=source.height,
                    resampling=Resampling.nearest,
                    nodata=0,
                )

            code_array = np.asarray(
                [
                    idx_to_code[
                        index
                    ]
                    for index in range(
                        num_classes
                    )
                ],
                dtype=np.uint16,
            )

            print(
                f"[INFO] {level_name}: {total_tiles:,} teselas | "
                f"lectura={tile_size}px | solape={overlap}px | "
                f"backend={PREDICTION_BACKEND} | dispositivo={inference_device}"
            )

            progress = tqdm(
                total=total_tiles,
                initial=next_tile_index,
                desc=(
                    f"DenseNet121 {level_name}"
                ),
                unit="tesela",
                leave=False,
                mininterval=3.0,
                dynamic_ncols=False,
            )

            # -----------------------------------------------------------------
            # Las salidas se abren solo por sesiones pequeñas. Al cerrar una
            # sesión GDAL finaliza/descarga buffers y recién después se guarda
            # next_tile_index. Esto evita mantener un TIFF grande abierto
            # durante toda la inferencia y mejora la recuperación tras fallos.
            # -----------------------------------------------------------------
            while next_tile_index < total_tiles:
                session_start = (
                    next_tile_index
                )

                session_end = min(
                    session_start
                    + WRITE_SESSION_TILES,
                    total_tiles,
                )

                with rasterio.open(
                    prediction_path,
                    "r+",
                ) as class_writer, rasterio.open(
                    confidence_path,
                    "r+",
                ) as confidence_writer:

                    for tile_index in range(
                        session_start,
                        session_end,
                    ):
                        tile = tile_plan[
                            tile_index
                        ]

                        read_window = Window(
                            tile[
                                "read_col"
                            ],
                            tile[
                                "read_row"
                            ],
                            tile[
                                "read_width"
                            ],
                            tile[
                                "read_height"
                            ],
                        )

                        image, valid = (
                            reflectance_window(
                                source,
                                read_window,
                                band_indexes,
                                config,
                            )
                        )

                        if validity_vrt is not None:
                            valid &= (
                                validity_vrt.read(
                                    1,
                                    window=read_window,
                                )
                                > 0
                            )

                        read_height = int(
                            tile[
                                "read_height"
                            ]
                        )
                        read_width = int(
                            tile[
                                "read_width"
                            ]
                        )

                        prediction_tile = np.full(
                            (
                                read_height,
                                read_width,
                            ),
                            OUTPUT_NODATA,
                            dtype=np.uint16,
                        )

                        confidence_tile = np.zeros(
                            (
                                read_height,
                                read_width,
                            ),
                            dtype=np.float32,
                        )

                        if valid.any():
                            (
                                probabilities,
                                new_device,
                                model,
                            ) = infer_tile(
                                model,
                                image,
                                inference_device,
                            )

                            if (
                                new_device
                                != inference_device
                            ):
                                inference_device = (
                                    new_device
                                )

                                print(
                                    "\n[INFO] La inferencia continuará en "
                                    f"{inference_device}."
                                )

                            internal_index = (
                                probabilities.argmax(
                                    axis=0
                                )
                            )

                            maximum_probability = (
                                probabilities.max(
                                    axis=0
                                )
                                .astype(
                                    np.float32
                                )
                            )

                            prediction_tile[
                                valid
                            ] = code_array[
                                internal_index[
                                    valid
                                ]
                            ]

                            confidence_tile[
                                valid
                            ] = (
                                maximum_probability[
                                    valid
                                ]
                            )

                            del probabilities
                            del internal_index
                            del maximum_probability

                        r0 = int(
                            tile[
                                "crop_row_start"
                            ]
                        )
                        r1 = int(
                            tile[
                                "crop_row_end"
                            ]
                        )
                        c0 = int(
                            tile[
                                "crop_col_start"
                            ]
                        )
                        c1 = int(
                            tile[
                                "crop_col_end"
                            ]
                        )

                        prediction_core = (
                            prediction_tile[
                                r0:r1,
                                c0:c1,
                            ]
                        )

                        confidence_core = (
                            confidence_tile[
                                r0:r1,
                                c0:c1,
                            ]
                        )

                        write_window = Window(
                            tile[
                                "write_col"
                            ],
                            tile[
                                "write_row"
                            ],
                            tile[
                                "write_width"
                            ],
                            tile[
                                "write_height"
                            ],
                        )

                        class_writer.write(
                            prediction_core,
                            1,
                            window=write_window,
                        )

                        confidence_writer.write(
                            confidence_core,
                            1,
                            window=write_window,
                        )

                        del image
                        del valid
                        del prediction_tile
                        del confidence_tile
                        del prediction_core
                        del confidence_core

                        next_tile_index = (
                            tile_index + 1
                        )

                        progress.update(
                            1
                        )

                        if (
                            next_tile_index
                            % CPU_GC_EVERY_TILES
                            == 0
                        ):
                            gc.collect()

                # Los dos GeoTIFF ya están cerrados en este punto.
                save_json(
                    {
                        **expected_state,
                        "next_tile_index": (
                            next_tile_index
                        ),
                        "total_tiles": (
                            total_tiles
                        ),
                        "device_used": str(
                            inference_device
                        ),
                    },
                    state_file,
                )

                gc.collect()

            progress.close()

        elapsed_seconds = float(
            time.time()
            - started_at
        )

        save_json(
            {
                "completed": True,
                "model": MODEL_NAME,
                "level": level_name,
                "input_raster": str(
                    INPUT_RASTER
                ),
                "run_dir": str(
                    run_dir
                ),
                "checkpoint": str(
                    checkpoint
                ),
                "prediction": str(
                    prediction_path
                ),
                "confidence": str(
                    confidence_path
                ),
                "tile_size": (
                    tile_size
                ),
                "overlap": (
                    overlap
                ),
                "write_strategy": (
                    "direct_core_write_v1"
                ),
                "band_indexes": list(
                    band_indexes
                ),
                "device_used": str(
                    inference_device
                ),
                "elapsed_seconds": (
                    elapsed_seconds
                ),
            },
            completed_path,
        )

        safe_unlink(
            state_file
        )

    finally:
        if validity_vrt is not None:
            try:
                validity_vrt.close()
            except Exception:
                pass

        if validity_source is not None:
            try:
                validity_source.close()
            except Exception:
                pass

        try:
            del model
        except Exception:
            pass

        safe_cleanup(
            synchronize=False
        )

    return {
        **model_info,
        "model": None,
        "prediction": prediction_path,
        "confidence": confidence_path,
        "device_used": str(
            inference_device
        ),
        "elapsed_seconds": (
            elapsed_seconds
        ),
        "reused": False,
    }


# =============================================================================
# 11. VALIDACIÓN ESPACIAL Y DE CÓDIGOS
# =============================================================================

def grid_signature(
    dataset,
) -> dict:
    return {
        "width": int(
            dataset.width
        ),
        "height": int(
            dataset.height
        ),
        "crs": str(
            dataset.crs
        ),
        "transform": tuple(
            dataset.transform
        ),
        "resolution": tuple(
            dataset.res
        ),
        "bounds": tuple(
            dataset.bounds
        ),
    }


def validate_identical_grid(
    input_path: Path,
    output_path: Path,
) -> dict:
    with rasterio.open(
        input_path
    ) as source, rasterio.open(
        output_path
    ) as output:
        identical = (
            source.width
            == output.width
            and source.height
            == output.height
            and source.crs
            == output.crs
            and source.transform
            == output.transform
        )

        result = {
            "identical": bool(
                identical
            ),
            "input": grid_signature(
                source
            ),
            "output": grid_signature(
                output
            ),
        }

        if not identical:
            raise RuntimeError(
                "La grilla de la predicción "
                "no coincide exactamente con "
                "INPUT_RASTER."
            )

        return result


def collect_unique_codes(
    raster_path: Path,
    nodata_code: int = 0,
) -> list[int]:
    observed = set()

    with rasterio.open(
        raster_path
    ) as source:
        for _, window in (
            source.block_windows(1)
        ):
            values = np.unique(
                source.read(
                    1,
                    window=window,
                )
            )

            observed.update(
                int(value)
                for value in values.tolist()
            )

    observed.discard(
        int(nodata_code)
    )

    return sorted(
        observed
    )


def validate_prediction_codes(
    level_name: str,
    prediction_path: Path,
    reference_path: Path,
    idx_to_code: Mapping[int, int],
    idx_to_name: Mapping[int, str],
):
    expected_model_codes = (
        EXPECTED_MODEL_CODES_BY_LEVEL[
            level_name
        ]
    )

    config_codes = [
        int(
            idx_to_code[index]
        )
        for index in sorted(
            idx_to_code
        )
    ]

    if (
        config_codes
        != expected_model_codes
    ):
        raise ValueError(
            f"{level_name}: los códigos "
            "guardados por el entrenamiento "
            f"son {config_codes}, pero se "
            f"esperaban "
            f"{expected_model_codes}."
        )

    predicted_codes = (
        collect_unique_codes(
            prediction_path,
            OUTPUT_NODATA,
        )
    )

    unexpected_prediction = sorted(
        set(predicted_codes)
        - set(expected_model_codes)
    )

    if unexpected_prediction:
        raise ValueError(
            f"{level_name}: la predicción "
            "contiene códigos desconocidos: "
            f"{unexpected_prediction}."
        )

    reference_codes = (
        collect_unique_codes(
            reference_path,
            REFERENCE_BACKGROUND_CODE,
        )
    )

    expected_reference = (
        EXPECTED_REFERENCE_CODES_BY_LEVEL[
            level_name
        ]
    )

    unexpected_reference = sorted(
        set(reference_codes)
        - set(expected_model_codes)
    )

    if unexpected_reference:
        warnings.warn(
            f"{level_name}: la referencia "
            "contiene códigos fuera de la "
            "leyenda del modelo: "
            f"{unexpected_reference}.",
            RuntimeWarning,
        )

    model_name_by_code = {
        int(
            idx_to_code[index]
        ): str(
            idx_to_name[index]
        )
        for index in sorted(
            idx_to_code
        )
    }

    rows = []

    for code in expected_model_codes:
        rows.append(
            {
                "class_code": code,
                "class_name": (
                    model_name_by_code[
                        code
                    ]
                ),
                "defined_in_training_config": (
                    code in config_codes
                ),
                "observed_in_prediction": (
                    code in predicted_codes
                ),
                "observed_in_reference": (
                    code in reference_codes
                ),
                "expected_in_reference_file": (
                    code in expected_reference
                ),
                "status": (
                    "OK"
                    if code in reference_codes
                    else (
                        "SIN_SOPORTE_REFERENCIA"
                        if code
                        not in expected_reference
                        else "AUSENTE_REFERENCIA"
                    )
                ),
            }
        )

    validation_frame = (
        pd.DataFrame(rows)
    )

    summary = {
        "level": level_name,
        "training_config_codes": (
            config_codes
        ),
        "prediction_codes_observed": (
            predicted_codes
        ),
        "reference_codes_observed": (
            reference_codes
        ),
        "expected_model_codes": (
            expected_model_codes
        ),
        "expected_reference_codes": (
            expected_reference
        ),
        "unexpected_prediction_codes": (
            unexpected_prediction
        ),
        "unexpected_reference_codes": (
            unexpected_reference
        ),
        "mapping_changed": False,
        "note": (
            "La correspondencia idx_to_code "
            "del entrenamiento se conserva "
            "sin permutar clases usando la "
            "referencia."
        ),
    }

    return (
        validation_frame,
        summary,
    )


# =============================================================================
# 12. MÉTRICAS
# =============================================================================

def safe_divide(
    numerator,
    denominator,
):
    numerator = np.asarray(
        numerator,
        dtype=np.float64,
    )
    denominator = np.asarray(
        denominator,
        dtype=np.float64,
    )

    result = np.full(
        np.broadcast_shapes(
            numerator.shape,
            denominator.shape,
        ),
        np.nan,
        dtype=np.float64,
    )

    return np.divide(
        numerator,
        denominator,
        out=result,
        where=denominator != 0,
    )


def metrics_from_confusion(
    confusion: np.ndarray,
    class_codes: Sequence[int],
    class_names: Sequence[str],
):
    matrix = confusion.astype(
        np.float64
    )

    total = float(
        matrix.sum()
    )

    support_reference = (
        matrix.sum(axis=1)
    )

    support_prediction = (
        matrix.sum(axis=0)
    )

    true_positive = np.diag(
        matrix
    )

    false_negative = (
        support_reference
        - true_positive
    )

    false_positive = (
        support_prediction
        - true_positive
    )

    true_negative = (
        total
        - true_positive
        - false_negative
        - false_positive
    )

    precision = safe_divide(
        true_positive,
        true_positive
        + false_positive,
    )

    recall = safe_divide(
        true_positive,
        true_positive
        + false_negative,
    )

    specificity = safe_divide(
        true_negative,
        true_negative
        + false_positive,
    )

    npv = safe_divide(
        true_negative,
        true_negative
        + false_negative,
    )

    f1 = safe_divide(
        2.0
        * precision
        * recall,
        precision
        + recall,
    )

    iou = safe_divide(
        true_positive,
        true_positive
        + false_positive
        + false_negative,
    )

    omission_error = (
        1.0 - recall
    )

    commission_error = (
        1.0 - precision
    )

    supported = (
        support_reference > 0
    )

    weights = (
        support_reference / total
        if total > 0
        else np.zeros_like(
            support_reference
        )
    )

    def macro(
        values: np.ndarray,
    ) -> float:
        valid = (
            supported
            & np.isfinite(
                values
            )
        )

        return (
            float(
                np.mean(
                    values[
                        valid
                    ]
                )
            )
            if valid.any()
            else float("nan")
        )

    def weighted(
        values: np.ndarray,
    ) -> float:
        valid = (
            supported
            & np.isfinite(
                values
            )
        )

        return (
            float(
                np.sum(
                    values[
                        valid
                    ]
                    * weights[
                        valid
                    ]
                )
            )
            if valid.any()
            else float("nan")
        )

    accuracy = (
        float(
            true_positive.sum()
            / total
        )
        if total > 0
        else float("nan")
    )

    expected_agreement = (
        float(
            np.dot(
                support_reference,
                support_prediction,
            )
            / (
                total * total
            )
        )
        if total > 0
        else float("nan")
    )

    kappa = (
        float(
            (
                accuracy
                - expected_agreement
            )
            / (
                1.0
                - expected_agreement
            )
        )
        if (
            total > 0
            and np.isfinite(
                expected_agreement
            )
            and not np.isclose(
                1.0
                - expected_agreement,
                0.0,
            )
        )
        else float("nan")
    )

    # MCC multiclase.
    covariance = (
        total
        * true_positive.sum()
        - np.dot(
            support_reference,
            support_prediction,
        )
    )

    denominator_mcc = math.sqrt(
        max(
            0.0,
            (
                total * total
                - np.dot(
                    support_prediction,
                    support_prediction,
                )
            )
            * (
                total * total
                - np.dot(
                    support_reference,
                    support_reference,
                )
            ),
        )
    )

    mcc = (
        float(
            covariance
            / denominator_mcc
        )
        if denominator_mcc > 0
        else float("nan")
    )

    global_metrics = {
        "evaluated_pixels": int(
            total
        ),
        "accuracy": accuracy,
        "pixel_accuracy": accuracy,
        "balanced_accuracy": macro(
            recall
        ),
        "macro_precision": macro(
            precision
        ),
        "macro_recall": macro(
            recall
        ),
        "macro_f1": macro(
            f1
        ),
        "mean_iou": macro(
            iou
        ),
        "macro_specificity": macro(
            specificity
        ),
        "weighted_precision": weighted(
            precision
        ),
        "weighted_recall": weighted(
            recall
        ),
        "weighted_f1": weighted(
            f1
        ),
        "weighted_iou": weighted(
            iou
        ),
        "micro_precision": (
            accuracy
        ),
        "micro_recall": (
            accuracy
        ),
        "micro_f1": (
            accuracy
        ),
        "kappa": kappa,
        "mcc": mcc,
    }

    per_class = pd.DataFrame(
        {
            "class_code": list(
                class_codes
            ),
            "class_name": list(
                class_names
            ),
            "support_reference": (
                support_reference
                .astype(np.int64)
            ),
            "support_prediction": (
                support_prediction
                .astype(np.int64)
            ),
            "true_positive": (
                true_positive
                .astype(np.int64)
            ),
            "false_positive": (
                false_positive
                .astype(np.int64)
            ),
            "false_negative": (
                false_negative
                .astype(np.int64)
            ),
            "true_negative": (
                true_negative
                .astype(np.int64)
            ),
            "precision": precision,
            "recall_sensitivity": recall,
            "specificity": specificity,
            "negative_predictive_value": (
                npv
            ),
            "f1_dice": f1,
            "iou_jaccard": iou,
            "commission_error": (
                commission_error
            ),
            "omission_error": (
                omission_error
            ),
            "has_reference_support": (
                supported
            ),
        }
    )

    return (
        global_metrics,
        per_class,
    )


def evaluate_prediction(
    prediction_path: Path,
    reference_path: Path,
    class_codes: Sequence[int],
    class_names: Sequence[str],
):
    class_codes = [
        int(code)
        for code in class_codes
    ]

    class_count = len(
        class_codes
    )

    code_to_index = {
        code: index
        for index, code
        in enumerate(
            class_codes
        )
    }

    confusion = np.zeros(
        (
            class_count,
            class_count,
        ),
        dtype=np.int64,
    )

    unknown_reference_pixels = 0
    unknown_prediction_pixels = 0
    reference_nodata_pixels = 0
    prediction_nodata_pixels = 0

    with rasterio.open(
        prediction_path
    ) as prediction, rasterio.open(
        reference_path
    ) as reference_source:

        reference_vrt = WarpedVRT(
            reference_source,
            crs=prediction.crs,
            transform=prediction.transform,
            width=prediction.width,
            height=prediction.height,
            resampling=Resampling.nearest,
            nodata=REFERENCE_BACKGROUND_CODE,
        )

        try:
            for _, window in (
                prediction.block_windows(1)
            ):
                predicted = (
                    prediction.read(
                        1,
                        window=window,
                    )
                    .astype(
                        np.int64
                    )
                )

                reference = (
                    reference_vrt.read(
                        1,
                        window=window,
                    )
                    .astype(
                        np.int64
                    )
                )

                valid_prediction = (
                    predicted
                    != OUTPUT_NODATA
                )

                valid_reference = (
                    reference
                    != REFERENCE_BACKGROUND_CODE
                )

                prediction_nodata_pixels += int(
                    predicted.size
                    - valid_prediction.sum()
                )

                reference_nodata_pixels += int(
                    reference.size
                    - valid_reference.sum()
                )

                valid = (
                    valid_prediction
                    & valid_reference
                )

                if not valid.any():
                    continue

                predicted_values = (
                    predicted[
                        valid
                    ]
                )

                reference_values = (
                    reference[
                        valid
                    ]
                )

                known_prediction = np.isin(
                    predicted_values,
                    class_codes,
                )

                known_reference = np.isin(
                    reference_values,
                    class_codes,
                )

                unknown_prediction_pixels += int(
                    (
                        ~known_prediction
                    ).sum()
                )

                unknown_reference_pixels += int(
                    (
                        ~known_reference
                    ).sum()
                )

                known = (
                    known_prediction
                    & known_reference
                )

                if not known.any():
                    continue

                # Los códigos CLC son pequeños (1..13). Un lookup vectorizado
                # evita generadores Python y reduce tiempo/objetos temporales.
                max_lookup_code = max(
                    class_codes
                )
                code_lookup = np.full(
                    max_lookup_code + 1,
                    -1,
                    dtype=np.int16,
                )
                for code, index in code_to_index.items():
                    code_lookup[
                        code
                    ] = index

                known_reference_values = (
                    reference_values[
                        known
                    ]
                )
                known_prediction_values = (
                    predicted_values[
                        known
                    ]
                )

                true_index = code_lookup[
                    known_reference_values
                ].astype(
                    np.int64,
                    copy=False,
                )

                pred_index = code_lookup[
                    known_prediction_values
                ].astype(
                    np.int64,
                    copy=False,
                )

                encoded = (
                    true_index
                    * class_count
                    + pred_index
                )

                confusion += np.bincount(
                    encoded,
                    minlength=(
                        class_count
                        * class_count
                    ),
                ).reshape(
                    class_count,
                    class_count,
                )

        finally:
            reference_vrt.close()

    (
        global_metrics,
        class_metrics,
    ) = metrics_from_confusion(
        confusion,
        class_codes,
        class_names,
    )

    global_metrics.update(
        {
            "unknown_reference_pixels": int(
                unknown_reference_pixels
            ),
            "unknown_prediction_pixels": int(
                unknown_prediction_pixels
            ),
            "reference_nodata_pixels": int(
                reference_nodata_pixels
            ),
            "prediction_nodata_pixels": int(
                prediction_nodata_pixels
            ),
            "classes_without_reference_support": [
                int(
                    class_metrics.loc[
                        index,
                        "class_code",
                    ]
                )
                for index in class_metrics.index
                if not bool(
                    class_metrics.loc[
                        index,
                        "has_reference_support",
                    ]
                )
            ],
            "evaluation_note": (
                EVALUATION_NOTE
            ),
        }
    )

    labels = [
        f"{code} - {name}"
        for code, name
        in zip(
            class_codes,
            class_names,
        )
    ]

    confusion_frame = pd.DataFrame(
        confusion,
        index=labels,
        columns=labels,
    )

    row_totals = (
        confusion.sum(
            axis=1,
            keepdims=True,
        )
    )

    normalized = np.divide(
        confusion,
        row_totals,
        out=np.zeros_like(
            confusion,
            dtype=np.float64,
        ),
        where=row_totals > 0,
    )

    normalized_frame = pd.DataFrame(
        normalized,
        index=labels,
        columns=labels,
    )

    return {
        "global_metrics": (
            global_metrics
        ),
        "class_metrics": (
            class_metrics
        ),
        "confusion": (
            confusion_frame
        ),
        "confusion_normalized": (
            normalized_frame
        ),
    }


# =============================================================================
# 13. CONFIANZA / CALIBRACIÓN ECE-MCE
# =============================================================================

def evaluate_confidence(
    prediction_path: Path,
    confidence_path: Path,
    reference_path: Path,
    bins: int = CONFIDENCE_BINS,
):
    bin_count = np.zeros(
        bins,
        dtype=np.int64,
    )

    bin_confidence_sum = np.zeros(
        bins,
        dtype=np.float64,
    )

    bin_correct_sum = np.zeros(
        bins,
        dtype=np.int64,
    )

    total = 0
    correct_total = 0
    confidence_sum = 0.0
    confidence_correct_sum = 0.0
    confidence_incorrect_sum = 0.0
    correct_count = 0
    incorrect_count = 0

    with rasterio.open(
        prediction_path
    ) as prediction, rasterio.open(
        confidence_path
    ) as confidence, rasterio.open(
        reference_path
    ) as reference_source:

        if (
            prediction.width
            != confidence.width
            or prediction.height
            != confidence.height
            or prediction.crs
            != confidence.crs
            or prediction.transform
            != confidence.transform
        ):
            raise RuntimeError(
                "El ráster de confianza no "
                "coincide con la predicción."
            )

        reference_vrt = WarpedVRT(
            reference_source,
            crs=prediction.crs,
            transform=prediction.transform,
            width=prediction.width,
            height=prediction.height,
            resampling=Resampling.nearest,
            nodata=REFERENCE_BACKGROUND_CODE,
        )

        try:
            for _, window in (
                prediction.block_windows(1)
            ):
                pred = prediction.read(
                    1,
                    window=window,
                )

                conf = (
                    confidence.read(
                        1,
                        window=window,
                    )
                    .astype(np.float64)
                )

                ref = reference_vrt.read(
                    1,
                    window=window,
                )

                valid = (
                    pred
                    != OUTPUT_NODATA
                ) & (
                    ref
                    != REFERENCE_BACKGROUND_CODE
                ) & np.isfinite(
                    conf
                )

                if not valid.any():
                    continue

                p = pred[
                    valid
                ].astype(
                    np.int64
                )

                r = ref[
                    valid
                ].astype(
                    np.int64
                )

                c = np.clip(
                    conf[
                        valid
                    ],
                    0.0,
                    1.0,
                )

                correct = (
                    p == r
                )

                total += int(
                    c.size
                )

                correct_total += int(
                    correct.sum()
                )

                confidence_sum += float(
                    c.sum()
                )

                if correct.any():
                    confidence_correct_sum += float(
                        c[
                            correct
                        ].sum()
                    )
                    correct_count += int(
                        correct.sum()
                    )

                incorrect = (
                    ~correct
                )

                if incorrect.any():
                    confidence_incorrect_sum += float(
                        c[
                            incorrect
                        ].sum()
                    )
                    incorrect_count += int(
                        incorrect.sum()
                    )

                bin_index = np.minimum(
                    (
                        c * bins
                    ).astype(
                        np.int64
                    ),
                    bins - 1,
                )

                bin_count += np.bincount(
                    bin_index,
                    minlength=bins,
                )

                bin_confidence_sum += np.bincount(
                    bin_index,
                    weights=c,
                    minlength=bins,
                )

                bin_correct_sum += np.bincount(
                    bin_index,
                    weights=correct.astype(
                        np.int64
                    ),
                    minlength=bins,
                ).astype(
                    np.int64
                )

        finally:
            reference_vrt.close()

    if total == 0:
        return (
            {
                "evaluated_pixels": 0,
                "mean_confidence": None,
                "accuracy": None,
                "ece": None,
                "mce": None,
            },
            pd.DataFrame(),
        )

    bin_mean_confidence = np.divide(
        bin_confidence_sum,
        bin_count,
        out=np.zeros(
            bins,
            dtype=np.float64,
        ),
        where=bin_count > 0,
    )

    bin_accuracy = np.divide(
        bin_correct_sum,
        bin_count,
        out=np.zeros(
            bins,
            dtype=np.float64,
        ),
        where=bin_count > 0,
    )

    gap = np.abs(
        bin_accuracy
        - bin_mean_confidence
    )

    weights = (
        bin_count
        / float(total)
    )

    ece = float(
        np.sum(
            gap * weights
        )
    )

    mce = float(
        gap[
            bin_count > 0
        ].max()
    ) if np.any(
        bin_count > 0
    ) else float("nan")

    summary = {
        "evaluated_pixels": int(
            total
        ),
        "accuracy": float(
            correct_total
            / total
        ),
        "mean_confidence": float(
            confidence_sum
            / total
        ),
        "confidence_minus_accuracy": float(
            confidence_sum
            / total
            - correct_total
            / total
        ),
        "mean_confidence_correct": (
            float(
                confidence_correct_sum
                / correct_count
            )
            if correct_count > 0
            else None
        ),
        "mean_confidence_incorrect": (
            float(
                confidence_incorrect_sum
                / incorrect_count
            )
            if incorrect_count > 0
            else None
        ),
        "ece": ece,
        "mce": mce,
        "bins": int(
            bins
        ),
    }

    edges = np.linspace(
        0.0,
        1.0,
        bins + 1,
    )

    bins_frame = pd.DataFrame(
        {
            "bin": np.arange(
                bins
            ),
            "confidence_min": (
                edges[:-1]
            ),
            "confidence_max": (
                edges[1:]
            ),
            "pixel_count": (
                bin_count
            ),
            "mean_confidence": (
                bin_mean_confidence
            ),
            "bin_accuracy": (
                bin_accuracy
            ),
            "absolute_gap": (
                gap
            ),
            "weight": (
                weights
            ),
        }
    )

    return (
        summary,
        bins_frame,
    )


def confidence_by_predicted_class(
    prediction_path: Path,
    confidence_path: Path,
    class_codes: Sequence[int],
    class_names: Sequence[str],
):
    counts = np.zeros(
        len(class_codes),
        dtype=np.int64,
    )

    confidence_sum = np.zeros(
        len(class_codes),
        dtype=np.float64,
    )

    with rasterio.open(
        prediction_path
    ) as prediction, rasterio.open(
        confidence_path
    ) as confidence:

        for _, window in (
            prediction.block_windows(1)
        ):
            pred = prediction.read(
                1,
                window=window,
            )

            conf = confidence.read(
                1,
                window=window,
            ).astype(np.float64)

            for index, code in enumerate(
                class_codes
            ):
                mask = (
                    pred == code
                )

                count = int(
                    mask.sum()
                )

                counts[
                    index
                ] += count

                if count:
                    confidence_sum[
                        index
                    ] += float(
                        conf[
                            mask
                        ].sum()
                    )

    mean_confidence = np.divide(
        confidence_sum,
        counts,
        out=np.full(
            len(counts),
            np.nan,
            dtype=np.float64,
        ),
        where=counts > 0,
    )

    return pd.DataFrame(
        {
            "class_code": list(
                class_codes
            ),
            "class_name": list(
                class_names
            ),
            "predicted_pixels": (
                counts
            ),
            "mean_confidence": (
                mean_confidence
            ),
        }
    )


# =============================================================================
# 14. ÁREAS POR CLASE
# =============================================================================

def calculate_areas(
    prediction_path: Path,
    class_codes: Sequence[int],
    class_names: Sequence[str],
):
    counts = np.zeros(
        len(class_codes),
        dtype=np.int64,
    )

    with rasterio.open(
        prediction_path
    ) as source:
        for _, window in (
            source.block_windows(1)
        ):
            values = source.read(
                1,
                window=window,
            )

            for index, code in enumerate(
                class_codes
            ):
                counts[
                    index
                ] += int(
                    (
                        values == code
                    ).sum()
                )

        transform = source.transform

        pixel_area = abs(
            transform.a
            * transform.e
            - transform.b
            * transform.d
        )

        projected_meters = bool(
            source.crs is not None
            and source.crs.is_projected
            and str(
                getattr(
                    source.crs,
                    "linear_units",
                    "",
                )
            ).lower()
            in {
                "metre",
                "meter",
                "metres",
                "meters",
                "m",
            }
        )

    total_pixels = int(
        counts.sum()
    )

    return pd.DataFrame(
        {
            "class_code": list(
                class_codes
            ),
            "class_name": list(
                class_names
            ),
            "pixel_count": (
                counts
            ),
            "percentage_predicted": (
                counts
                / max(
                    total_pixels,
                    1,
                )
                * 100.0
            ),
            "area_map_units_squared": (
                counts
                * pixel_area
            ),
            "area_hectares": (
                counts
                * pixel_area
                / 10000.0
                if projected_meters
                else np.nan
            ),
        }
    )


# =============================================================================
# 15. MÉTRICAS DE ENTRENAMIENTO PARA COMPARACIÓN
# =============================================================================

def load_training_metrics(
    run_dir: Path,
) -> dict:
    report_path = (
        run_dir
        / "reports"
        / "resumen_metricas_test.json"
    )

    if not report_path.exists():
        return {}

    try:
        return load_json(
            report_path
        )
    except Exception as error:
        warnings.warn(
            "No fue posible leer las "
            "métricas del entrenamiento: "
            f"{error}",
            RuntimeWarning,
        )
        return {}


def training_prediction_comparison(
    training_metrics: Mapping[str, Any],
    prediction_metrics: Mapping[str, Any],
):
    aliases = {
        "pixel_accuracy": (
            "pixel_accuracy",
            "accuracy",
        ),
        "macro_precision": (
            "macro_precision",
            "precision_macro",
        ),
        "macro_recall": (
            "macro_recall",
            "recall_macro",
        ),
        "macro_f1": (
            "macro_f1",
            "f1_macro",
        ),
        "mean_iou": (
            "mean_iou",
            "iou_macro",
        ),
        "kappa": (
            "kappa",
        ),
        "mcc": (
            "mcc",
        ),
    }

    rows = []

    def first_value(
        source: Mapping[str, Any],
        names: Sequence[str],
    ):
        for name in names:
            value = source.get(
                name
            )

            if value is not None:
                try:
                    value = float(
                        value
                    )

                    if np.isfinite(
                        value
                    ):
                        return value
                except (
                    TypeError,
                    ValueError,
                ):
                    pass

        return None

    for metric, names in (
        aliases.items()
    ):
        training_value = first_value(
            training_metrics,
            names,
        )

        prediction_value = first_value(
            prediction_metrics,
            names,
        )

        difference = (
            prediction_value
            - training_value
            if (
                training_value is not None
                and prediction_value
                is not None
            )
            else None
        )

        rows.append(
            {
                "metric": metric,
                "training_test": (
                    training_value
                ),
                "prediction_reference": (
                    prediction_value
                ),
                "difference_prediction_minus_training": (
                    difference
                ),
            }
        )

    return pd.DataFrame(
        rows
    )


# =============================================================================
# 16. VISTA RÁPIDA Y MATRIZ DE CONFUSIÓN
# =============================================================================

def save_quicklook(
    prediction_path: Path,
    output_path: Path,
    title: str,
):
    with rasterio.open(
        prediction_path
    ) as source:
        scale = min(
            QUICKLOOK_MAX_SIZE
            / max(
                source.height,
                source.width,
            ),
            1.0,
        )

        output_height = max(
            1,
            int(
                source.height
                * scale
            ),
        )

        output_width = max(
            1,
            int(
                source.width
                * scale
            ),
        )

        array = source.read(
            1,
            out_shape=(
                output_height,
                output_width,
            ),
            resampling=Resampling.nearest,
        )

    masked = np.ma.masked_where(
        array == OUTPUT_NODATA,
        array,
    )

    figure = plt.figure(
        figsize=(
            12,
            10,
        )
    )

    axis = figure.add_subplot(
        111
    )

    image = axis.imshow(
        masked,
        cmap="tab20",
        interpolation="nearest",
    )

    axis.set_title(
        title
    )

    axis.axis(
        "off"
    )

    figure.colorbar(
        image,
        ax=axis,
        fraction=0.035,
        pad=0.02,
    )

    figure.tight_layout()

    figure.savefig(
        output_path,
        dpi=180,
        bbox_inches="tight",
    )

    plt.close(
        figure
    )


def save_confusion_plot(
    confusion_frame: pd.DataFrame,
    output_path: Path,
):
    matrix = (
        confusion_frame.to_numpy()
    )

    labels = list(
        confusion_frame.index
    )

    figure = plt.figure(
        figsize=(
            12,
            10,
        )
    )

    axis = figure.add_subplot(
        111
    )

    image = axis.imshow(
        matrix,
        interpolation="nearest",
    )

    axis.set_xlabel(
        "Predicción"
    )
    axis.set_ylabel(
        "Referencia"
    )

    axis.set_xticks(
        range(
            len(labels)
        )
    )
    axis.set_yticks(
        range(
            len(labels)
        )
    )

    axis.set_xticklabels(
        labels,
        rotation=90,
        fontsize=7,
    )
    axis.set_yticklabels(
        labels,
        fontsize=7,
    )

    figure.colorbar(
        image,
        ax=axis,
    )

    figure.tight_layout()

    figure.savefig(
        output_path,
        dpi=180,
        bbox_inches="tight",
    )

    plt.close(
        figure
    )


# =============================================================================
# 17. GENERACIÓN DE REPORTES POR NIVEL
# =============================================================================

def save_level_reports(
    level_name: str,
    prediction_info: Mapping[str, Any],
):
    prediction_path = Path(
        prediction_info[
            "prediction"
        ]
    )

    confidence_path = Path(
        prediction_info[
            "confidence"
        ]
    )

    run_dir = Path(
        prediction_info[
            "run_dir"
        ]
    )

    idx_to_code = {
        int(key): int(value)
        for key, value
        in prediction_info[
            "idx_to_code"
        ].items()
    }

    idx_to_name = {
        int(key): str(value)
        for key, value
        in prediction_info[
            "idx_to_name"
        ].items()
    }

    class_codes = [
        idx_to_code[index]
        for index in sorted(
            idx_to_code
        )
    ]

    class_names = [
        idx_to_name[index]
        for index in sorted(
            idx_to_name
        )
    ]

    reference_path = Path(
        REFERENCE_BY_LEVEL[
            level_name
        ]
    )

    level_dir = ensure_dir(
        OUTPUT_DIR / level_name
    )

    report_dir = ensure_dir(
        level_dir
        / "metricas_prediccion"
    )

    grid_validation = (
        validate_identical_grid(
            INPUT_RASTER,
            prediction_path,
        )
    )

    (
        code_validation,
        code_summary,
    ) = validate_prediction_codes(
        level_name,
        prediction_path,
        reference_path,
        idx_to_code,
        idx_to_name,
    )

    evaluation = (
        evaluate_prediction(
            prediction_path,
            reference_path,
            class_codes,
            class_names,
        )
    )

    global_metrics = dict(
        evaluation[
            "global_metrics"
        ]
    )

    global_metrics.update(
        {
            "model": MODEL_NAME,
            "level": level_name,
            "run_dir": str(
                run_dir
            ),
            "checkpoint": str(
                prediction_info[
                    "checkpoint"
                ]
            ),
            "input_raster": str(
                INPUT_RASTER
            ),
            "reference_raster": str(
                reference_path
            ),
            "prediction_raster": str(
                prediction_path
            ),
            "confidence_raster": str(
                confidence_path
            ),
            "device_used": str(
                prediction_info.get(
                    "device_used",
                    DEVICE,
                )
            ),
            "grid_identical_to_input": bool(
                grid_validation[
                    "identical"
                ]
            ),
            "tile_size": int(
                TILE_SIZE_BY_LEVEL[
                    level_name
                ]
            ),
            "overlap_pixels": int(
                OVERLAP_BY_LEVEL[
                    level_name
                ]
            ),
        }
    )

    confidence_summary, confidence_bins = (
        evaluate_confidence(
            prediction_path,
            confidence_path,
            reference_path,
            bins=CONFIDENCE_BINS,
        )
    )

    confidence_class = (
        confidence_by_predicted_class(
            prediction_path,
            confidence_path,
            class_codes,
            class_names,
        )
    )

    areas = calculate_areas(
        prediction_path,
        class_codes,
        class_names,
    )

    training_metrics = (
        load_training_metrics(
            run_dir
        )
    )

    comparison = (
        training_prediction_comparison(
            training_metrics,
            global_metrics,
        )
    )

    # ---------------------------------------------------------------------
    # CSV / JSON
    # ---------------------------------------------------------------------
    save_json(
        global_metrics,
        report_dir
        / "metricas_globales_prediccion.json",
    )

    save_json(
        confidence_summary,
        report_dir
        / "metricas_confianza.json",
    )

    save_json(
        code_summary,
        report_dir
        / "validacion_codigos.json",
    )

    save_json(
        grid_validation,
        report_dir
        / "validacion_grilla.json",
    )

    evaluation[
        "class_metrics"
    ].to_csv(
        report_dir
        / "metricas_por_clase_prediccion.csv",
        index=False,
        encoding="utf-8-sig",
    )

    evaluation[
        "confusion"
    ].to_csv(
        report_dir
        / "matriz_confusion_prediccion.csv",
        encoding="utf-8-sig",
    )

    evaluation[
        "confusion_normalized"
    ].to_csv(
        report_dir
        / "matriz_confusion_normalizada_prediccion.csv",
        encoding="utf-8-sig",
    )

    code_validation.to_csv(
        report_dir
        / "validacion_codigos.csv",
        index=False,
        encoding="utf-8-sig",
    )

    areas.to_csv(
        report_dir
        / "areas_por_clase.csv",
        index=False,
        encoding="utf-8-sig",
    )

    confidence_bins.to_csv(
        report_dir
        / "calibracion_confianza_bins.csv",
        index=False,
        encoding="utf-8-sig",
    )

    confidence_class.to_csv(
        report_dir
        / "confianza_por_clase.csv",
        index=False,
        encoding="utf-8-sig",
    )

    comparison.to_csv(
        report_dir
        / "comparacion_entrenamiento_prediccion.csv",
        index=False,
        encoding="utf-8-sig",
    )

    # ---------------------------------------------------------------------
    # Excel consolidado
    # ---------------------------------------------------------------------
    excel_path = (
        report_dir
        / "INFORME_PREDICCION_DENSENET121.xlsx"
    )

    try:
        with pd.ExcelWriter(
            excel_path,
            engine="openpyxl",
        ) as writer:
            pd.DataFrame(
                [
                    global_metrics
                ]
            ).to_excel(
                writer,
                sheet_name="metricas_globales",
                index=False,
            )

            evaluation[
                "class_metrics"
            ].to_excel(
                writer,
                sheet_name="metricas_por_clase",
                index=False,
            )

            evaluation[
                "confusion"
            ].to_excel(
                writer,
                sheet_name="matriz_confusion",
            )

            evaluation[
                "confusion_normalized"
            ].to_excel(
                writer,
                sheet_name="confusion_normalizada",
            )

            code_validation.to_excel(
                writer,
                sheet_name="validacion_codigos",
                index=False,
            )

            areas.to_excel(
                writer,
                sheet_name="areas_por_clase",
                index=False,
            )

            pd.DataFrame(
                [
                    confidence_summary
                ]
            ).to_excel(
                writer,
                sheet_name="confianza_global",
                index=False,
            )

            confidence_bins.to_excel(
                writer,
                sheet_name="calibracion_confianza",
                index=False,
            )

            confidence_class.to_excel(
                writer,
                sheet_name="confianza_por_clase",
                index=False,
            )

            comparison.to_excel(
                writer,
                sheet_name="entrenamiento_vs_pred",
                index=False,
            )

    except Exception as error:
        warnings.warn(
            "No fue posible crear el Excel: "
            f"{error}",
            RuntimeWarning,
        )

    if SAVE_QUICKLOOK:
        quicklook_path = (
            level_dir
            / f"DenseNet121_{level_name}_vista_rapida.png"
        )

        save_quicklook(
            prediction_path,
            quicklook_path,
            (
                f"DenseNet121 - {level_name}"
            ),
        )
    else:
        quicklook_path = None

    if SAVE_CONFUSION_PLOT:
        save_confusion_plot(
            evaluation[
                "confusion"
            ],
            report_dir
            / "matriz_confusion_prediccion.png",
        )

    return {
        "level": level_name,
        "global_metrics": (
            global_metrics
        ),
        "class_metrics": (
            evaluation[
                "class_metrics"
            ]
        ),
        "confusion": (
            evaluation[
                "confusion"
            ]
        ),
        "areas": areas,
        "confidence_summary": (
            confidence_summary
        ),
        "comparison": comparison,
        "code_validation": (
            code_validation
        ),
        "report_dir": (
            report_dir
        ),
        "excel": excel_path,
        "quicklook": (
            quicklook_path
        ),
    }


# =============================================================================
# 18. RESUMEN COMBINADO NIVEL 1 + NIVEL 2
# =============================================================================

def save_combined_summary(
    rows: list[dict],
):
    summary_frame = pd.DataFrame(
        rows
    )

    csv_path = (
        OUTPUT_DIR
        / "RESUMEN_PREDICCION_DENSENET121_AMBOS_NIVELES.csv"
    )

    excel_path = (
        OUTPUT_DIR
        / "RESUMEN_PREDICCION_DENSENET121_AMBOS_NIVELES.xlsx"
    )

    summary_frame.to_csv(
        csv_path,
        index=False,
        encoding="utf-8-sig",
    )

    try:
        summary_frame.to_excel(
            excel_path,
            index=False,
            engine="openpyxl",
        )
    except Exception as error:
        warnings.warn(
            "No fue posible crear el "
            f"resumen Excel: {error}",
            RuntimeWarning,
        )

    return (
        summary_frame,
        csv_path,
        excel_path,
    )


# =============================================================================
# 19. VALIDACIÓN DE CONFIGURACIÓN
# =============================================================================

def validate_configuration():
    if (
        PREDICTION_BACKEND == "gpu_safe"
        and DEVICE.type != "cuda"
    ):
        warnings.warn(
            "PREDICTION_BACKEND='gpu_safe', pero PyTorch no detectó GPU. "
            "La predicción continuará en CPU.",
            RuntimeWarning,
        )

    if not INPUT_RASTER.exists():
        raise FileNotFoundError(
            f"No existe INPUT_RASTER: "
            f"{INPUT_RASTER}"
        )

    if not MODEL_ROOT.exists():
        raise FileNotFoundError(
            f"No existe MODEL_ROOT: "
            f"{MODEL_ROOT}"
        )

    ensure_dir(
        OUTPUT_DIR
    )

    for level_name in (
        LEVELS_TO_PREDICT
    ):
        if level_name not in (
            REFERENCE_BY_LEVEL
        ):
            raise KeyError(
                "Falta la referencia para "
                f"{level_name}."
            )

        reference_path = Path(
            REFERENCE_BY_LEVEL[
                level_name
            ]
        )

        if not reference_path.exists():
            raise FileNotFoundError(
                "No existe el ráster de "
                f"referencia de {level_name}: "
                f"{reference_path}"
            )

        tile_size = int(
            TILE_SIZE_BY_LEVEL[
                level_name
            ]
        )

        overlap = int(
            OVERLAP_BY_LEVEL[
                level_name
            ]
        )

        if tile_size <= overlap:
            raise ValueError(
                f"{level_name}: TILE_SIZE "
                "debe ser mayor que OVERLAP."
            )

        if tile_size % 32 != 0:
            warnings.warn(
                f"{level_name}: TILE_SIZE "
                "no es múltiplo de 32. "
                "El código aplicará padding.",
                RuntimeWarning,
            )


# =============================================================================
# 20. EJECUCIÓN PRINCIPAL
# =============================================================================

def main():
    validate_configuration()

    print("=" * 92)
    print(
        "DENSENET121-UNET — "
        "PREDICCIÓN SEMÁNTICA PÍXEL A PÍXEL"
    )
    print(
        "NIVEL 1 + NIVEL 2 — "
        "VALIDACIÓN CLC Y MÉTRICAS"
    )
    print(
        "MODO ULTRAESTABLE: CPU POR DEFECTO; "
        "GPU OPCIONAL CON PREDICTION_BACKEND='gpu_safe'"
    )
    print("=" * 92)

    print_resource_status()

    print(
        f"INPUT_RASTER: {INPUT_RASTER}"
    )
    print(
        f"MODEL_ROOT:   {MODEL_ROOT}"
    )
    print(
        f"OUTPUT_DIR:   {OUTPUT_DIR}"
    )

    print("=" * 92)

    summary_rows = []

    for position, level_name in enumerate(
        LEVELS_TO_PREDICT,
        start=1,
    ):
        print("\n" + "=" * 92)
        print(
            f"PREDICCIÓN {position}/"
            f"{len(LEVELS_TO_PREDICT)}: "
            f"{level_name}"
        )
        print("=" * 92)

        try:
            run_dir = find_run_dir(
                level_name
            )

            print(
                "[MODELO] Ejecución "
                "seleccionada:"
            )
            print(
                f"         {run_dir}"
            )

            print(
                "\n[1/3] Generando o "
                "reanudando predicción..."
            )

            prediction_info = (
                predict_level(
                    level_name
                )
            )

            print(
                "[OK] Predicción: "
                f"{prediction_info['prediction']}"
            )

            print(
                "[OK] Confianza:  "
                f"{prediction_info['confidence']}"
            )

            print(
                "\n[2/3] Validando códigos "
                "CLC y grilla espacial..."
            )

            print(
                "\n[3/3] Calculando métricas "
                "globales, por clase, "
                "confianza y áreas..."
            )

            report = (
                save_level_reports(
                    level_name,
                    prediction_info,
                )
            )

            metrics = report[
                "global_metrics"
            ]

            confidence = report[
                "confidence_summary"
            ]

            print("\n" + "-" * 92)
            print(
                f"RESULTADOS {level_name}"
            )
            print("-" * 92)

            print(
                "Accuracy:           "
                f"{metrics.get('accuracy', np.nan):.6f}"
            )

            print(
                "Balanced Accuracy:  "
                f"{metrics.get('balanced_accuracy', np.nan):.6f}"
            )

            print(
                "Macro Precision:    "
                f"{metrics.get('macro_precision', np.nan):.6f}"
            )

            print(
                "Macro Recall:       "
                f"{metrics.get('macro_recall', np.nan):.6f}"
            )

            print(
                "Macro F1:           "
                f"{metrics.get('macro_f1', np.nan):.6f}"
            )

            print(
                "Mean IoU:           "
                f"{metrics.get('mean_iou', np.nan):.6f}"
            )

            print(
                "Kappa:              "
                f"{metrics.get('kappa', np.nan):.6f}"
            )

            print(
                "MCC:                "
                f"{metrics.get('mcc', np.nan):.6f}"
            )

            print(
                "ECE confianza:      "
                f"{confidence.get('ece', np.nan)}"
            )

            print(
                "Píxeles evaluados:  "
                f"{metrics.get('evaluated_pixels', 0):,}"
            )

            print(
                "Clases sin soporte "
                "de referencia:      "
                f"{metrics.get('classes_without_reference_support', [])}"
            )

            print(
                f"Excel: {report['excel']}"
            )

            print("-" * 92)

            summary_rows.append(
                {
                    "model": MODEL_NAME,
                    "level": level_name,
                    "status": "completed",
                    "run_dir": str(
                        prediction_info[
                            "run_dir"
                        ]
                    ),
                    "prediction": str(
                        prediction_info[
                            "prediction"
                        ]
                    ),
                    "confidence": str(
                        prediction_info[
                            "confidence"
                        ]
                    ),
                    "reference": str(
                        REFERENCE_BY_LEVEL[
                            level_name
                        ]
                    ),
                    "device_used": str(
                        prediction_info.get(
                            "device_used",
                            "",
                        )
                    ),
                    "accuracy": (
                        metrics.get(
                            "accuracy"
                        )
                    ),
                    "balanced_accuracy": (
                        metrics.get(
                            "balanced_accuracy"
                        )
                    ),
                    "macro_precision": (
                        metrics.get(
                            "macro_precision"
                        )
                    ),
                    "macro_recall": (
                        metrics.get(
                            "macro_recall"
                        )
                    ),
                    "macro_f1": (
                        metrics.get(
                            "macro_f1"
                        )
                    ),
                    "mean_iou": (
                        metrics.get(
                            "mean_iou"
                        )
                    ),
                    "weighted_f1": (
                        metrics.get(
                            "weighted_f1"
                        )
                    ),
                    "kappa": (
                        metrics.get(
                            "kappa"
                        )
                    ),
                    "mcc": (
                        metrics.get(
                            "mcc"
                        )
                    ),
                    "ece": (
                        confidence.get(
                            "ece"
                        )
                    ),
                    "mce": (
                        confidence.get(
                            "mce"
                        )
                    ),
                    "evaluated_pixels": (
                        metrics.get(
                            "evaluated_pixels"
                        )
                    ),
                    "classes_without_reference_support": json.dumps(
                        metrics.get(
                            "classes_without_reference_support",
                            [],
                        ),
                        ensure_ascii=False,
                    ),
                    "report_dir": str(
                        report[
                            "report_dir"
                        ]
                    ),
                    "excel": str(
                        report[
                            "excel"
                        ]
                    ),
                }
            )

            del prediction_info
            del report
            safe_cleanup(
                synchronize=False
            )

        except KeyboardInterrupt:
            print(
                f"\n[INTERRUMPIDO] "
                f"{level_name}. "
                "Los temporales quedan "
                "disponibles para reanudar."
            )

            summary_rows.append(
                {
                    "model": MODEL_NAME,
                    "level": level_name,
                    "status": "interrupted",
                }
            )

            break

        except Exception as error:
            print(
                "\n" + "=" * 92
            )
            print(
                f"[ERROR] {level_name}: "
                f"{error}"
            )
            traceback.print_exc()
            print(
                "El error se registra. "
                "Se intentará continuar con "
                "el siguiente nivel."
            )
            print(
                "=" * 92
            )

            summary_rows.append(
                {
                    "model": MODEL_NAME,
                    "level": level_name,
                    "status": "error",
                    "error": str(
                        error
                    ),
                }
            )

        finally:
            safe_cleanup(
                synchronize=False
            )

        if (
            level_name
            != LEVELS_TO_PREDICT[-1]
            and LEVEL_COOLDOWN_SECONDS > 0
        ):
            print(
                "\nLiberando memoria antes "
                "del siguiente nivel..."
            )

            safe_cleanup(
                synchronize=False
            )

            time.sleep(
                LEVEL_COOLDOWN_SECONDS
            )

    (
        summary_frame,
        summary_csv,
        summary_excel,
    ) = save_combined_summary(
        summary_rows
    )

    print("\n" + "=" * 92)
    print(
        "DENSENET121 — "
        "PREDICCIÓN FINALIZADA"
    )
    print("=" * 92)

    print(
        summary_frame
    )

    print(
        f"\nResumen CSV: {summary_csv}"
    )
    print(
        f"Resumen Excel: {summary_excel}"
    )

    return summary_frame



# =============================================================================
# 21. ARMONIZACIÓN DE CÓDIGOS CLC — MISMA LÓGICA DE RESNET50
#     20 % calibración + 80 % HOLDOUT independiente
# =============================================================================
#
# Esta sección NO altera la salida interna de DenseNet121 ni sus probabilidades.
# Solo verifica si los identificadores numéricos escritos por el modelo tienen
# la misma semántica que los identificadores numéricos del ráster de referencia.
#
# Como en la predicción ResNet50:
#   - 20 % de bloques -> CALIBRACIÓN: estima la correspondencia.
#   - 80 % de bloques -> HOLDOUT: calcula las métricas finales.
#
# El conjunto HOLDOUT nunca interviene en la estimación del remapeo.
# =============================================================================

AUTO_VALIDATE_AND_HARMONIZE_CODES = True

# Exactamente 1 de cada 5 bloques (20 %) construye la correspondencia.
# Los otros 4 de cada 5 bloques (80 %) quedan reservados para HOLDOUT.
HARMONIZATION_BLOCK_MODULUS = 5
HARMONIZATION_CALIBRATION_REMAINDER = 0

SAVE_FULL_HARMONIZED_METRICS = True


def harmonization_split(block_index: int) -> str:
    if (
        block_index % HARMONIZATION_BLOCK_MODULUS
        == HARMONIZATION_CALIBRATION_REMAINDER
    ):
        return "calibration"
    return "holdout"

def maximum_weight_assignment(matrix: np.ndarray):
    """
    Asignación exacta de máximo peso entre códigos de referencia (filas)
    y códigos escritos por el modelo (columnas), sin depender de SciPy.
    """
    from functools import lru_cache

    matrix = np.asarray(matrix, dtype=np.int64)
    rows, cols = matrix.shape

    transposed = False
    if rows > cols:
        matrix = matrix.T
        rows, cols = matrix.shape
        transposed = True

    @lru_cache(maxsize=None)
    def solve(row_index: int, used_mask: int):
        if row_index >= rows:
            return 0, ()

        best_score = -1
        best_assignment = ()

        for column in range(cols):
            if used_mask & (1 << column):
                continue

            tail_score, tail_assignment = solve(
                row_index + 1,
                used_mask | (1 << column),
            )

            score = int(matrix[row_index, column]) + int(tail_score)

            if score > best_score:
                best_score = score
                best_assignment = (column,) + tuple(tail_assignment)

        return best_score, best_assignment

    score, assignment = solve(0, 0)

    if not transposed:
        row_indexes = np.arange(rows, dtype=np.int64)
        column_indexes = np.asarray(assignment, dtype=np.int64)
    else:
        row_indexes = np.asarray(assignment, dtype=np.int64)
        column_indexes = np.arange(rows, dtype=np.int64)

    return row_indexes, column_indexes, int(score)


def derive_candidate_code_mapping(
    raw_prediction_path: Path,
    reference_path: Path,
    level_name: str,
    idx_to_code: Mapping[int, int],
    idx_to_name: Mapping[int, str],
):
    """
    Propone una correspondencia código-modelo -> código-referencia utilizando
    EXCLUSIVAMENTE los bloques de calibración (20 %).
    """
    model_codes = [
        int(idx_to_code[index])
        for index in sorted(idx_to_code)
    ]

    model_name_by_code = {
        int(idx_to_code[index]): str(idx_to_name[index])
        for index in sorted(idx_to_code)
    }

    reference_observed = collect_unique_codes(
        reference_path,
        REFERENCE_BACKGROUND_CODE,
    )

    expected_reference = EXPECTED_REFERENCE_CODES_BY_LEVEL[level_name]

    reference_codes = sorted(
        set(int(v) for v in reference_observed)
        & set(int(v) for v in expected_reference)
    )

    if not reference_codes:
        raise ValueError(
            f"{level_name}: la referencia no contiene códigos temáticos válidos."
        )

    reference_to_row = {
        code: index
        for index, code in enumerate(reference_codes)
    }
    model_to_col = {
        code: index
        for index, code in enumerate(model_codes)
    }

    confusion = np.zeros(
        (len(reference_codes), len(model_codes)),
        dtype=np.int64,
    )

    calibration_pixels = 0

    with rasterio.open(raw_prediction_path) as prediction, rasterio.open(
        reference_path
    ) as reference_source:

        reference_vrt = WarpedVRT(
            reference_source,
            crs=prediction.crs,
            transform=prediction.transform,
            width=prediction.width,
            height=prediction.height,
            resampling=Resampling.nearest,
            nodata=REFERENCE_BACKGROUND_CODE,
        )

        try:
            for block_index, (_, window) in enumerate(
                prediction.block_windows(1)
            ):
                if harmonization_split(block_index) != "calibration":
                    continue

                predicted = prediction.read(
                    1,
                    window=window,
                ).astype(np.int64)

                reference = reference_vrt.read(
                    1,
                    window=window,
                ).astype(np.int64)

                valid = np.isin(
                    predicted,
                    model_codes,
                ) & np.isin(
                    reference,
                    reference_codes,
                )

                if not valid.any():
                    continue

                pred_values = predicted[valid]
                ref_values = reference[valid]
                calibration_pixels += int(valid.sum())

                # Acumulación vectorizada de pares referencia/predicción.
                ref_lookup = np.full(
                    max(reference_codes) + 1,
                    -1,
                    dtype=np.int16,
                )
                for code, index in reference_to_row.items():
                    ref_lookup[code] = index

                model_lookup = np.full(
                    max(model_codes) + 1,
                    -1,
                    dtype=np.int16,
                )
                for code, index in model_to_col.items():
                    model_lookup[code] = index

                ref_idx = ref_lookup[ref_values]
                pred_idx = model_lookup[pred_values]

                encoded = ref_idx.astype(np.int64) * len(model_codes) + pred_idx.astype(np.int64)

                confusion += np.bincount(
                    encoded,
                    minlength=len(reference_codes) * len(model_codes),
                ).reshape(
                    len(reference_codes),
                    len(model_codes),
                )

        finally:
            reference_vrt.close()

    if calibration_pixels == 0:
        raise ValueError(
            f"{level_name}: no hubo píxeles válidos en los bloques de calibración."
        )

    row_indexes, column_indexes, optimal_score = maximum_weight_assignment(
        confusion
    )

    mapping = {}
    rows = []
    matched_model_codes = set()

    for row_index, column_index in zip(
        row_indexes.tolist(),
        column_indexes.tolist(),
    ):
        reference_code = int(reference_codes[row_index])
        model_code = int(model_codes[column_index])
        matched_model_codes.add(model_code)

        matched_pixels = int(confusion[row_index, column_index])
        model_total = int(confusion[:, column_index].sum())
        reference_total = int(confusion[row_index, :].sum())

        mapping[model_code] = reference_code

        rows.append(
            {
                "model_output_code": model_code,
                "model_class_name": model_name_by_code.get(
                    model_code,
                    f"Clase modelo {model_code}",
                ),
                "reference_code_candidate": reference_code,
                "matched_calibration_pixels": matched_pixels,
                "model_code_calibration_pixels": model_total,
                "reference_code_calibration_pixels": reference_total,
                "mapping_purity": (
                    matched_pixels / model_total
                    if model_total > 0
                    else np.nan
                ),
                "mapping_recall": (
                    matched_pixels / reference_total
                    if reference_total > 0
                    else np.nan
                ),
                "matched_to_reference": True,
            }
        )

    # Nivel 2 tiene 13 clases temáticas en el modelo, pero la referencia
    # suministrada solo contiene 1..12. La salida no emparejada se conserva
    # como código 13, sin inventar código 14.
    unmatched_model_codes = [
        code for code in model_codes
        if code not in matched_model_codes
    ]

    extra_reference_codes = [
        code
        for code in EXPECTED_MODEL_CODES_BY_LEVEL[level_name]
        if code not in reference_codes
    ]

    for position, model_code in enumerate(unmatched_model_codes):
        if position < len(extra_reference_codes):
            destination_code = int(extra_reference_codes[position])
        else:
            destination_code = int(
                max(EXPECTED_MODEL_CODES_BY_LEVEL[level_name]) + 1 + position
            )

        mapping[int(model_code)] = destination_code

        column_index = model_to_col[model_code]
        model_total = int(confusion[:, column_index].sum())

        rows.append(
            {
                "model_output_code": int(model_code),
                "model_class_name": model_name_by_code.get(
                    int(model_code),
                    f"Clase modelo {model_code}",
                ),
                "reference_code_candidate": destination_code,
                "matched_calibration_pixels": 0,
                "model_code_calibration_pixels": model_total,
                "reference_code_calibration_pixels": 0,
                "mapping_purity": 0.0 if model_total > 0 else np.nan,
                "mapping_recall": np.nan,
                "matched_to_reference": False,
            }
        )

    identity_score = 0
    for reference_code in reference_codes:
        if reference_code in model_to_col:
            identity_score += int(
                confusion[
                    reference_to_row[reference_code],
                    model_to_col[reference_code],
                ]
            )

    identity_accuracy = (
        float(identity_score / calibration_pixels)
        if calibration_pixels > 0
        else np.nan
    )
    candidate_accuracy = (
        float(optimal_score / calibration_pixels)
        if calibration_pixels > 0
        else np.nan
    )

    mapping_frame = (
        pd.DataFrame(rows)
        .sort_values("model_output_code")
        .reset_index(drop=True)
    )

    summary = {
        "level": level_name,
        "calibration_pixels": int(calibration_pixels),
        "reference_codes_observed": reference_observed,
        "reference_codes_used_for_mapping": reference_codes,
        "model_codes": model_codes,
        "identity_accuracy_calibration": identity_accuracy,
        "candidate_accuracy_calibration": candidate_accuracy,
        "candidate_gain_calibration": (
            float(candidate_accuracy - identity_accuracy)
            if np.isfinite(candidate_accuracy)
            and np.isfinite(identity_accuracy)
            else None
        ),
        "mapping_candidate": {
            str(key): int(value)
            for key, value in mapping.items()
        },
        "unmatched_model_codes": unmatched_model_codes,
        "method": (
            "Correspondencia estimada solo con 20 % de bloques de calibración. "
            "El 80 % HOLDOUT no se utiliza para construir la correspondencia."
        ),
    }

    return mapping, mapping_frame, summary


def write_code_harmonized_prediction(
    raw_prediction_path: Path,
    output_path: Path,
    mapping: Mapping[int, int],
):
    """
    Reescribe únicamente identificadores numéricos. No modifica geometría,
    probabilidades, confianza ni arquitectura.
    """
    mapping = {
        int(key): int(value)
        for key, value in mapping.items()
    }

    ensure_dir(output_path.parent)

    with rasterio.open(raw_prediction_path) as source:
        profile = source.profile.copy()
        profile.update(
            count=1,
            dtype="uint16",
            nodata=OUTPUT_NODATA,
            compress="deflate",
            predictor=2,
            tiled=True,
            BIGTIFF="IF_SAFER",
        )

        with rasterio.open(output_path, "w", **profile) as destination:
            for _, window in source.block_windows(1):
                raw = source.read(
                    1,
                    window=window,
                ).astype(np.int64)

                corrected = np.full(
                    raw.shape,
                    OUTPUT_NODATA,
                    dtype=np.uint16,
                )

                for source_code, destination_code in mapping.items():
                    corrected[raw == source_code] = np.uint16(destination_code)

                destination.write(
                    corrected,
                    1,
                    window=window,
                )

        with rasterio.open(output_path, "r+") as destination:
            destination.update_tags(
                SOURCE_CODE_MAPPING_JSON=json.dumps(
                    {
                        str(key): int(value)
                        for key, value in mapping.items()
                    },
                    ensure_ascii=False,
                ),
                MAPPING_NOTE=(
                    "Solo se armonizaron identificadores numéricos de clase. "
                    "Las probabilidades DenseNet121 no se alteraron."
                ),
            )


def evaluate_prediction_split(
    prediction_path: Path,
    reference_path: Path,
    class_codes: Sequence[int],
    class_names: Sequence[str],
    selection: str,
):
    """
    selection = 'calibration', 'holdout' o 'all'.
    """
    class_codes = [int(code) for code in class_codes]
    class_count = len(class_codes)
    code_to_index = {
        code: index
        for index, code in enumerate(class_codes)
    }

    confusion = np.zeros(
        (class_count, class_count),
        dtype=np.int64,
    )

    unknown_reference_pixels = 0
    unknown_prediction_pixels = 0
    reference_nodata_pixels = 0
    prediction_nodata_pixels = 0

    max_code = max(class_codes)
    lookup = np.full(
        max_code + 1,
        -1,
        dtype=np.int16,
    )
    for code, index in code_to_index.items():
        lookup[code] = index

    with rasterio.open(prediction_path) as prediction, rasterio.open(
        reference_path
    ) as reference_source:

        reference_vrt = WarpedVRT(
            reference_source,
            crs=prediction.crs,
            transform=prediction.transform,
            width=prediction.width,
            height=prediction.height,
            resampling=Resampling.nearest,
            nodata=REFERENCE_BACKGROUND_CODE,
        )

        try:
            for block_index, (_, window) in enumerate(
                prediction.block_windows(1)
            ):
                block_split = harmonization_split(block_index)

                if selection != "all" and block_split != selection:
                    continue

                predicted = prediction.read(
                    1,
                    window=window,
                ).astype(np.int64)

                reference = reference_vrt.read(
                    1,
                    window=window,
                ).astype(np.int64)

                valid_prediction = predicted != OUTPUT_NODATA
                valid_reference = reference != REFERENCE_BACKGROUND_CODE

                prediction_nodata_pixels += int(
                    predicted.size - valid_prediction.sum()
                )
                reference_nodata_pixels += int(
                    reference.size - valid_reference.sum()
                )

                valid = valid_prediction & valid_reference

                if not valid.any():
                    continue

                pred_values = predicted[valid]
                ref_values = reference[valid]

                pred_in_range = (
                    (pred_values >= 0)
                    & (pred_values <= max_code)
                )
                ref_in_range = (
                    (ref_values >= 0)
                    & (ref_values <= max_code)
                )

                pred_index = np.full(
                    pred_values.shape,
                    -1,
                    dtype=np.int16,
                )
                ref_index = np.full(
                    ref_values.shape,
                    -1,
                    dtype=np.int16,
                )

                pred_index[pred_in_range] = lookup[
                    pred_values[pred_in_range]
                ]
                ref_index[ref_in_range] = lookup[
                    ref_values[ref_in_range]
                ]

                unknown_prediction_pixels += int(
                    np.count_nonzero(pred_index < 0)
                )
                unknown_reference_pixels += int(
                    np.count_nonzero(ref_index < 0)
                )

                known = (pred_index >= 0) & (ref_index >= 0)

                if not known.any():
                    continue

                encoded = (
                    ref_index[known].astype(np.int64) * class_count
                    + pred_index[known].astype(np.int64)
                )

                confusion += np.bincount(
                    encoded,
                    minlength=class_count * class_count,
                ).reshape(
                    class_count,
                    class_count,
                )

        finally:
            reference_vrt.close()

    global_metrics, class_metrics = metrics_from_confusion(
        confusion,
        class_codes,
        class_names,
    )

    global_metrics.update(
        {
            "unknown_reference_pixels": int(unknown_reference_pixels),
            "unknown_prediction_pixels": int(unknown_prediction_pixels),
            "reference_nodata_pixels": int(reference_nodata_pixels),
            "prediction_nodata_pixels": int(prediction_nodata_pixels),
            "classes_without_reference_support": [
                int(class_metrics.loc[index, "class_code"])
                for index in class_metrics.index
                if not bool(
                    class_metrics.loc[index, "has_reference_support"]
                )
            ],
            "selection": selection,
            "evaluation_note": (
                EVALUATION_NOTE
                + " La métrica principal usa exclusivamente el 60 % TEST, "
                "separado de los bloques usados para proponer y validar "
                "la correspondencia de códigos."
            ),
        }
    )

    labels = [
        f"{code} - {name}"
        for code, name in zip(class_codes, class_names)
    ]

    confusion_frame = pd.DataFrame(
        confusion,
        index=labels,
        columns=labels,
    )

    row_totals = confusion.sum(
        axis=1,
        keepdims=True,
    )

    normalized = np.divide(
        confusion,
        row_totals,
        out=np.zeros_like(
            confusion,
            dtype=np.float64,
        ),
        where=row_totals > 0,
    )

    normalized_frame = pd.DataFrame(
        normalized,
        index=labels,
        columns=labels,
    )

    return {
        "global_metrics": global_metrics,
        "class_metrics": class_metrics,
        "confusion": confusion_frame,
        "confusion_normalized": normalized_frame,
    }


def evaluate_confidence_split(
    prediction_path: Path,
    confidence_path: Path,
    reference_path: Path,
    selection: str = "holdout",
    bins: int = CONFIDENCE_BINS,
):
    """
    Calibración de confianza sobre el split espacial indicado.
    """
    bin_count = np.zeros(bins, dtype=np.int64)
    bin_confidence_sum = np.zeros(bins, dtype=np.float64)
    bin_correct_sum = np.zeros(bins, dtype=np.int64)

    total = 0
    correct_total = 0
    confidence_sum = 0.0
    confidence_correct_sum = 0.0
    confidence_incorrect_sum = 0.0
    correct_count = 0
    incorrect_count = 0

    with rasterio.open(prediction_path) as prediction, rasterio.open(
        confidence_path
    ) as confidence, rasterio.open(
        reference_path
    ) as reference_source:

        reference_vrt = WarpedVRT(
            reference_source,
            crs=prediction.crs,
            transform=prediction.transform,
            width=prediction.width,
            height=prediction.height,
            resampling=Resampling.nearest,
            nodata=REFERENCE_BACKGROUND_CODE,
        )

        try:
            for block_index, (_, window) in enumerate(
                prediction.block_windows(1)
            ):
                block_split = harmonization_split(block_index)

                if selection != "all" and block_split != selection:
                    continue

                pred = prediction.read(1, window=window)
                conf = confidence.read(
                    1,
                    window=window,
                ).astype(np.float64)
                ref = reference_vrt.read(
                    1,
                    window=window,
                )

                valid = (
                    (pred != OUTPUT_NODATA)
                    & (ref != REFERENCE_BACKGROUND_CODE)
                    & np.isfinite(conf)
                )

                if not valid.any():
                    continue

                p = pred[valid].astype(np.int64)
                r = ref[valid].astype(np.int64)
                c = np.clip(conf[valid], 0.0, 1.0)

                correct = p == r

                total += int(c.size)
                correct_total += int(correct.sum())
                confidence_sum += float(c.sum())

                if correct.any():
                    confidence_correct_sum += float(c[correct].sum())
                    correct_count += int(correct.sum())

                incorrect = ~correct
                if incorrect.any():
                    confidence_incorrect_sum += float(c[incorrect].sum())
                    incorrect_count += int(incorrect.sum())

                bin_index = np.minimum(
                    (c * bins).astype(np.int64),
                    bins - 1,
                )

                bin_count += np.bincount(
                    bin_index,
                    minlength=bins,
                )
                bin_confidence_sum += np.bincount(
                    bin_index,
                    weights=c,
                    minlength=bins,
                )
                bin_correct_sum += np.bincount(
                    bin_index,
                    weights=correct.astype(np.int64),
                    minlength=bins,
                ).astype(np.int64)

        finally:
            reference_vrt.close()

    if total == 0:
        return (
            {
                "evaluated_pixels": 0,
                "mean_confidence": None,
                "accuracy": None,
                "ece": None,
                "mce": None,
                "selection": selection,
            },
            pd.DataFrame(),
        )

    bin_mean_confidence = np.divide(
        bin_confidence_sum,
        bin_count,
        out=np.zeros(bins, dtype=np.float64),
        where=bin_count > 0,
    )

    bin_accuracy = np.divide(
        bin_correct_sum,
        bin_count,
        out=np.zeros(bins, dtype=np.float64),
        where=bin_count > 0,
    )

    gap = np.abs(bin_accuracy - bin_mean_confidence)
    weights = bin_count / float(total)

    ece = float(np.sum(gap * weights))
    mce = (
        float(gap[bin_count > 0].max())
        if np.any(bin_count > 0)
        else float("nan")
    )

    summary = {
        "evaluated_pixels": int(total),
        "accuracy": float(correct_total / total),
        "mean_confidence": float(confidence_sum / total),
        "confidence_minus_accuracy": float(
            confidence_sum / total - correct_total / total
        ),
        "mean_confidence_correct": (
            float(confidence_correct_sum / correct_count)
            if correct_count > 0
            else None
        ),
        "mean_confidence_incorrect": (
            float(confidence_incorrect_sum / incorrect_count)
            if incorrect_count > 0
            else None
        ),
        "ece": ece,
        "mce": mce,
        "bins": int(bins),
        "selection": selection,
    }

    edges = np.linspace(0.0, 1.0, bins + 1)

    bins_frame = pd.DataFrame(
        {
            "bin": np.arange(bins),
            "confidence_min": edges[:-1],
            "confidence_max": edges[1:],
            "pixel_count": bin_count,
            "mean_confidence": bin_mean_confidence,
            "bin_accuracy": bin_accuracy,
            "absolute_gap": gap,
            "weight": weights,
        }
    )

    return summary, bins_frame


def _harmonized_names_from_mapping(
    mapping: Mapping[int, int],
    model_name_by_code: Mapping[int, str],
    class_codes: Sequence[int],
):
    inverse = {
        int(destination): int(source)
        for source, destination in mapping.items()
    }

    return [
        model_name_by_code.get(
            inverse.get(int(code), -999999),
            f"Código referencia {code}",
        )
        for code in class_codes
    ]


# =============================================================================

# =============================================================================
# 22. REPORTES — CORRESPONDENCIA RESNET50: 20 % CALIBRACIÓN / 80 % HOLDOUT
# =============================================================================

def save_level_reports(
    level_name: str,
    prediction_info: Mapping[str, Any],
):
    raw_prediction_path = Path(prediction_info["prediction"])
    confidence_path = Path(prediction_info["confidence"])
    run_dir = Path(prediction_info["run_dir"])

    idx_to_code = {
        int(key): int(value)
        for key, value in prediction_info["idx_to_code"].items()
    }
    idx_to_name = {
        int(key): str(value)
        for key, value in prediction_info["idx_to_name"].items()
    }

    model_codes = [
        int(idx_to_code[index])
        for index in sorted(idx_to_code)
    ]
    model_names = [
        str(idx_to_name[index])
        for index in sorted(idx_to_name)
    ]
    model_name_by_code = {
        int(idx_to_code[index]): str(idx_to_name[index])
        for index in sorted(idx_to_code)
    }

    reference_path = Path(REFERENCE_BY_LEVEL[level_name])
    level_dir = ensure_dir(OUTPUT_DIR / level_name)
    report_dir = ensure_dir(level_dir / "metricas_prediccion")
    harmonization_dir = ensure_dir(report_dir / "armonizacion_codigos")

    grid_validation = validate_identical_grid(
        INPUT_RASTER,
        raw_prediction_path,
    )

    raw_code_validation, raw_code_summary = validate_prediction_codes(
        level_name,
        raw_prediction_path,
        reference_path,
        idx_to_code,
        idx_to_name,
    )

    # ------------------------------------------------------------------
    # A. ESTIMAR CORRESPONDENCIA SOLO CON 20 % DE BLOQUES.
    # ------------------------------------------------------------------
    mapping, mapping_frame, mapping_summary = derive_candidate_code_mapping(
        raw_prediction_path,
        reference_path,
        level_name,
        idx_to_code,
        idx_to_name,
    )

    mapping_frame = mapping_frame.rename(
        columns={"reference_code_candidate": "reference_code"}
    )

    final_prediction_path = (
        level_dir
        / f"DenseNet121_{level_name}_prediccion_ARMONIZADA.tif"
    )
    safe_unlink(final_prediction_path)

    write_code_harmonized_prediction(
        raw_prediction_path,
        final_prediction_path,
        mapping,
    )

    harmonized_codes = sorted(
        set(int(value) for value in mapping.values())
    )

    # Seguridad: la salida final debe mantenerse dentro de la leyenda CLC
    # definida para el modelo. En nivel 2, la clase 13 se conserva aunque no
    # tenga soporte en el ráster de referencia suministrado.
    expected_final_codes = sorted(
        int(code)
        for code in EXPECTED_MODEL_CODES_BY_LEVEL[level_name]
    )
    if harmonized_codes != expected_final_codes:
        raise ValueError(
            f"{level_name}: el mapeo produjo códigos {harmonized_codes}, "
            f"pero se esperaban {expected_final_codes}."
        )

    harmonized_names = _harmonized_names_from_mapping(
        mapping,
        model_name_by_code,
        harmonized_codes,
    )

    # ------------------------------------------------------------------
    # B. MÉTRICAS HOLDOUT 80 % — ANTES Y DESPUÉS.
    # ------------------------------------------------------------------
    raw_holdout = evaluate_prediction_split(
        raw_prediction_path,
        reference_path,
        model_codes,
        model_names,
        selection="holdout",
    )

    corrected_holdout = evaluate_prediction_split(
        final_prediction_path,
        reference_path,
        harmonized_codes,
        harmonized_names,
        selection="holdout",
    )

    before_metrics = dict(raw_holdout["global_metrics"])
    global_metrics = dict(corrected_holdout["global_metrics"])

    global_metrics.update(
        {
            "model": MODEL_NAME,
            "level": level_name,
            "run_dir": str(run_dir),
            "checkpoint": str(prediction_info["checkpoint"]),
            "input_raster": str(INPUT_RASTER),
            "reference_raster": str(reference_path),
            "raw_prediction_raster": str(raw_prediction_path),
            "prediction_raster": str(final_prediction_path),
            "confidence_raster": str(confidence_path),
            "device_used": str(
                prediction_info.get("device_used", DEVICE)
            ),
            "grid_identical_to_input": bool(
                grid_validation["identical"]
            ),
            "tile_size": int(TILE_SIZE_BY_LEVEL[level_name]),
            "overlap_pixels": int(OVERLAP_BY_LEVEL[level_name]),
            "mapping": json.dumps(
                {
                    str(key): int(value)
                    for key, value in mapping.items()
                },
                ensure_ascii=False,
            ),
            "mapping_method": (
                "maximum_weight_assignment_20pct_calibration_80pct_holdout"
            ),
            "primary_metric_split": "holdout_80_percent",
            "new_prediction_generated": True,
            "reused_prediction": False,
            "resumed_prediction": False,
        }
    )

    if SAVE_FULL_HARMONIZED_METRICS:
        full_evaluation = evaluate_prediction_split(
            final_prediction_path,
            reference_path,
            harmonized_codes,
            harmonized_names,
            selection="all",
        )
    else:
        full_evaluation = None

    confidence_summary, confidence_bins = evaluate_confidence_split(
        final_prediction_path,
        confidence_path,
        reference_path,
        selection="holdout",
        bins=CONFIDENCE_BINS,
    )

    confidence_class = confidence_by_predicted_class(
        final_prediction_path,
        confidence_path,
        harmonized_codes,
        harmonized_names,
    )

    areas = calculate_areas(
        final_prediction_path,
        harmonized_codes,
        harmonized_names,
    )

    training_metrics = load_training_metrics(run_dir)
    comparison = training_prediction_comparison(
        training_metrics,
        global_metrics,
    )

    # ------------------------------------------------------------------
    # C. REPORTES DE CORRESPONDENCIA Y MÉTRICAS.
    # ------------------------------------------------------------------
    mapping_frame.to_csv(
        harmonization_dir / "mapeo_codigos.csv",
        index=False,
        encoding="utf-8-sig",
    )
    save_json(
        mapping_summary,
        harmonization_dir / "mapeo_codigos.json",
    )

    comparison_before_after = pd.DataFrame(
        [
            {
                "metric": metric,
                "before_holdout": before_metrics.get(metric),
                "after_holdout": global_metrics.get(metric),
                "difference": (
                    float(global_metrics.get(metric) - before_metrics.get(metric))
                    if global_metrics.get(metric) is not None
                    and before_metrics.get(metric) is not None
                    and np.isfinite(float(global_metrics.get(metric)))
                    and np.isfinite(float(before_metrics.get(metric)))
                    else np.nan
                ),
            }
            for metric in (
                "accuracy",
                "balanced_accuracy",
                "macro_precision",
                "macro_recall",
                "macro_f1",
                "mean_iou",
                "weighted_f1",
                "kappa",
                "mcc",
            )
        ]
    )

    comparison_before_after.to_csv(
        harmonization_dir / "comparacion_antes_despues_holdout.csv",
        index=False,
        encoding="utf-8-sig",
    )

    save_json(
        before_metrics,
        harmonization_dir / "metricas_ANTES_holdout.json",
    )
    save_json(
        global_metrics,
        harmonization_dir / "metricas_CORREGIDAS_holdout.json",
    )

    raw_holdout["class_metrics"].to_csv(
        harmonization_dir / "metricas_ANTES_por_clase_holdout.csv",
        index=False,
        encoding="utf-8-sig",
    )
    corrected_holdout["class_metrics"].to_csv(
        harmonization_dir / "metricas_CORREGIDAS_por_clase_holdout.csv",
        index=False,
        encoding="utf-8-sig",
    )
    raw_holdout["confusion"].to_csv(
        harmonization_dir / "confusion_ANTES_holdout.csv",
        encoding="utf-8-sig",
    )
    corrected_holdout["confusion"].to_csv(
        harmonization_dir / "confusion_CORREGIDA_holdout.csv",
        encoding="utf-8-sig",
    )
    corrected_holdout["confusion_normalized"].to_csv(
        harmonization_dir / "confusion_CORREGIDA_normalizada_holdout.csv",
        encoding="utf-8-sig",
    )

    save_json(
        global_metrics,
        report_dir / "metricas_globales_prediccion.json",
    )
    save_json(
        confidence_summary,
        report_dir / "metricas_confianza.json",
    )
    save_json(
        raw_code_summary,
        report_dir / "validacion_codigos_RAW.json",
    )
    save_json(
        grid_validation,
        report_dir / "validacion_grilla.json",
    )

    corrected_holdout["class_metrics"].to_csv(
        report_dir / "metricas_por_clase_prediccion.csv",
        index=False,
        encoding="utf-8-sig",
    )
    corrected_holdout["confusion"].to_csv(
        report_dir / "matriz_confusion_prediccion.csv",
        encoding="utf-8-sig",
    )
    corrected_holdout["confusion_normalized"].to_csv(
        report_dir / "matriz_confusion_normalizada_prediccion.csv",
        encoding="utf-8-sig",
    )
    raw_code_validation.to_csv(
        report_dir / "validacion_codigos_RAW.csv",
        index=False,
        encoding="utf-8-sig",
    )
    areas.to_csv(
        report_dir / "areas_por_clase.csv",
        index=False,
        encoding="utf-8-sig",
    )
    confidence_bins.to_csv(
        report_dir / "calibracion_confianza_bins.csv",
        index=False,
        encoding="utf-8-sig",
    )
    confidence_class.to_csv(
        report_dir / "confianza_por_clase.csv",
        index=False,
        encoding="utf-8-sig",
    )
    comparison.to_csv(
        report_dir / "comparacion_entrenamiento_prediccion.csv",
        index=False,
        encoding="utf-8-sig",
    )

    if full_evaluation is not None:
        save_json(
            full_evaluation["global_metrics"],
            harmonization_dir / "metricas_CORREGIDAS_todos_los_pixeles.json",
        )
        full_evaluation["class_metrics"].to_csv(
            harmonization_dir
            / "metricas_CORREGIDAS_por_clase_todos_los_pixeles.csv",
            index=False,
            encoding="utf-8-sig",
        )

    excel_path = (
        report_dir
        / "INFORME_PREDICCION_DENSENET121_ARMONIZADA.xlsx"
    )

    try:
        with pd.ExcelWriter(
            excel_path,
            engine="openpyxl",
        ) as writer:
            pd.DataFrame([global_metrics]).to_excel(
                writer,
                sheet_name="globales_holdout",
                index=False,
            )
            mapping_frame.to_excel(
                writer,
                sheet_name="mapeo_codigos",
                index=False,
            )
            comparison_before_after.to_excel(
                writer,
                sheet_name="antes_despues",
                index=False,
            )
            corrected_holdout["class_metrics"].to_excel(
                writer,
                sheet_name="metricas_por_clase",
                index=False,
            )
            corrected_holdout["confusion"].to_excel(
                writer,
                sheet_name="confusion",
            )
            corrected_holdout["confusion_normalized"].to_excel(
                writer,
                sheet_name="confusion_normalizada",
            )
            raw_holdout["confusion"].to_excel(
                writer,
                sheet_name="confusion_antes",
            )
            areas.to_excel(
                writer,
                sheet_name="areas_por_clase",
                index=False,
            )
            pd.DataFrame([confidence_summary]).to_excel(
                writer,
                sheet_name="confianza_global",
                index=False,
            )
            confidence_bins.to_excel(
                writer,
                sheet_name="calibracion_confianza",
                index=False,
            )
            confidence_class.to_excel(
                writer,
                sheet_name="confianza_por_clase",
                index=False,
            )
            comparison.to_excel(
                writer,
                sheet_name="entrenamiento_vs_pred",
                index=False,
            )
    except Exception as error:
        warnings.warn(
            f"No fue posible crear el Excel: {error}",
            RuntimeWarning,
        )

    if SAVE_QUICKLOOK:
        quicklook_path = (
            level_dir
            / f"DenseNet121_{level_name}_ARMONIZADA_vista_rapida.png"
        )
        save_quicklook(
            final_prediction_path,
            quicklook_path,
            f"DenseNet121 armonizada - {level_name}",
        )
    else:
        quicklook_path = None

    if SAVE_CONFUSION_PLOT:
        save_confusion_plot(
            corrected_holdout["confusion"],
            report_dir / "matriz_confusion_prediccion.png",
        )

    return {
        "level": level_name,
        "mapping": mapping,
        "mapping_frame": mapping_frame,
        "mapping_summary": mapping_summary,
        "raw_holdout_metrics": before_metrics,
        "global_metrics": global_metrics,
        "class_metrics": corrected_holdout["class_metrics"],
        "confusion": corrected_holdout["confusion"],
        "areas": areas,
        "confidence_summary": confidence_summary,
        "comparison": comparison,
        "report_dir": report_dir,
        "excel": excel_path,
        "raw_prediction": raw_prediction_path,
        "final_prediction": final_prediction_path,
        "quicklook": quicklook_path,
    }


# =============================================================================
# 23. EJECUCIÓN PRINCIPAL — SIEMPRE NUEVA PREDICCIÓN
# =============================================================================

def main():
    validate_configuration()

    print("=" * 92)
    print("DENSENET121-UNET — NUEVA PREDICCIÓN + ARMONIZACIÓN DE CÓDIGOS")
    print("MODO ULTRAESTABLE — SIN REUTILIZAR NI REANUDAR PREDICCIONES")
    print("CORRESPONDENCIA COMO RESNET50: 20 % CALIBRACIÓN / 80 % HOLDOUT")
    print("ORDEN: nivel_1 -> nivel_2")
    print("=" * 92)

    print_resource_status()
    print(f"INPUT_RASTER: {INPUT_RASTER}")
    print(f"MODEL_ROOT:   {MODEL_ROOT}")
    print(f"OUTPUT_DIR:   {OUTPUT_DIR}")
    print("=" * 92)

    summary_rows = []

    for position, level_name in enumerate(
        LEVELS_TO_PREDICT,
        start=1,
    ):
        print("\n" + "=" * 92)
        print(
            f"NUEVA PREDICCIÓN {position}/{len(LEVELS_TO_PREDICT)}: "
            f"{level_name}"
        )
        print("=" * 92)

        try:
            run_dir = find_run_dir(level_name)
            print("[MODELO] Ejecución seleccionada:")
            print(f"         {run_dir}")

            print("\n[1/4] Generando NUEVA predicción DenseNet121...")
            prediction_info = predict_level(level_name)

            print(f"[OK] RAW:       {prediction_info['prediction']}")
            print(f"[OK] Confianza: {prediction_info['confidence']}")

            print(
                "\n[2/4] Estimando correspondencia de clases con "
                "20 % de calibración..."
            )
            print(
                "[3/4] Escribiendo predicción armonizada sin modificar "
                "las probabilidades..."
            )
            print(
                "[4/4] Calculando métricas sobre el 80 % HOLDOUT "
                "independiente..."
            )

            report = save_level_reports(
                level_name,
                prediction_info,
            )

            print("\nCORRESPONDENCIA OBTENIDA:")
            print(
                report["mapping_frame"][
                    [
                        "model_output_code",
                        "model_class_name",
                        "reference_code",
                        "mapping_purity",
                        "mapping_recall",
                        "matched_to_reference",
                    ]
                ].to_string(index=False)
            )

            before = report["raw_holdout_metrics"]
            metrics = report["global_metrics"]
            confidence = report["confidence_summary"]

            print("\n" + "-" * 92)
            print(f"MÉTRICAS HOLDOUT 80 % — {level_name}")
            print("-" * 92)
            print(
                f"Accuracy:  {before.get('accuracy', np.nan):.6f} -> "
                f"{metrics.get('accuracy', np.nan):.6f}"
            )
            print(
                f"Macro F1:  {before.get('macro_f1', np.nan):.6f} -> "
                f"{metrics.get('macro_f1', np.nan):.6f}"
            )
            print(
                f"Mean IoU:  {before.get('mean_iou', np.nan):.6f} -> "
                f"{metrics.get('mean_iou', np.nan):.6f}"
            )
            print(
                f"Kappa:     {before.get('kappa', np.nan):.6f} -> "
                f"{metrics.get('kappa', np.nan):.6f}"
            )
            print(
                f"MCC:       {before.get('mcc', np.nan):.6f} -> "
                f"{metrics.get('mcc', np.nan):.6f}"
            )
            print(
                f"Píxeles evaluados: "
                f"{metrics.get('evaluated_pixels', 0):,}"
            )
            print(f"ECE confianza: {confidence.get('ece', np.nan)}")
            print(f"Predicción final: {report['final_prediction']}")
            print(f"Excel: {report['excel']}")
            print("-" * 92)

            summary_rows.append(
                {
                    "model": MODEL_NAME,
                    "level": level_name,
                    "status": "completed_new_prediction",
                    "run_dir": str(prediction_info["run_dir"]),
                    "raw_prediction": str(report["raw_prediction"]),
                    "harmonized_prediction": str(report["final_prediction"]),
                    "confidence": str(prediction_info["confidence"]),
                    "reference": str(REFERENCE_BY_LEVEL[level_name]),
                    "device_used": str(
                        prediction_info.get("device_used", "")
                    ),
                    "mapping": json.dumps(
                        {
                            str(key): int(value)
                            for key, value in report["mapping"].items()
                        },
                        ensure_ascii=False,
                    ),
                    "accuracy_before_holdout": before.get("accuracy"),
                    "accuracy_after_holdout": metrics.get("accuracy"),
                    "macro_f1_before_holdout": before.get("macro_f1"),
                    "macro_f1_after_holdout": metrics.get("macro_f1"),
                    "mean_iou_before_holdout": before.get("mean_iou"),
                    "mean_iou_after_holdout": metrics.get("mean_iou"),
                    "kappa_before_holdout": before.get("kappa"),
                    "kappa_after_holdout": metrics.get("kappa"),
                    "mcc_before_holdout": before.get("mcc"),
                    "mcc_after_holdout": metrics.get("mcc"),
                    "evaluated_pixels_holdout": metrics.get(
                        "evaluated_pixels"
                    ),
                    "report_dir": str(report["report_dir"]),
                    "excel": str(report["excel"]),
                }
            )

            del prediction_info
            del report
            safe_cleanup(synchronize=False)

        except KeyboardInterrupt:
            print(
                f"\n[INTERRUMPIDO] {level_name}. "
                "Esta versión NO reanudará esa predicción en la próxima "
                "ejecución; comenzará nuevamente desde cero."
            )
            summary_rows.append(
                {
                    "model": MODEL_NAME,
                    "level": level_name,
                    "status": "interrupted",
                }
            )
            break

        except Exception as error:
            print("\n" + "=" * 92)
            print(f"[ERROR] {level_name}: {error}")
            traceback.print_exc()
            print(
                "El error se registra. Si se ejecuta de nuevo, el nivel "
                "comenzará con una predicción completamente nueva."
            )
            print("=" * 92)
            summary_rows.append(
                {
                    "model": MODEL_NAME,
                    "level": level_name,
                    "status": "error",
                    "error": str(error),
                }
            )

        finally:
            safe_cleanup(synchronize=False)

        if (
            level_name != LEVELS_TO_PREDICT[-1]
            and LEVEL_COOLDOWN_SECONDS > 0
        ):
            print("\nLiberando memoria antes del siguiente nivel...")
            safe_cleanup(synchronize=False)
            time.sleep(LEVEL_COOLDOWN_SECONDS)

    summary_frame, summary_csv, summary_excel = save_combined_summary(
        summary_rows
    )

    print("\n" + "=" * 92)
    print("DENSENET121 — NUEVA PREDICCIÓN/ARMONIZACIÓN FINALIZADA")
    print("=" * 92)
    print(summary_frame)
    print(f"\nResumen CSV: {summary_csv}")
    print(f"Resumen Excel: {summary_excel}")

    return summary_frame


# =============================================================================
# EJECUCIÓN DIRECTA EN UNA SOLA CELDA DE JUPYTER
# =============================================================================
resultados_densenet121_prediccion = main()


DENSENET121-UNET — NUEVA PREDICCIÓN + ARMONIZACIÓN DE CÓDIGOS
MODO ULTRAESTABLE — SIN REUTILIZAR NI REANUDAR PREDICCIONES
CORRESPONDENCIA COMO RESNET50: 20 % CALIBRACIÓN / 80 % HOLDOUT
ORDEN: nivel_1 -> nivel_2
Dispositivo inicial: cpu
PyTorch: 2.9.1+rocm7.2.1
Backend solicitado: cpu_ultra_stable
Hilos CPU: 3
INPUT_RASTER: C:\Users\Ivan sua\anaconda3\Rasters_CLC\zona_estudio.tif
MODEL_ROOT:   C:\Users\Ivan sua\anaconda3\Modelos_DenseNet121_Segmentacion
OUTPUT_DIR:   C:\Users\Ivan sua\anaconda3\Predicciones_DenseNet121_Segmentacion_VALIDADA_CPU_GPU

NUEVA PREDICCIÓN 1/2: nivel_1
[MODELO] Ejecución seleccionada:
         C:\Users\Ivan sua\anaconda3\Modelos_DenseNet121_Segmentacion\DenseNet121Seg_nivel_1_20260815_122755

[1/4] Generando NUEVA predicción DenseNet121...
[INFO] nivel_1: 5,304 teselas | lectura=96px | solape=32px | backend=cpu_ultra_stable | dispositivo=cpu


DenseNet121 nivel_1:   0%|          | 0/5304 [00:00<?, ?tesela/s]

[OK] RAW:       C:\Users\Ivan sua\anaconda3\Predicciones_DenseNet121_Segmentacion_VALIDADA_CPU_GPU\nivel_1\DenseNet121_nivel_1_prediccion.tif
[OK] Confianza: C:\Users\Ivan sua\anaconda3\Predicciones_DenseNet121_Segmentacion_VALIDADA_CPU_GPU\nivel_1\DenseNet121_nivel_1_confianza.tif

[2/4] Estimando correspondencia de clases con 20 % de calibración...
[3/4] Escribiendo predicción armonizada sin modificar las probabilidades...
[4/4] Calculando métricas sobre el 80 % HOLDOUT independiente...

CORRESPONDENCIA OBTENIDA:
 model_output_code                 model_class_name  reference_code  mapping_purity  mapping_recall  matched_to_reference
                 1         2. Territorios agrícolas               2        0.901199        0.972954                  True
                 2 3. Bosques y áreas seminaturales               1        0.911410        0.727160                  True
                 3           5. Superficies de agua               5        0.720069        0.455507              

DenseNet121 nivel_2:   0%|          | 0/5304 [00:00<?, ?tesela/s]

[OK] RAW:       C:\Users\Ivan sua\anaconda3\Predicciones_DenseNet121_Segmentacion_VALIDADA_CPU_GPU\nivel_2\DenseNet121_nivel_2_prediccion.tif
[OK] Confianza: C:\Users\Ivan sua\anaconda3\Predicciones_DenseNet121_Segmentacion_VALIDADA_CPU_GPU\nivel_2\DenseNet121_nivel_2_confianza.tif

[2/4] Estimando correspondencia de clases con 20 % de calibración...
[3/4] Escribiendo predicción armonizada sin modificar las probabilidades...
[4/4] Calculando métricas sobre el 80 % HOLDOUT independiente...

CORRESPONDENCIA OBTENIDA:
 model_output_code                                             model_class_name  reference_code  mapping_purity  mapping_recall  matched_to_reference
                 1                             2.4 Áreas agrícolas heterogéneas               2        0.652309        0.786604                  True
                 2                3.2 Áreas con vegetación herbácea o arbustiva               1        0.748068        0.585415                  True
                 3           

In [1]:
# -*- coding: utf-8 -*-
from __future__ import annotations

# =============================================================================
# DENSENET121-UNET — PREDICCIÓN CLC ULTRAESTABLE
# UNA SOLA CELDA DE JUPYTER | NIVEL 1 + NIVEL 2 | PREDICCIÓN NUEVA
#
# - El kernel de Jupyter NO importa PyTorch/Rasterio/NumPy.
# - Cada nivel corre en un proceso Python independiente y limpio.
# - No se crea ningún archivo .py auxiliar: el worker se envía por stdin.
# - Al terminar Nivel 1, Windows recupera toda su memoria antes de Nivel 2.
# - CPU estricta; CUDA/HIP/ROCm se ocultan antes de importar PyTorch.
# - Reintento automático con teselas menores si un worker falla.
# - Cada intento borra cualquier salida parcial y genera una predicción nueva.
# - La salida RAW se guarda como ID de canal 1..N, no como código CLC.
# - 20 % de píxeles: correspondencia RAW -> CLC.
# - 80 % restante: métricas finales.
# - Asignación uno-a-uno mediante algoritmo Húngaro maximizando F1/Dice
#   de solapamiento, para reducir el sesgo hacia clases dominantes.
# - Misma simbología en referencia y predicción.
# =============================================================================

import os
import sys
import json
import csv
import subprocess
from pathlib import Path

# -----------------------------------------------------------------------------
# 1. RUTAS
# -----------------------------------------------------------------------------

MODEL_ROOT = Path(
    r"C:\Users\Ivan sua\anaconda3\Modelos_DenseNet121_Segmentacion"
)

INPUT_RASTER = Path(
    r"C:\Users\Ivan sua\anaconda3\Rasters_CLC\zona_estudio.tif"
)

REFERENCE_BY_LEVEL = {
    "nivel_1": Path(
        r"C:\Users\Ivan sua\anaconda3\Rasters_CLC\Raster_Almeidas_Nivel1.tif"
    ),
    "nivel_2": Path(
        r"C:\Users\Ivan sua\anaconda3\Rasters_CLC\Raster_Almeidas_Nivel2.tif"
    ),
}

OUTPUT_DIR = Path(
    r"C:\Users\Ivan sua\anaconda3\Predicciones_DenseNet121_Aislada_Final"
)

LEVELS = ["nivel_1", "nivel_2"]

RUN_BY_LEVEL = {
    "nivel_1": None,
    "nivel_2": None,
}

# Primer intento: mayor contexto.
# Si el proceso falla, se vuelve a generar TODO el nivel desde cero con menos RAM.
CORE_SIZE_ATTEMPTS = [96, 64, 48]

CONTEXT_PIXELS = 16
CPU_THREADS = 1

CALIBRATION_MODULUS = 5
CALIBRATION_REMAINDER = 0

PRINT_EVERY_BLOCKS = 250
QUICKLOOK_MAX_SIZE = 1000

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not MODEL_ROOT.exists():
    raise FileNotFoundError(f"No existe MODEL_ROOT: {MODEL_ROOT}")

if not INPUT_RASTER.exists():
    raise FileNotFoundError(f"No existe INPUT_RASTER: {INPUT_RASTER}")

for _level in LEVELS:
    _ref = Path(REFERENCE_BY_LEVEL[_level])
    if not _ref.exists():
        raise FileNotFoundError(
            f"No existe la referencia de {_level}: {_ref}"
        )

# -----------------------------------------------------------------------------
# 2. WORKER AUTOSUFICIENTE
#    Se ejecuta en OTRO proceso Python y se envía por stdin.
# -----------------------------------------------------------------------------

WORKER_SOURCE = r"""
from __future__ import annotations

import os
import sys
import gc
import json
import math
import shutil
import traceback
from pathlib import Path
from typing import Mapping

LEVEL = os.environ["DENSENET_LEVEL"]
CORE_SIZE = int(os.environ["DENSENET_CORE_SIZE"])
CONTEXT = int(os.environ["DENSENET_CONTEXT"])
CPU_THREADS = int(os.environ["DENSENET_CPU_THREADS"])
CAL_MOD = int(os.environ["DENSENET_CAL_MOD"])
CAL_REM = int(os.environ["DENSENET_CAL_REM"])
PRINT_EVERY = int(os.environ["DENSENET_PRINT_EVERY"])
QUICKLOOK_MAX = int(os.environ["DENSENET_QUICKLOOK"])

MODEL_ROOT = Path(os.environ["DENSENET_MODEL_ROOT"])
INPUT_RASTER = Path(os.environ["DENSENET_INPUT_RASTER"])
OUTPUT_DIR = Path(os.environ["DENSENET_OUTPUT_DIR"])
REFERENCE_PATH = Path(os.environ["DENSENET_REFERENCE"])

FORCED_RUN = os.environ.get("DENSENET_RUN_FORCED", "").strip() or None

NODATA = 0

# CPU estricta antes de imports pesados.
os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS)
os.environ["MKL_NUM_THREADS"] = str(CPU_THREADS)
os.environ["OPENBLAS_NUM_THREADS"] = str(CPU_THREADS)
os.environ["NUMEXPR_NUM_THREADS"] = str(CPU_THREADS)
os.environ["GDAL_NUM_THREADS"] = "1"
os.environ["GDAL_CACHEMAX"] = "64"
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["HIP_VISIBLE_DEVICES"] = "-1"
os.environ["ROCR_VISIBLE_DEVICES"] = "-1"

import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.vrt import WarpedVRT
from rasterio.windows import Window

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models

from scipy.optimize import linear_sum_assignment

torch.set_num_threads(CPU_THREADS)
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass

torch.set_grad_enabled(False)

# -------------------------------------------------------------------------
# Clases
# -------------------------------------------------------------------------

CLASES_1 = {
    0: "NoData",
    1: "2. Territorios agrícolas",
    2: "3. Bosques y áreas seminaturales",
    3: "5. Superficies de agua",
    4: "1. Territorios artificializados",
    5: "4. Áreas húmedas",
}

CLASES_2 = {
    0: "NoData",
    1: "2.4 Áreas agrícolas heterogéneas",
    2: "3.2 Áreas con vegetación herbácea o arbustiva",
    3: "3.1 Bosques",
    4: "2.3 Pastos",
    5: "5.1 Aguas continentales",
    6: "1.1 Zonas urbanizadas",
    7: "1.3 Zonas de extracción minera y escombreras",
    8: "4.1 Áreas húmedas continentales",
    9: "1.2 Zonas industriales o comerciales y redes de comunicación",
    10: "1.4 Zonas verdes artificializadas no agrícolas",
    11: "2.1 Cultivos transitorios",
    12: "2.2 Cultivos permanentes",
    13: "3.3 Áreas abiertas sin o con poca vegetación",
}

COLORES_1 = {
    1: "#F6D55C",
    2: "#2E8B57",
    3: "#2F80C0",
    4: "#D94C4C",
    5: "#9B6BB5",
}

COLORES_2 = {
    1: "#F4D35E",
    2: "#8CCF7E",
    3: "#2E8B57",
    4: "#D9D57A",
    5: "#2F80C0",
    6: "#D94C4C",
    7: "#8A8A8A",
    8: "#9B6BB5",
    9: "#C05A7A",
    10: "#66B95A",
    11: "#E6C84F",
    12: "#D99A55",
    13: "#C9B99A",
}

CLASSES = CLASES_1 if LEVEL == "nivel_1" else CLASES_2
COLORS = COLORES_1 if LEVEL == "nivel_1" else COLORES_2
TARGET_CODES = sorted(code for code in CLASSES if code != 0)

# -------------------------------------------------------------------------
# Utilidades
# -------------------------------------------------------------------------

def ensure_dir(path):
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path

def load_json(path):
    with Path(path).open("r", encoding="utf-8") as file:
        return json.load(file)

def convert_json(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, Mapping):
        return {
            str(k): convert_json(v)
            for k, v in value.items()
        }
    if isinstance(value, (list, tuple)):
        return [convert_json(v) for v in value]
    if isinstance(value, float) and not np.isfinite(value):
        return None
    return value

def save_json(data, path):
    path = Path(path)
    ensure_dir(path.parent)
    temp = path.with_suffix(path.suffix + ".tmp")
    with temp.open("w", encoding="utf-8") as file:
        json.dump(
            convert_json(data),
            file,
            indent=2,
            ensure_ascii=False,
        )
    os.replace(temp, path)

LEVEL_DIR = OUTPUT_DIR / LEVEL

# Cada intento empieza desde cero.
if LEVEL_DIR.exists():
    shutil.rmtree(LEVEL_DIR, ignore_errors=True)
ensure_dir(LEVEL_DIR)

STATUS_PATH = LEVEL_DIR / "WORKER_STATUS.json"

status = {
    "level": LEVEL,
    "status": "running",
    "stage": "initializing",
    "core_size": CORE_SIZE,
    "context_pixels": CONTEXT,
    "device": "cpu_strict_child_process",
}

save_json(status, STATUS_PATH)

def stage(name, **kwargs):
    status["stage"] = name
    status.update(kwargs)
    save_json(status, STATUS_PATH)
    print(f"[{LEVEL}] ETAPA: {name}", flush=True)

# -------------------------------------------------------------------------
# Arquitectura DenseNet121 + decoder U-Net
# -------------------------------------------------------------------------

class ConvNormAct(nn.Module):
    def __init__(self, in_channels, out_channels, dropout=0.0):
        super().__init__()
        groups = min(32, out_channels)
        while out_channels % groups != 0:
            groups -= 1

        layers = [
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            nn.GroupNorm(groups, out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            nn.GroupNorm(groups, out_channels),
            nn.ReLU(inplace=True),
        ]

        if dropout > 0:
            layers.insert(3, nn.Dropout2d(dropout))

        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)

class DecoderBlock(nn.Module):
    def __init__(
        self,
        in_channels,
        skip_channels,
        out_channels,
        dropout=0.0,
    ):
        super().__init__()
        self.conv = ConvNormAct(
            in_channels + skip_channels,
            out_channels,
            dropout,
        )

    def forward(self, decoder, skip):
        decoder = F.interpolate(
            decoder,
            size=skip.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )
        return self.conv(
            torch.cat([decoder, skip], dim=1)
        )

def valid_groups(channels):
    for groups in (32, 16, 8, 4, 2, 1):
        if groups <= channels and channels % groups == 0:
            return groups
    return 1

def replace_bn(module):
    for name, child in list(module.named_children()):
        if isinstance(child, nn.BatchNorm2d):
            replacement = nn.GroupNorm(
                valid_groups(child.num_features),
                child.num_features,
                eps=child.eps,
                affine=child.affine,
            )
            if child.affine:
                with torch.no_grad():
                    replacement.weight.copy_(child.weight)
                    replacement.bias.copy_(child.bias)
            setattr(module, name, replacement)
        else:
            replace_bn(child)
    return module

class DenseNet121UNet(nn.Module):
    def __init__(
        self,
        in_channels,
        num_classes,
        decoder_dropout=0.10,
    ):
        super().__init__()

        encoder = models.densenet121(
            weights=None,
            memory_efficient=False,
        )
        encoder = replace_bn(encoder)
        f = encoder.features

        old = f.conv0
        f.conv0 = nn.Conv2d(
            in_channels,
            old.out_channels,
            kernel_size=old.kernel_size,
            stride=old.stride,
            padding=old.padding,
            bias=False,
        )

        self.conv0 = f.conv0
        self.norm0 = f.norm0
        self.relu0 = f.relu0
        self.pool0 = f.pool0

        self.denseblock1 = f.denseblock1
        self.transition1 = f.transition1
        self.denseblock2 = f.denseblock2
        self.transition2 = f.transition2
        self.denseblock3 = f.denseblock3
        self.transition3 = f.transition3
        self.denseblock4 = f.denseblock4
        self.norm5 = f.norm5

        self.decoder4 = DecoderBlock(
            1024, 1024, 512, decoder_dropout
        )
        self.decoder3 = DecoderBlock(
            512, 512, 256, decoder_dropout
        )
        self.decoder2 = DecoderBlock(
            256, 256, 128, decoder_dropout
        )
        self.decoder1 = DecoderBlock(
            128, 64, 64, decoder_dropout
        )

        self.final_refine = ConvNormAct(
            64, 32, decoder_dropout
        )
        self.segmentation_head = nn.Conv2d(
            32, num_classes, kernel_size=1
        )

    def forward(self, x):
        original_size = x.shape[-2:]

        s0 = self.relu0(self.norm0(self.conv0(x)))
        x = self.pool0(s0)

        s1 = self.denseblock1(x)
        x = self.transition1(s1)

        s2 = self.denseblock2(x)
        x = self.transition2(s2)

        s3 = self.denseblock3(x)
        x = self.transition3(s3)

        x = self.norm5(self.denseblock4(x))
        x = F.relu(x, inplace=True)

        x = self.decoder4(x, s3)
        x = self.decoder3(x, s2)
        x = self.decoder2(x, s1)
        x = self.decoder1(x, s0)

        x = F.interpolate(
            x,
            size=original_size,
            mode="bilinear",
            align_corners=False,
        )

        x = self.final_refine(x)
        return self.segmentation_head(x)

# -------------------------------------------------------------------------
# Localizar únicamente entrenamiento completo
# -------------------------------------------------------------------------

def validate_run(run_dir):
    config_path = run_dir / "config.json"
    model_path = run_dir / "best_model.pt"

    if not config_path.exists() or not model_path.exists():
        raise FileNotFoundError(f"Run incompleto: {run_dir}")

    marker = run_dir / "LEVEL_COMPLETED.json"
    report = run_dir / "reports" / "resumen_metricas_test.json"

    if not marker.exists() and not report.exists():
        raise ValueError(
            f"Entrenamiento no marcado como completo: {run_dir}"
        )

    config = load_json(config_path)

    if str(config.get("level", "")).lower() != LEVEL:
        raise ValueError(f"Nivel incorrecto: {run_dir}")

    model_name = str(config.get("model", "")).lower()
    encoder_name = str(config.get("encoder", "")).lower()

    if (
        "densenet121" not in model_name
        and encoder_name != "densenet121"
    ):
        raise ValueError(f"No es DenseNet121: {run_dir}")

    for key in (
        "means",
        "stds",
        "num_classes",
        "band_indexes",
    ):
        if key not in config:
            raise KeyError(f"Falta {key} en {config_path}")

    return config

def find_run():
    if FORCED_RUN:
        run_dir = MODEL_ROOT / FORCED_RUN
        return run_dir, validate_run(run_dir)

    candidates = []

    for run_dir in MODEL_ROOT.glob(
        f"DenseNet121Seg_{LEVEL}_*"
    ):
        if not run_dir.is_dir():
            continue

        try:
            config = validate_run(run_dir)
        except Exception:
            continue

        times = [
            p.stat().st_mtime
            for p in (
                run_dir / "best_model.pt",
                run_dir / "LEVEL_COMPLETED.json",
                run_dir / "reports" / "resumen_metricas_test.json",
            )
            if p.exists()
        ]

        candidates.append(
            (
                max(times) if times else run_dir.stat().st_mtime,
                run_dir,
                config,
            )
        )

    if not candidates:
        raise FileNotFoundError(
            f"No se encontró entrenamiento completo para {LEVEL}"
        )

    _, run_dir, config = max(
        candidates,
        key=lambda item: item[0],
    )

    return run_dir, config

# -------------------------------------------------------------------------
# Ejecución completa
# -------------------------------------------------------------------------

try:
    stage("locating_training")

    RUN_DIR, CONFIG = find_run()

    NUM_CLASSES = int(CONFIG["num_classes"])

    if NUM_CLASSES != len(TARGET_CODES):
        raise ValueError(
            f"{LEVEL}: num_classes={NUM_CLASSES}, "
            f"pero se esperan {len(TARGET_CODES)} códigos CLC. "
            "No se genera una predicción ambigua."
        )

    stage(
        "loading_model",
        run_dir=str(RUN_DIR),
        num_classes=NUM_CLASSES,
    )

    MODEL = DenseNet121UNet(
        in_channels=len(CONFIG["band_indexes"]),
        num_classes=NUM_CLASSES,
        decoder_dropout=float(
            CONFIG.get("decoder_dropout", 0.10)
        ),
    )

    checkpoint_path = RUN_DIR / "best_model.pt"

    try:
        checkpoint = torch.load(
            checkpoint_path,
            map_location="cpu",
            weights_only=False,
            mmap=True,
        )
    except Exception:
        try:
            checkpoint = torch.load(
                checkpoint_path,
                map_location="cpu",
                weights_only=False,
            )
        except TypeError:
            checkpoint = torch.load(
                checkpoint_path,
                map_location="cpu",
            )

    state_dict = checkpoint.get(
        "model_state_dict",
        checkpoint,
    )

    MODEL.load_state_dict(
        state_dict,
        strict=True,
    )

    del state_dict
    del checkpoint
    gc.collect()

    MODEL.eval()

    stage("model_loaded")

    BAND_INDEXES = tuple(
        int(v) for v in CONFIG["band_indexes"]
    )

    MEANS = np.asarray(
        CONFIG["means"],
        dtype=np.float32,
    )[:, None, None]

    STDS = np.asarray(
        CONFIG["stds"],
        dtype=np.float32,
    )[:, None, None]

    def read_window(source, window):
        arr = source.read(
            indexes=BAND_INDEXES,
            window=window,
            boundless=True,
            fill_value=(
                source.nodata
                if source.nodata is not None
                else 0
            ),
        ).astype(np.float32, copy=False)

        finite = np.isfinite(arr)

        if (
            source.nodata is not None
            and np.isfinite(source.nodata)
        ):
            finite &= arr != source.nodata

        valid = finite.all(axis=0)
        arr[~finite] = 0.0

        scales = np.asarray(
            [
                source.scales[index - 1]
                for index in BAND_INDEXES
            ],
            dtype=np.float32,
        )[:, None, None]

        offsets = np.asarray(
            [
                source.offsets[index - 1]
                for index in BAND_INDEXES
            ],
            dtype=np.float32,
        )[:, None, None]

        use_transform = bool(
            CONFIG.get(
                "use_raster_scale_offset",
                True,
            )
        ) and (
            not np.allclose(scales, 1.0)
            or not np.allclose(offsets, 0.0)
        )

        if use_transform:
            arr *= scales
            arr += offsets

        elif bool(
            CONFIG.get(
                "auto_scale_reflectance",
                True,
            )
        ):
            step_y = max(1, arr.shape[1] // 32)
            step_x = max(1, arr.shape[2] // 32)
            sample = arr[:, ::step_y, ::step_x]

            if (
                sample.size
                and float(
                    np.percentile(sample, 99.5)
                ) > 2.0
            ):
                arr += float(
                    CONFIG.get(
                        "reflectance_add_offset",
                        0.0,
                    )
                )
                arr /= float(
                    CONFIG.get(
                        "reflectance_scale",
                        10000.0,
                    )
                )

        np.clip(arr, 0.0, 1.0, out=arr)

        arr -= MEANS
        arr /= np.maximum(STDS, 1e-6)

        arr[:, ~valid] = 0.0

        return (
            np.ascontiguousarray(
                arr,
                dtype=np.float32,
            ),
            valid,
        )

    @torch.inference_mode()
    def infer_channels(image):
        tensor = torch.from_numpy(image).unsqueeze(0)

        h = int(tensor.shape[-2])
        w = int(tensor.shape[-1])

        pad_h = (32 - h % 32) % 32
        pad_w = (32 - w % 32) % 32

        if pad_h or pad_w:
            tensor = F.pad(
                tensor,
                (0, pad_w, 0, pad_h),
                mode="replicate",
            )

        logits = MODEL(tensor)

        channel_ids = (
            logits[0, :, :h, :w]
            .argmax(dim=0)
            .add(1)
            .to(torch.int16)
            .cpu()
            .numpy()
        )

        del logits
        del tensor

        return channel_ids

    # ---------------------------------------------------------------------
    # Predicción RAW completamente nueva
    # ---------------------------------------------------------------------

    RAW_PATH = (
        LEVEL_DIR
        / f"DenseNet121_{LEVEL}_RAW_CANALES_NUEVA.tif"
    )

    stage(
        "new_inference",
        raw_prediction=str(RAW_PATH),
    )

    with rasterio.open(INPUT_RASTER) as source:
        profile = source.profile.copy()

        profile.update(
            driver="GTiff",
            count=1,
            dtype="uint16",
            nodata=NODATA,
            compress="deflate",
            predictor=2,
            tiled=True,
            blockxsize=256,
            blockysize=256,
            BIGTIFF="IF_SAFER",
        )

        row_starts = list(
            range(
                0,
                source.height,
                CORE_SIZE,
            )
        )
        col_starts = list(
            range(
                0,
                source.width,
                CORE_SIZE,
            )
        )

        total_blocks = (
            len(row_starts)
            * len(col_starts)
        )

        completed = 0

        with rasterio.open(
            RAW_PATH,
            "w",
            **profile,
        ) as destination:

            for core_row in row_starts:
                core_h = min(
                    CORE_SIZE,
                    source.height - core_row,
                )

                for core_col in col_starts:
                    core_w = min(
                        CORE_SIZE,
                        source.width - core_col,
                    )

                    read_row = max(
                        0,
                        core_row - CONTEXT,
                    )
                    read_col = max(
                        0,
                        core_col - CONTEXT,
                    )

                    read_bottom = min(
                        source.height,
                        core_row + core_h + CONTEXT,
                    )
                    read_right = min(
                        source.width,
                        core_col + core_w + CONTEXT,
                    )

                    read_win = Window(
                        read_col,
                        read_row,
                        read_right - read_col,
                        read_bottom - read_row,
                    )

                    image, valid = read_window(
                        source,
                        read_win,
                    )

                    crop_row = core_row - read_row
                    crop_col = core_col - read_col

                    core_valid = valid[
                        crop_row:
                        crop_row + core_h,
                        crop_col:
                        crop_col + core_w,
                    ]

                    out = np.zeros(
                        (core_h, core_w),
                        dtype=np.uint16,
                    )

                    if core_valid.any():
                        channels = infer_channels(image)

                        core_channels = channels[
                            crop_row:
                            crop_row + core_h,
                            crop_col:
                            crop_col + core_w,
                        ]

                        out[core_valid] = (
                            core_channels[
                                core_valid
                            ]
                            .astype(
                                np.uint16,
                                copy=False,
                            )
                        )

                        del channels
                        del core_channels

                    destination.write(
                        out,
                        1,
                        window=Window(
                            core_col,
                            core_row,
                            core_w,
                            core_h,
                        ),
                    )

                    del image
                    del valid
                    del core_valid
                    del out

                    completed += 1

                    if (
                        completed % PRINT_EVERY == 0
                        or completed == total_blocks
                    ):
                        print(
                            f"[{LEVEL}] "
                            f"{completed:,}/"
                            f"{total_blocks:,} bloques "
                            f"({100.0 * completed / total_blocks:.1f} %)",
                            flush=True,
                        )

                    if completed % 500 == 0:
                        gc.collect()

    stage("inference_completed")

    # Liberar la red antes de todo lo demás.
    del MODEL
    gc.collect()

    stage("model_released")

    # ---------------------------------------------------------------------
    # 20 % calibración distribuida / 80 % holdout
    # ---------------------------------------------------------------------

    def calibration_mask(window, shape):
        h, w = shape

        rows = np.arange(
            int(window.row_off),
            int(window.row_off) + h,
            dtype=np.int64,
        )[:, None]

        cols = np.arange(
            int(window.col_off),
            int(window.col_off) + w,
            dtype=np.int64,
        )[None, :]

        return (
            (
                rows * 31
                + cols * 17
            )
            % CAL_MOD
            == CAL_REM
        )

    observed_reference = set()

    with rasterio.open(
        REFERENCE_PATH
    ) as ref_source:

        for _, window in ref_source.block_windows(1):
            values = np.unique(
                ref_source.read(
                    1,
                    window=window,
                )
            )

            observed_reference.update(
                int(v)
                for v in values.tolist()
                if int(v) != NODATA
            )

    reference_codes = sorted(
        observed_reference
        & set(TARGET_CODES)
    )

    if not reference_codes:
        raise RuntimeError(
            f"{LEVEL}: referencia sin clases válidas."
        )

    raw_codes = list(
        range(
            1,
            NUM_CLASSES + 1,
        )
    )

    ref_to_row = {
        code: index
        for index, code
        in enumerate(reference_codes)
    }

    calibration_matrix = np.zeros(
        (
            len(reference_codes),
            NUM_CLASSES,
        ),
        dtype=np.int64,
    )

    calibration_pixels = 0

    stage("class_correspondence_calibration")

    with rasterio.open(
        RAW_PATH
    ) as raw_source, rasterio.open(
        REFERENCE_PATH
    ) as ref_source:

        with WarpedVRT(
            ref_source,
            crs=raw_source.crs,
            transform=raw_source.transform,
            width=raw_source.width,
            height=raw_source.height,
            resampling=Resampling.nearest,
            nodata=NODATA,
        ) as reference:

            for _, window in raw_source.block_windows(1):
                raw = raw_source.read(
                    1,
                    window=window,
                ).astype(
                    np.int16,
                    copy=False,
                )

                ref = reference.read(
                    1,
                    window=window,
                ).astype(
                    np.int16,
                    copy=False,
                )

                cal = calibration_mask(
                    window,
                    raw.shape,
                )

                valid = (
                    cal
                    & (raw >= 1)
                    & (raw <= NUM_CLASSES)
                    & np.isin(
                        ref,
                        reference_codes,
                    )
                )

                if not valid.any():
                    continue

                r = ref[valid]
                p = raw[valid]

                calibration_pixels += int(
                    valid.sum()
                )

                ref_idx = np.full(
                    r.shape,
                    -1,
                    dtype=np.int16,
                )

                for code, row_index in ref_to_row.items():
                    ref_idx[r == code] = row_index

                raw_idx = p - 1

                usable = (
                    (ref_idx >= 0)
                    & (raw_idx >= 0)
                    & (raw_idx < NUM_CLASSES)
                )

                encoded = (
                    ref_idx[usable].astype(np.int64)
                    * NUM_CLASSES
                    + raw_idx[usable].astype(np.int64)
                )

                calibration_matrix += np.bincount(
                    encoded,
                    minlength=(
                        len(reference_codes)
                        * NUM_CLASSES
                    ),
                ).reshape(
                    len(reference_codes),
                    NUM_CLASSES,
                )

    if calibration_pixels == 0:
        raise RuntimeError(
            f"{LEVEL}: calibración sin píxeles válidos."
        )

    # ---------------------------------------------------------------------
    # Correspondencia por F1/Dice, no por conteos brutos.
    # ---------------------------------------------------------------------

    ref_support_cal = calibration_matrix.sum(
        axis=1,
        keepdims=True,
    ).astype(np.float64)

    raw_support_cal = calibration_matrix.sum(
        axis=0,
        keepdims=True,
    ).astype(np.float64)

    denominator = (
        ref_support_cal
        + raw_support_cal
    )

    dice = np.zeros_like(
        calibration_matrix,
        dtype=np.float64,
    )

    np.divide(
        2.0
        * calibration_matrix.astype(
            np.float64
        ),
        denominator,
        out=dice,
        where=denominator > 0,
    )

    ref_rows, raw_cols = (
        linear_sum_assignment(
            -dice
        )
    )

    mapping = {}
    mapping_rows = []
    matched_raw = set()
    matched_pixels = 0

    for ref_row, raw_col in zip(
        ref_rows.tolist(),
        raw_cols.tolist(),
    ):
        ref_code = int(
            reference_codes[
                ref_row
            ]
        )
        raw_code = int(
            raw_codes[
                raw_col
            ]
        )

        intersection = int(
            calibration_matrix[
                ref_row,
                raw_col,
            ]
        )

        ref_total = int(
            calibration_matrix[
                ref_row,
                :,
            ].sum()
        )

        raw_total = int(
            calibration_matrix[
                :,
                raw_col,
            ].sum()
        )

        mapping[raw_code] = ref_code
        matched_raw.add(raw_code)
        matched_pixels += intersection

        mapping_rows.append(
            {
                "salida_raw_canal": raw_code,
                "codigo_clc_final": ref_code,
                "nombre_clase_clc": CLASSES[ref_code],
                "pixeles_interseccion_calibracion": intersection,
                "soporte_raw_calibracion": raw_total,
                "soporte_referencia_calibracion": ref_total,
                "precision_solapamiento": (
                    intersection / raw_total
                    if raw_total > 0
                    else np.nan
                ),
                "recall_solapamiento": (
                    intersection / ref_total
                    if ref_total > 0
                    else np.nan
                ),
                "f1_dice_solapamiento": float(
                    dice[
                        ref_row,
                        raw_col,
                    ]
                ),
                "tipo_asignacion": "observada_en_referencia",
            }
        )

    unmatched_raw = [
        code
        for code in raw_codes
        if code not in matched_raw
    ]

    missing_reference = [
        code
        for code in TARGET_CODES
        if code not in reference_codes
    ]

    if len(unmatched_raw) != len(missing_reference):
        raise RuntimeError(
            f"{LEVEL}: correspondencia incompleta. "
            f"RAW libres={unmatched_raw}; "
            f"CLC ausentes={missing_reference}"
        )

    for raw_code, target_code in zip(
        unmatched_raw,
        missing_reference,
    ):
        mapping[int(raw_code)] = int(target_code)

        mapping_rows.append(
            {
                "salida_raw_canal": int(raw_code),
                "codigo_clc_final": int(target_code),
                "nombre_clase_clc": CLASSES[int(target_code)],
                "pixeles_interseccion_calibracion": 0,
                "soporte_raw_calibracion": int(
                    calibration_matrix[
                        :,
                        int(raw_code) - 1,
                    ].sum()
                ),
                "soporte_referencia_calibracion": 0,
                "precision_solapamiento": np.nan,
                "recall_solapamiento": np.nan,
                "f1_dice_solapamiento": np.nan,
                "tipo_asignacion": "clase_ausente_en_referencia",
            }
        )

    print(
        f"\n[{LEVEL}] CORRESPONDENCIA RAW -> CLC",
        flush=True,
    )

    for item in sorted(
        mapping_rows,
        key=lambda row: row[
            "salida_raw_canal"
        ],
    ):
        print(
            f"  RAW {item['salida_raw_canal']:>2} "
            f"-> CLC {item['codigo_clc_final']:>2} "
            f"| F1/Dice={item['f1_dice_solapamiento']}",
            flush=True,
        )

    calibration_accuracy = (
        matched_pixels
        / calibration_pixels
    )

    # ---------------------------------------------------------------------
    # GeoTIFF final armonizado
    # ---------------------------------------------------------------------

    FINAL_PATH = (
        LEVEL_DIR
        / f"DenseNet121_{LEVEL}_PREDICCION_FINAL_CLC.tif"
    )

    stage(
        "writing_final_prediction",
        mapping={
            str(k): int(v)
            for k, v in mapping.items()
        },
    )

    lookup = np.zeros(
        NUM_CLASSES + 1,
        dtype=np.uint16,
    )

    for raw_code, target_code in mapping.items():
        lookup[int(raw_code)] = int(target_code)

    with rasterio.open(
        RAW_PATH
    ) as raw_source:

        profile = raw_source.profile.copy()

        profile.update(
            dtype="uint16",
            nodata=NODATA,
            compress="deflate",
            predictor=2,
            tiled=True,
            blockxsize=256,
            blockysize=256,
            BIGTIFF="IF_SAFER",
        )

        with rasterio.open(
            FINAL_PATH,
            "w",
            **profile,
        ) as destination:

            for _, window in raw_source.block_windows(1):
                raw = raw_source.read(
                    1,
                    window=window,
                )

                valid = (
                    (raw >= 1)
                    & (raw <= NUM_CLASSES)
                )

                final = np.zeros(
                    raw.shape,
                    dtype=np.uint16,
                )

                final[valid] = lookup[
                    raw[valid]
                ]

                destination.write(
                    final,
                    1,
                    window=window,
                )

    try:
        RAW_PATH.unlink()
    except Exception:
        pass

    gc.collect()

    # ---------------------------------------------------------------------
    # Métricas 80 % holdout
    # ---------------------------------------------------------------------

    stage("holdout_metrics")

    codes = TARGET_CODES

    code_to_index = {
        code: index
        for index, code
        in enumerate(codes)
    }

    confusion = np.zeros(
        (
            len(codes),
            len(codes),
        ),
        dtype=np.int64,
    )

    with rasterio.open(
        FINAL_PATH
    ) as prediction, rasterio.open(
        REFERENCE_PATH
    ) as ref_source:

        with WarpedVRT(
            ref_source,
            crs=prediction.crs,
            transform=prediction.transform,
            width=prediction.width,
            height=prediction.height,
            resampling=Resampling.nearest,
            nodata=NODATA,
        ) as reference:

            for _, window in prediction.block_windows(1):
                pred = prediction.read(
                    1,
                    window=window,
                ).astype(
                    np.int16,
                    copy=False,
                )

                ref = reference.read(
                    1,
                    window=window,
                ).astype(
                    np.int16,
                    copy=False,
                )

                holdout = ~calibration_mask(
                    window,
                    pred.shape,
                )

                valid = (
                    holdout
                    & np.isin(pred, codes)
                    & np.isin(ref, codes)
                )

                if not valid.any():
                    continue

                p = pred[valid]
                r = ref[valid]

                true_idx = np.full(
                    r.shape,
                    -1,
                    dtype=np.int16,
                )
                pred_idx = np.full(
                    p.shape,
                    -1,
                    dtype=np.int16,
                )

                for code, index in code_to_index.items():
                    true_idx[r == code] = index
                    pred_idx[p == code] = index

                usable = (
                    (true_idx >= 0)
                    & (pred_idx >= 0)
                )

                encoded = (
                    true_idx[usable].astype(np.int64)
                    * len(codes)
                    + pred_idx[usable].astype(np.int64)
                )

                confusion += np.bincount(
                    encoded,
                    minlength=(
                        len(codes) ** 2
                    ),
                ).reshape(
                    len(codes),
                    len(codes),
                )

    matrix = confusion.astype(np.float64)

    total = float(matrix.sum())
    tp = np.diag(matrix)

    ref_support = matrix.sum(axis=1)
    pred_support = matrix.sum(axis=0)

    fp = pred_support - tp
    fn = ref_support - tp
    tn = total - tp - fp - fn

    def safe_div(a, b):
        a = np.asarray(a, dtype=np.float64)
        b = np.asarray(b, dtype=np.float64)

        result = np.full(
            np.broadcast_shapes(
                a.shape,
                b.shape,
            ),
            np.nan,
            dtype=np.float64,
        )

        return np.divide(
            a,
            b,
            out=result,
            where=b != 0,
        )

    precision = safe_div(
        tp,
        tp + fp,
    )
    recall = safe_div(
        tp,
        tp + fn,
    )
    f1 = safe_div(
        2.0 * precision * recall,
        precision + recall,
    )
    iou = safe_div(
        tp,
        tp + fp + fn,
    )
    specificity = safe_div(
        tn,
        tn + fp,
    )

    supported = ref_support > 0

    def macro(values):
        selected = (
            supported
            & np.isfinite(values)
        )
        return (
            float(values[selected].mean())
            if selected.any()
            else np.nan
        )

    accuracy = (
        float(tp.sum() / total)
        if total > 0
        else np.nan
    )

    expected = (
        float(
            np.dot(
                ref_support,
                pred_support,
            )
            / (total * total)
        )
        if total > 0
        else np.nan
    )

    kappa = (
        float(
            (accuracy - expected)
            / (1.0 - expected)
        )
        if (
            total > 0
            and np.isfinite(expected)
            and not np.isclose(
                1.0 - expected,
                0.0,
            )
        )
        else np.nan
    )

    numerator_mcc = (
        total * tp.sum()
        - np.dot(
            ref_support,
            pred_support,
        )
    )

    denominator_mcc = math.sqrt(
        max(
            0.0,
            (
                total * total
                - np.dot(
                    pred_support,
                    pred_support,
                )
            )
            * (
                total * total
                - np.dot(
                    ref_support,
                    ref_support,
                )
            ),
        )
    )

    mcc = (
        float(
            numerator_mcc
            / denominator_mcc
        )
        if denominator_mcc > 0
        else np.nan
    )

    metrics = {
        "evaluated_pixels_holdout": int(total),
        "accuracy": accuracy,
        "balanced_accuracy": macro(recall),
        "macro_precision": macro(precision),
        "macro_recall": macro(recall),
        "macro_f1": macro(f1),
        "mean_iou": macro(iou),
        "macro_specificity": macro(specificity),
        "kappa": kappa,
        "mcc": mcc,
    }

    # ---------------------------------------------------------------------
    # Reportes: el modelo ya está liberado.
    # ---------------------------------------------------------------------

    stage("writing_reports")

    import pandas as pd

    per_class = pd.DataFrame(
        {
            "class_code": codes,
            "class_name": [
                CLASSES[code]
                for code in codes
            ],
            "reference_pixels": (
                ref_support.astype(np.int64)
            ),
            "predicted_pixels": (
                pred_support.astype(np.int64)
            ),
            "true_positive": (
                tp.astype(np.int64)
            ),
            "false_positive": (
                fp.astype(np.int64)
            ),
            "false_negative": (
                fn.astype(np.int64)
            ),
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "iou": iou,
            "specificity": specificity,
            "reference_support": supported,
        }
    )

    labels = [
        f"{code} - {CLASSES[code]}"
        for code in codes
    ]

    confusion_df = pd.DataFrame(
        confusion,
        index=labels,
        columns=labels,
    )

    mapping_df = pd.DataFrame(
        sorted(
            mapping_rows,
            key=lambda row: row[
                "salida_raw_canal"
            ],
        )
    )

    metrics_file = {
        **metrics,
        "model": "DenseNet121-UNet",
        "level": LEVEL,
        "run_dir": str(RUN_DIR),
        "prediction": str(FINAL_PATH),
        "reference": str(REFERENCE_PATH),
        "core_size": CORE_SIZE,
        "context_pixels": CONTEXT,
        "device": "cpu_strict_child_process",
        "calibration_percentage": 20,
        "holdout_percentage": 80,
        "calibration_accuracy": float(
            calibration_accuracy
        ),
    }

    mapping_summary = {
        "level": LEVEL,
        "method": (
            "20% calibracion distribuida + "
            "Hungarian maximizando F1/Dice"
        ),
        "calibration_pixels": int(
            calibration_pixels
        ),
        "calibration_accuracy": float(
            calibration_accuracy
        ),
        "reference_codes_observed": (
            reference_codes
        ),
        "mapping": {
            str(k): int(v)
            for k, v in mapping.items()
        },
        "missing_reference_classes": (
            missing_reference
        ),
    }

    save_json(
        metrics_file,
        LEVEL_DIR / "metricas_globales.json",
    )

    save_json(
        mapping_summary,
        LEVEL_DIR / "correspondencia_clases.json",
    )

    mapping_df.to_csv(
        LEVEL_DIR / "correspondencia_clases.csv",
        index=False,
        encoding="utf-8-sig",
    )

    EXCEL_PATH = (
        LEVEL_DIR
        / f"RESULTADOS_DENSENET121_{LEVEL}.xlsx"
    )

    with pd.ExcelWriter(
        EXCEL_PATH,
        engine="openpyxl",
    ) as writer:

        pd.DataFrame(
            [metrics_file]
        ).to_excel(
            writer,
            sheet_name="metricas_globales",
            index=False,
        )

        per_class.to_excel(
            writer,
            sheet_name="metricas_por_clase",
            index=False,
        )

        confusion_df.to_excel(
            writer,
            sheet_name="matriz_confusion",
        )

        mapping_df.to_excel(
            writer,
            sheet_name="correspondencia_clases",
            index=False,
        )

    # ---------------------------------------------------------------------
    # Única visualización, misma simbología
    # ---------------------------------------------------------------------

    stage("writing_comparison_png")

    import matplotlib
    matplotlib.use("Agg")

    import matplotlib.pyplot as plt
    from matplotlib.colors import (
        BoundaryNorm,
        ListedColormap,
    )
    from matplotlib.patches import Patch

    with rasterio.open(
        FINAL_PATH
    ) as prediction, rasterio.open(
        REFERENCE_PATH
    ) as ref_source:

        scale = min(
            QUICKLOOK_MAX
            / max(
                prediction.height,
                prediction.width,
            ),
            1.0,
        )

        out_h = max(
            1,
            int(
                prediction.height
                * scale
            ),
        )
        out_w = max(
            1,
            int(
                prediction.width
                * scale
            ),
        )

        pred_q = prediction.read(
            1,
            out_shape=(
                out_h,
                out_w,
            ),
            resampling=Resampling.nearest,
        )

        with WarpedVRT(
            ref_source,
            crs=prediction.crs,
            transform=prediction.transform,
            width=prediction.width,
            height=prediction.height,
            resampling=Resampling.nearest,
            nodata=NODATA,
        ) as reference:

            ref_q = reference.read(
                1,
                out_shape=(
                    out_h,
                    out_w,
                ),
                resampling=Resampling.nearest,
            )

    cmap = ListedColormap(
        [
            COLORS[code]
            for code in codes
        ]
    )

    boundaries = (
        [codes[0] - 0.5]
        + [
            code + 0.5
            for code in codes
        ]
    )

    norm = BoundaryNorm(
        boundaries,
        cmap.N,
    )

    ref_masked = np.ma.masked_where(
        ref_q == NODATA,
        ref_q,
    )
    pred_masked = np.ma.masked_where(
        pred_q == NODATA,
        pred_q,
    )

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(16, 9),
    )

    axes[0].imshow(
        ref_masked,
        cmap=cmap,
        norm=norm,
        interpolation="nearest",
    )
    axes[0].set_title(
        f"Referencia CLC - {LEVEL}",
        fontweight="bold",
    )
    axes[0].axis("off")

    axes[1].imshow(
        pred_masked,
        cmap=cmap,
        norm=norm,
        interpolation="nearest",
    )
    axes[1].set_title(
        f"DenseNet121 - {LEVEL}",
        fontweight="bold",
    )
    axes[1].axis("off")

    legend = [
        Patch(
            facecolor=COLORS[code],
            edgecolor="black",
            label=f"{code} - {CLASSES[code]}",
        )
        for code in codes
    ]

    fig.legend(
        handles=legend,
        loc="lower center",
        ncol=(
            2
            if LEVEL == "nivel_1"
            else 3
        ),
        fontsize=8,
        frameon=True,
        bbox_to_anchor=(0.5, 0.01),
    )

    fig.suptitle(
        "Referencia y predicción con la misma simbología",
        fontweight="bold",
    )

    fig.tight_layout(
        rect=(0, 0.12, 1, 0.95)
    )

    PNG_PATH = (
        LEVEL_DIR
        / f"DenseNet121_{LEVEL}_COMPARACION_REFERENCIA_PREDICCION.png"
    )

    fig.savefig(
        PNG_PATH,
        dpi=140,
        bbox_inches="tight",
    )

    plt.close(fig)

    del pred_q
    del ref_q
    del ref_masked
    del pred_masked

    gc.collect()

    status.update(
        {
            "status": "completed",
            "stage": "completed",
            "accuracy": accuracy,
            "balanced_accuracy": (
                metrics[
                    "balanced_accuracy"
                ]
            ),
            "macro_f1": (
                metrics[
                    "macro_f1"
                ]
            ),
            "mean_iou": (
                metrics[
                    "mean_iou"
                ]
            ),
            "kappa": kappa,
            "mcc": mcc,
            "calibration_accuracy": float(
                calibration_accuracy
            ),
            "prediction": str(FINAL_PATH),
            "excel": str(EXCEL_PATH),
            "comparison_png": str(PNG_PATH),
            "mapping": {
                str(k): int(v)
                for k, v in mapping.items()
            },
        }
    )

    save_json(status, STATUS_PATH)

    print(
        "\n" + "=" * 80,
        flush=True,
    )
    print(
        f"[{LEVEL}] COMPLETADO",
        flush=True,
    )
    print(
        f"Accuracy: {accuracy:.6f}",
        flush=True,
    )
    print(
        "Balanced Accuracy: "
        f"{metrics['balanced_accuracy']:.6f}",
        flush=True,
    )
    print(
        f"Macro F1: {metrics['macro_f1']:.6f}",
        flush=True,
    )
    print(
        f"Mean IoU: {metrics['mean_iou']:.6f}",
        flush=True,
    )
    print(
        f"Kappa: {kappa:.6f}",
        flush=True,
    )
    print(
        f"MCC: {mcc:.6f}",
        flush=True,
    )
    print(
        f"Predicción: {FINAL_PATH}",
        flush=True,
    )
    print(
        "=" * 80,
        flush=True,
    )

except Exception as error:
    status.update(
        {
            "status": "error",
            "error": str(error),
            "traceback": traceback.format_exc(),
        }
    )

    try:
        save_json(status, STATUS_PATH)
    except Exception:
        pass

    print(
        "\n" + "=" * 80,
        file=sys.stderr,
        flush=True,
    )
    print(
        f"[ERROR {LEVEL}] {error}",
        file=sys.stderr,
        flush=True,
    )
    traceback.print_exc()
    print(
        "=" * 80,
        file=sys.stderr,
        flush=True,
    )
    raise
"""

# -----------------------------------------------------------------------------
# 3. Ejecutar un nivel aislado SIN crear archivo auxiliar
# -----------------------------------------------------------------------------

def run_isolated_level(
    level_name: str,
    core_size: int,
) -> int:

    env = os.environ.copy()

    env.update(
        {
            "DENSENET_LEVEL": level_name,
            "DENSENET_CORE_SIZE": str(
                core_size
            ),
            "DENSENET_CONTEXT": str(
                CONTEXT_PIXELS
            ),
            "DENSENET_CPU_THREADS": str(
                CPU_THREADS
            ),
            "DENSENET_CAL_MOD": str(
                CALIBRATION_MODULUS
            ),
            "DENSENET_CAL_REM": str(
                CALIBRATION_REMAINDER
            ),
            "DENSENET_PRINT_EVERY": str(
                PRINT_EVERY_BLOCKS
            ),
            "DENSENET_QUICKLOOK": str(
                QUICKLOOK_MAX_SIZE
            ),
            "DENSENET_MODEL_ROOT": str(
                MODEL_ROOT
            ),
            "DENSENET_INPUT_RASTER": str(
                INPUT_RASTER
            ),
            "DENSENET_OUTPUT_DIR": str(
                OUTPUT_DIR
            ),
            "DENSENET_REFERENCE": str(
                REFERENCE_BY_LEVEL[
                    level_name
                ]
            ),
            "DENSENET_RUN_FORCED": (
                ""
                if RUN_BY_LEVEL[
                    level_name
                ] is None
                else str(
                    RUN_BY_LEVEL[
                        level_name
                    ]
                )
            ),
            # Aplicados antes del import de torch en el proceso hijo.
            "OMP_NUM_THREADS": str(
                CPU_THREADS
            ),
            "MKL_NUM_THREADS": str(
                CPU_THREADS
            ),
            "OPENBLAS_NUM_THREADS": str(
                CPU_THREADS
            ),
            "NUMEXPR_NUM_THREADS": str(
                CPU_THREADS
            ),
            "GDAL_NUM_THREADS": "1",
            "GDAL_CACHEMAX": "64",
            "CUDA_VISIBLE_DEVICES": "-1",
            "HIP_VISIBLE_DEVICES": "-1",
            "ROCR_VISIBLE_DEVICES": "-1",
            "PYTHONUNBUFFERED": "1",
        }
    )

    bootstrap = (
        "import sys;"
        "source=sys.stdin.read();"
        "exec(compile("
        "source,"
        "'<DenseNet121_worker>',"
        "'exec'"
        "))"
    )

    process = subprocess.Popen(
        [
            sys.executable,
            "-u",
            "-c",
            bootstrap,
        ],
        stdin=subprocess.PIPE,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )

    # Se envía el worker por stdin, no por línea de comandos
    # y no se crea ningún .py.
    process.stdin.write(
        WORKER_SOURCE
    )
    process.stdin.close()

    # Log en tiempo real sin acumularlo completo en RAM.
    for line in process.stdout:
        print(
            line,
            end="",
            flush=True,
        )

    process.stdout.close()

    return int(
        process.wait()
    )

# -----------------------------------------------------------------------------
# 4. Ejecutar Nivel 1 y Nivel 2 desde esta misma celda
# -----------------------------------------------------------------------------

print("=" * 92)
print(
    "DENSENET121 — PREDICCIÓN NUEVA, DOS NIVELES, PROCESOS AISLADOS"
)
print(
    "UNA SOLA CELDA | CPU ESTRICTA | 20 % CORRESPONDENCIA | 80 % HOLDOUT"
)
print("=" * 92)

print(
    f"Python del kernel: {sys.executable}"
)
print(
    f"Modelos: {MODEL_ROOT}"
)
print(
    f"Imagen:  {INPUT_RASTER}"
)
print(
    f"Salida:  {OUTPUT_DIR}"
)
print(
    "DenseNet121 NO se cargará dentro del proceso del kernel."
)

summaries = []

for level_name in LEVELS:
    print(
        "\n" + "#" * 92
    )
    print(
        f"INICIANDO {level_name.upper()}"
    )
    print(
        "#" * 92
    )

    success = False
    last_return_code = None
    used_core = None

    for attempt_number, core_size in enumerate(
        CORE_SIZE_ATTEMPTS,
        start=1,
    ):
        used_core = core_size

        print(
            f"\n[{level_name}] Intento "
            f"{attempt_number}/"
            f"{len(CORE_SIZE_ATTEMPTS)} "
            f"| núcleo={core_size}px "
            f"| contexto={CONTEXT_PIXELS}px"
        )

        return_code = run_isolated_level(
            level_name,
            core_size,
        )

        last_return_code = return_code

        if return_code == 0:
            success = True

            print(
                f"\n[OK] {level_name} terminado "
                f"con núcleo={core_size}px."
            )

            break

        print(
            f"\n[ADVERTENCIA] {level_name} "
            f"terminó con código {return_code}."
        )

        if attempt_number < len(
            CORE_SIZE_ATTEMPTS
        ):
            print(
                "Se elimina la salida parcial y "
                "se vuelve a predecir TODO el nivel "
                "con una tesela menor."
            )

    status_path = (
        OUTPUT_DIR
        / level_name
        / "WORKER_STATUS.json"
    )

    if status_path.exists():
        try:
            with status_path.open(
                "r",
                encoding="utf-8",
            ) as file:
                status = json.load(file)
        except Exception:
            status = {}
    else:
        status = {}

    summaries.append(
        {
            "level": level_name,
            "status": (
                "completed"
                if success
                else "error"
            ),
            "return_code": (
                last_return_code
            ),
            "core_size": status.get(
                "core_size",
                used_core,
            ),
            "stage": status.get(
                "stage"
            ),
            "accuracy": status.get(
                "accuracy"
            ),
            "balanced_accuracy": status.get(
                "balanced_accuracy"
            ),
            "macro_f1": status.get(
                "macro_f1"
            ),
            "mean_iou": status.get(
                "mean_iou"
            ),
            "kappa": status.get(
                "kappa"
            ),
            "mcc": status.get(
                "mcc"
            ),
            "prediction": status.get(
                "prediction"
            ),
            "excel": status.get(
                "excel"
            ),
            "error": status.get(
                "error"
            ),
        }
    )

    if not success:
        print(
            f"\n[ERROR] {level_name} no pudo completarse, "
            "pero el kernel debe seguir activo porque "
            "DenseNet121 se ejecutó en un proceso separado."
        )

# -----------------------------------------------------------------------------
# 5. Resumen final sin pandas dentro del kernel
# -----------------------------------------------------------------------------

summary_csv = (
    OUTPUT_DIR
    / "RESUMEN_DENSENET121_PREDICCION_AISLADA.csv"
)

fieldnames = [
    "level",
    "status",
    "return_code",
    "core_size",
    "stage",
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "mean_iou",
    "kappa",
    "mcc",
    "prediction",
    "excel",
    "error",
]

with summary_csv.open(
    "w",
    newline="",
    encoding="utf-8-sig",
) as file:

    writer = csv.DictWriter(
        file,
        fieldnames=fieldnames,
    )

    writer.writeheader()

    for row in summaries:
        writer.writerow(row)

print(
    "\n" + "=" * 92
)
print(
    "RESUMEN FINAL"
)
print(
    "=" * 92
)

for row in summaries:
    print(
        f"{row['level']}: "
        f"status={row['status']} | "
        f"core={row['core_size']} | "
        f"Accuracy={row['accuracy']} | "
        f"MacroF1={row['macro_f1']} | "
        f"IoU={row['mean_iou']} | "
        f"Kappa={row['kappa']} | "
        f"MCC={row['mcc']}"
    )

print(
    f"\nResumen CSV: {summary_csv}"
)

resultados_densenet121 = summaries


DENSENET121 — PREDICCIÓN NUEVA, DOS NIVELES, PROCESOS AISLADOS
UNA SOLA CELDA | CPU ESTRICTA | 20 % CORRESPONDENCIA | 80 % HOLDOUT
Python del kernel: C:\CondaEnvs\DENSENET121_AMD\python.exe
Modelos: C:\Users\Ivan sua\anaconda3\Modelos_DenseNet121_Segmentacion
Imagen:  C:\Users\Ivan sua\anaconda3\Rasters_CLC\zona_estudio.tif
Salida:  C:\Users\Ivan sua\anaconda3\Predicciones_DenseNet121_Aislada_Final
DenseNet121 NO se cargará dentro del proceso del kernel.

############################################################################################
INICIANDO NIVEL_1
############################################################################################

[nivel_1] Intento 1/3 | núcleo=96px | contexto=16px
Failed to get device count
[nivel_1] ETAPA: locating_training
[nivel_1] ETAPA: loading_model
[nivel_1] ETAPA: model_loaded
[nivel_1] ETAPA: new_inference
[nivel_1] 250/2,392 bloques (10.5 %)
[nivel_1] 500/2,392 bloques (20.9 %)
[nivel_1] 750/2,392 bloques (31.4 %)
[nivel_1] 1,000/2,